# Pretrain simclr on HyperKvasir unlabeled

ViT-S/16 @ 224, global batch 512, 100 epochs. Estimated **~15.8 GPU-hours** (~3 session(s) at the 7.5h guard).

**This notebook is resumable.** It stops cleanly before the session cap, saves full training state (model, EMA target, optimizer, scaler, schedule position) to a Kaggle Dataset, and picks up exactly where it left off next run. Just *Save & Run All* again until it prints `run complete`.

Requires **GPU T4 x2** and Internet ON, plus `KAGGLE_USERNAME`/`KAGGLE_KEY` under Add-ons → Secrets for cross-session checkpointing.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')
    return r

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Report the accelerator and the precision that follows from it.
# T4 (sm_75) has fp16 tensor cores but NO bf16 hardware; P100 (sm_60) has
# neither and runs ~2x slower. The code adapts either way — this cell is
# here so you know what you were given before spending 8 hours on it.
import torch
from src.config import amp_config
print('CUDA devices:', torch.cuda.device_count())
amp = amp_config()
print(amp)
if torch.cuda.device_count() < 2:
    print('\n*** Only one GPU. I-JEPA and MAE will still run correctly, but\n'
          '    SimCLR/MoCo v3 need 2 GPUs to preserve global_batch=512.\n'
          '    Set Session options -> Accelerator -> GPU T4 x2 and re-run. ***')


### Source files (editable)

Each cell below writes one file. Edit a cell and re-run it to patch the code in this session without a git round-trip. Re-running the clone cell above discards these edits.

Run the guard cell first — `%%writefile` writes relative to the working directory and will not create missing folders, so it fails with `FileNotFoundError` if the clone has not run or the kernel was restarted.

In [ ]:
# Guard for the %%writefile cells below. Safe to re-run at any time.
#   1. %%writefile resolves paths relative to os.getcwd(), and a kernel
#      restart resets that to /kaggle/working — so re-assert it.
#   2. %%writefile will NOT create parent directories; it raises
#      FileNotFoundError instead. So create them up front.
#   3. Empty __init__.py files cannot be written by %%writefile at all
#      ('cell body is empty'), so they are created here.
import os, pathlib

if not os.path.isdir(WORKDIR):
    raise RuntimeError(
        f'{WORKDIR} does not exist — run the git clone cell above first.')
os.chdir(WORKDIR)
for d in ['configs', 'src', 'src/data', 'src/engine', 'src/eval', 'src/masks', 'src/model', 'src/utils', 'src/viz']:
    pathlib.Path(WORKDIR, d).mkdir(parents=True, exist_ok=True)
for f in ['src/__init__.py', 'src/utils/__init__.py', 'src/engine/__init__.py']:
    pathlib.Path(WORKDIR, f).touch(exist_ok=True)
print('cwd:', os.getcwd())
print('ready for the %%writefile cells')


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/config.py
"""Central configuration.

Two things this module owns that the rest of the codebase must not duplicate:

1. **The AMP capability gate** (`amp_config`). Kaggle hands out T4 (sm_75) or, less
   happily, P100 (sm_60). Turing has fp16 tensor cores but *no bf16 hardware* — a
   bfloat16 autocast there is emulated and runs slower than fp32. P100 has no tensor
   cores at all but does support fp16 AMP fine. So we pick the dtype from the compute
   capability rather than assuming, and we never abort on old hardware: a slow session
   beats a dead one.

2. **Global-batch pinning** (`derive_batch`). T4 x2 gives world_size 2, P100 gives 1.
   If per-GPU batch stayed constant the *global* batch would silently halve mid-run and
   the LR schedule, contrastive negative count and EMA schedule would all shift —
   invalidating the experiment. We pin the global batch and derive per-GPU from it.

Configs are dataclasses overlaid with a YAML file from `configs/`. There is deliberately
no module-level singleton: dataset resolution touches the filesystem and must not run at
import time (the previous version raised on `import src.config` when data was absent).
Call `build_config(...)` explicitly.
"""
from __future__ import annotations

import hashlib
import json
import os
import warnings
from dataclasses import asdict, dataclass, field, fields, is_dataclass
from pathlib import Path
from typing import Any, Sequence

REPO_ROOT = Path(__file__).resolve().parent.parent

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

# ViT variants, matching facebookresearch/ijepa's model zoo exactly.
VIT_ARCHS: dict[str, dict[str, int]] = {
    "vit_tiny": {"embed_dim": 192, "depth": 12, "num_heads": 3},
    "vit_small": {"embed_dim": 384, "depth": 12, "num_heads": 6},
    "vit_base": {"embed_dim": 768, "depth": 12, "num_heads": 12},
    "vit_large": {"embed_dim": 1024, "depth": 24, "num_heads": 16},
}


# ------------------------------------------------------------------ environment

def on_kaggle() -> bool:
    return os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ


def default_output_dir() -> Path:
    """Persisted-within-session output. Capped at 20 GiB on Kaggle."""
    return Path("/kaggle/working") if on_kaggle() else REPO_ROOT / "outputs"


def scratch_dir() -> Path:
    """Large scratch space that does NOT count against the 20 GiB output cap.

    Kaggle gives ~60 GiB here; it is wiped when the session ends. Use it for
    downloads and intermediate archives, never for anything you need to keep.
    """
    if on_kaggle():
        for c in (Path("/kaggle/tmp"), Path("/kaggle/temp")):
            if c.parent.exists():
                c.mkdir(parents=True, exist_ok=True)
                return c
    return Path("/tmp")


# ------------------------------------------------------- AMP / device capability

@dataclass(frozen=True)
class AmpConfig:
    """Resolved autocast settings for whatever accelerator we actually landed on."""

    device: str  # "cuda" | "mps" | "cpu"
    dtype: Any  # torch.dtype or None (None => run in fp32)
    use_scaler: bool
    sm: int  # compute capability as major*10+minor, 0 if not CUDA
    name: str
    speed_factor: float  # rough throughput vs a single T4, for time estimates

    @property
    def enabled(self) -> bool:
        return self.dtype is not None


def amp_config(force_fp32: bool = False) -> AmpConfig:
    """Pick device + autocast dtype from the actual hardware.

    sm_80+ (A100/L4/A10) -> bfloat16, no GradScaler needed (bf16 has fp32 range).
    sm_70/75 (V100/T4)   -> float16 + GradScaler. Tensor cores; bf16 would be emulated.
    sm_60/61 (P100)      -> float16 + GradScaler. No tensor cores, ~2x slower, but works.
    older / non-CUDA     -> fp32.
    """
    try:
        import torch
    except ImportError:  # figure scripts run without torch
        return AmpConfig("cpu", None, False, 0, "cpu (no torch)", 0.02)

    if torch.cuda.is_available():
        try:
            major, minor = torch.cuda.get_device_capability(0)
            sm = major * 10 + minor
            name = torch.cuda.get_device_name(0)
            if force_fp32:
                return AmpConfig("cuda", None, False, sm, name, 1.0)
            if sm >= 80:
                return AmpConfig("cuda", torch.bfloat16, False, sm, name, 3.0)
            if sm >= 70:
                return AmpConfig("cuda", torch.float16, True, sm, name, 1.0)
            if sm >= 60:
                warnings.warn(
                    f"{name} (sm_{sm}) has no tensor cores; expect roughly 2x slower "
                    "training than a T4. Continuing in fp16. For a faster run, set "
                    "Accelerator -> GPU T4 x2 in the Kaggle session options.",
                    RuntimeWarning,
                    stacklevel=2,
                )
                return AmpConfig("cuda", torch.float16, True, sm, name, 0.45)
            warnings.warn(f"{name} (sm_{sm}) is too old for AMP; running fp32.", RuntimeWarning)
            return AmpConfig("cuda", None, False, sm, name, 0.2)
        except Exception as e:  # noqa: BLE001 - CUDA present but unusable
            warnings.warn(f"CUDA present but unusable ({e}); falling back to CPU.", RuntimeWarning)

    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return AmpConfig("mps", None, False, 0, "apple-mps", 0.1)
    return AmpConfig("cpu", None, False, 0, "cpu", 0.02)


def derive_batch(global_batch: int, world_size: int, accum_steps: int = 1) -> int:
    """Per-GPU batch size that exactly preserves `global_batch`.

    Raises rather than rounding: a global batch that silently drifts between sessions
    is the kind of bug that invalidates a whole experiment without ever erroring.
    """
    denom = world_size * accum_steps
    if global_batch % denom != 0:
        raise ValueError(
            f"global_batch={global_batch} is not divisible by world_size({world_size}) "
            f"* accum_steps({accum_steps})={denom}. Adjust accum_steps so the global "
            "batch is preserved exactly."
        )
    return global_batch // denom


# ----------------------------------------------------------- dataset resolution

def _count_images(d: Path, recursive: bool = False) -> int:
    it = d.rglob("*") if recursive else d.iterdir()
    n = 0
    for p in it:
        if p.suffix.lower() in IMAGE_EXTS:
            n += 1
            if n > 4:  # cheap existence probe; callers only need "has images"
                if not recursive:
                    return n
    return n


def resolve_dataset_dir(
    kind: str,
    override: str | os.PathLike[str] | None = None,
    required_subdirs: Sequence[str] = (),
) -> Path:
    """Locate a dataset directory, on Kaggle or locally.

    `kind` is one of the keys in `_DATASET_CANDIDATES`. On Kaggle we try the known
    mount points first, then fall back to a recursive scan of /kaggle/input picking the
    directory with the most images — Kaggle occasionally nests dataset archives one
    level deeper than the slug suggests, and that fallback has already saved this
    project once.
    """
    if override is not None:
        p = Path(override)
        if not p.exists():
            raise FileNotFoundError(f"dataset override path does not exist: {p}")
        return p

    candidates = _DATASET_CANDIDATES.get(kind)
    if candidates is None:
        raise KeyError(f"unknown dataset kind {kind!r}; known: {sorted(_DATASET_CANDIDATES)}")

    roots = [Path("/kaggle/input")] if on_kaggle() else [REPO_ROOT / "data"]

    def ok(d: Path) -> bool:
        if not d.is_dir():
            return False
        if required_subdirs and not all((d / s).is_dir() for s in required_subdirs):
            return False
        return required_subdirs != () or _count_images(d) > 0

    for root in roots:
        for rel in candidates:
            c = root / rel
            if ok(c):
                print(f"[data] {kind}: using {c}")
                return c

    # Fallback: recursive scan.
    for root in roots:
        if not root.exists():
            continue
        print(f"[data] {kind}: candidates missed, scanning {root} recursively ...")
        matches: list[tuple[Path, int]] = []
        for sub in root.rglob("*"):
            if sub.is_dir() and ok(sub):
                matches.append((sub, _count_images(sub, recursive=True)))
        matches.sort(key=lambda t: -t[1])
        for p, n in matches[:10]:
            print(f"[data]   candidate: {p}  ({n} images)")
        if matches:
            print(f"[data] {kind}: using {matches[0][0]}")
            return matches[0][0]

    raise FileNotFoundError(
        f"Could not locate dataset {kind!r} under {[str(r) for r in roots]}. "
        f"Tried {candidates}. On Kaggle, attach it via kernel-metadata.json -> "
        "dataset_sources; locally, place it under data/."
    )


_DATASET_CANDIDATES: dict[str, tuple[str, ...]] = {
    # Our own pre-resized 256px corpus, built by scripts/build_hyperkvasir_dataset.py
    "hyperkvasir": (
        "hyperkvasir-unlabeled-256/hk256",
        "hyperkvasir-unlabeled-256",
        "hyper-kvasir-unlabeled-256",
        "hyperkvasir/unlabeled-images",
        "HyperKvasir/unlabeled-images",
    ),
    # HyperKvasir labelled split (23 classes) — used only for k-NN / linear probe.
    "hyperkvasir_labeled": (
        "hyperkvasir-labeled-256/hk_labeled256",
        "hyperkvasir-labeled-256",
        "hyperkvasir/labeled-images",
    ),
    # kaggle.com/datasets/debeshjha1/kvasirseg
    "kvasir_seg": (
        "kvasirseg/Kvasir-SEG",
        "kvasir-seg/Kvasir-SEG",
        "kvasirseg",
        "Kvasir-SEG",
    ),
    # External generalisation set, inference only.
    "cvc_clinicdb": (
        "cvcclinicdb/PNG",
        "cvc-clinicdb/PNG",
        "cvcclinicdb",
        "CVC-ClinicDB",
    ),
}


# ------------------------------------------------------------------- dataclasses

@dataclass
class ModelCfg:
    arch: str = "vit_small"
    patch_size: int = 16
    img_size: int = 224
    drop_path_rate: float = 0.0

    @property
    def embed_dim(self) -> int:
        return VIT_ARCHS[self.arch]["embed_dim"]

    @property
    def depth(self) -> int:
        return VIT_ARCHS[self.arch]["depth"]

    @property
    def num_heads(self) -> int:
        return VIT_ARCHS[self.arch]["num_heads"]

    @property
    def grid_size(self) -> int:
        if self.img_size % self.patch_size:
            raise ValueError(f"img_size {self.img_size} not divisible by patch {self.patch_size}")
        return self.img_size // self.patch_size

    @property
    def num_patches(self) -> int:
        return self.grid_size**2


@dataclass
class OptimCfg:
    """Shared optimisation budget. Held identical across all four SSL methods.

    The controlled variable in the thesis is *samples seen* = epochs * corpus_size,
    not epochs, since every method sees the same corpus.
    """

    global_batch: int = 512
    accum_steps: int = 1
    epochs: int = 100
    warmup_epochs: int = 10
    start_lr: float = 2.0e-4
    ref_lr: float = 2.5e-4
    final_lr: float = 1.0e-6
    weight_decay: float = 0.04
    final_weight_decay: float = 0.4
    exclude_bias_and_norm_from_wd: bool = True
    ema: tuple[float, float] = (0.996, 1.0)
    ipe_scale: float = 1.0
    grad_clip: float = 3.0


@dataclass
class MaskCfg:
    """i-jepa multiblock masking. Defaults are the released in1k config values."""

    enc_mask_scale: tuple[float, float] = (0.85, 1.0)
    pred_mask_scale: tuple[float, float] = (0.15, 0.2)
    aspect_ratio: tuple[float, float] = (0.75, 1.5)
    num_enc_masks: int = 1
    num_pred_masks: int = 4
    min_keep: int = 10
    allow_overlap: bool = False


@dataclass
class JEPAHeadCfg:
    # 192 == 0.5 * 384, the same predictor/encoder width ratio the paper uses for ViT-B
    # (384/768). Copying the absolute 384 would be a ViT-B-sized head on a ViT-S body.
    pred_emb_dim: int = 192
    pred_depth: int = 6


@dataclass
class MAEHeadCfg:
    mask_ratio: float = 0.75
    decoder_embed_dim: int = 256  # 0.67 * 384, matching MAE's own 512/768 ratio
    decoder_depth: int = 8
    decoder_num_heads: int = 8
    norm_pix_loss: bool = True


@dataclass
class ContrastiveHeadCfg:
    proj_hidden_dim: int = 2048
    proj_out_dim: int = 256
    proj_num_layers: int = 3
    pred_hidden_dim: int = 2048  # MoCo v3 only
    temperature: float = 0.2  # MoCo v3; SimCLR overrides to 0.1
    moco_momentum: tuple[float, float] = (0.99, 1.0)
    freeze_patch_embed: bool = True  # MoCo v3's own fix for ViT training collapse
    symmetric_loss: bool = True


@dataclass
class AugCfg:
    """Pretraining view generation.

    I-JEPA/MAE defaults mirror the released i-jepa configs, where
    `use_horizontal_flip`, `use_color_distortion` and `use_gaussian_blur` are all
    False — RandomResizedCrop is genuinely the only augmentation.
    """

    crop_scale: tuple[float, float] = (0.3, 1.0)
    horizontal_flip: bool = False
    color_jitter_strength: float = 0.0
    color_distortion: bool = False
    gaussian_blur: bool = False
    solarize: bool = False
    two_views: bool = False
    # Corpus statistics; overwritten from configs/norm_stats.json once computed.
    mean: tuple[float, float, float] = (0.485, 0.456, 0.406)
    std: tuple[float, float, float] = (0.229, 0.224, 0.225)


@dataclass
class RuntimeCfg:
    seed: int = 0
    num_workers: int = 3  # 4 vCPUs on Kaggle; leave one for the main process
    prefetch_factor: int = 4
    persistent_workers: bool = True
    pin_memory: bool = True
    session_guard_hours: float = 7.5
    ckpt_push_minutes: float = 45.0
    log_every_steps: int = 50
    ckpt_dataset_slug: str = ""  # e.g. "morsalin101/jepa-thesis-ckpt"
    weights_dataset_slug: str = ""


@dataclass
class PretrainCfg:
    method: str = "ijepa"  # ijepa | mae | simclr | mocov3
    model: ModelCfg = field(default_factory=ModelCfg)
    optim: OptimCfg = field(default_factory=OptimCfg)
    mask: MaskCfg = field(default_factory=MaskCfg)
    jepa: JEPAHeadCfg = field(default_factory=JEPAHeadCfg)
    mae: MAEHeadCfg = field(default_factory=MAEHeadCfg)
    contrastive: ContrastiveHeadCfg = field(default_factory=ContrastiveHeadCfg)
    aug: AugCfg = field(default_factory=AugCfg)
    runtime: RuntimeCfg = field(default_factory=RuntimeCfg)

    @property
    def run_id(self) -> str:
        m, o = self.model, self.optim
        return f"{self.method}_{m.arch}_p{m.patch_size}_{m.img_size}_ep{o.epochs}_bs{o.global_batch}_s{self.runtime.seed}"


@dataclass
class SegCfg:
    """Segmentation fine-tuning. Held identical across every encoder, including
    random-init — that identity is what makes the comparison a comparison."""

    encoder: str = "ijepa"  # ijepa | mae | simclr | mocov3 | random | imagenet
    decoder: str = "segformer"  # segformer | unet  (unet kept as a decoder ablation)
    model: ModelCfg = field(default_factory=lambda: ModelCfg(img_size=352))
    fpn_layers: tuple[int, int, int, int] = (2, 5, 8, 11)  # 0-indexed blocks {3,6,9,12}
    decoder_embed_dim: int = 256
    epochs: int = 100
    batch_size: int = 16
    enc_lr: float = 1.0e-4
    dec_lr: float = 1.0e-3
    layer_decay: float = 0.75
    weight_decay: float = 0.05
    warmup_epochs: int = 5
    grad_clip: float = 1.0
    early_stop_patience: int = 20
    bce_weight: float = 0.5
    dice_weight: float = 0.5
    label_fraction: float = 1.0  # low-label ablation: 0.1 / 0.25 / 0.5 / 1.0
    split: str = "800_100_100"  # or "880_120"
    pretrained_ckpt: str = ""
    runtime: RuntimeCfg = field(default_factory=RuntimeCfg)

    @property
    def run_id(self) -> str:
        frac = "" if self.label_fraction == 1.0 else f"_lf{self.label_fraction}"
        return f"seg_{self.encoder}_{self.decoder}_{self.model.img_size}{frac}_s{self.runtime.seed}"


# ------------------------------------------------------------------ YAML overlay

def _coerce(value: Any, ref: Any) -> Any:
    """Make YAML values match the dataclass field's type where it matters.

    YAML gives lists; several fields are tuples that end up in the config hash, and
    `[0.15, 0.2] != (0.15, 0.2)` would make an otherwise-identical run refuse to resume.
    """
    if isinstance(ref, tuple) and isinstance(value, list):
        return tuple(value)
    if isinstance(ref, float) and isinstance(value, int):
        return float(value)
    if isinstance(ref, Path):
        return Path(value)
    return value


def apply_overrides(cfg: Any, data: dict[str, Any], path: str = "") -> None:
    """Recursively overlay a dict onto a dataclass instance, in place.

    Unknown keys raise. A silently ignored typo in a config file is a whole wasted
    Kaggle session.
    """
    known = {f.name for f in fields(cfg)}
    for key, value in data.items():
        where = f"{path}{key}"
        if key not in known:
            raise KeyError(f"unknown config key {where!r} for {type(cfg).__name__}")
        current = getattr(cfg, key)
        if is_dataclass(current) and isinstance(value, dict):
            apply_overrides(current, value, path=f"{where}.")
        else:
            setattr(cfg, key, _coerce(value, current))


def load_yaml(name_or_path: str | os.PathLike[str]) -> dict[str, Any]:
    p = Path(name_or_path)
    if not p.is_absolute() and not p.exists():
        p = REPO_ROOT / "configs" / p
    if not p.exists():
        raise FileNotFoundError(f"config file not found: {p}")
    import yaml

    with open(p) as f:
        return yaml.safe_load(f) or {}


def build_pretrain_cfg(
    method: str,
    config_file: str | os.PathLike[str] | None = None,
    overrides: dict[str, Any] | None = None,
) -> PretrainCfg:
    cfg = PretrainCfg(method=method)
    apply_overrides(cfg, load_yaml(config_file or f"pretrain_{method}.yaml"))
    if overrides:
        apply_overrides(cfg, overrides)
    if cfg.method != method:
        raise ValueError(f"config declares method={cfg.method!r} but {method!r} was requested")
    return cfg


def build_seg_cfg(
    config_file: str | os.PathLike[str] | None = None,
    overrides: dict[str, Any] | None = None,
) -> SegCfg:
    cfg = SegCfg()
    apply_overrides(cfg, load_yaml(config_file or "segment_segformer.yaml"))
    if overrides:
        apply_overrides(cfg, overrides)
    return cfg


# -------------------------------------------------------------------- hashing

def config_dict(cfg: Any) -> dict[str, Any]:
    """Dataclass -> plain JSON-able dict, tuples flattened to lists."""

    def enc(o: Any) -> Any:
        if isinstance(o, Path):
            return str(o)
        if isinstance(o, tuple):
            return list(o)
        return o

    return json.loads(json.dumps(asdict(cfg), default=enc, sort_keys=True))


def config_hash(cfg: Any, ignore: Sequence[str] = ("runtime",)) -> str:
    """Stable hash of the experiment-defining config.

    `runtime` is excluded because num_workers / push cadence / guard hours legitimately
    differ between sessions and must not block a resume. Everything else must match or
    the checkpoint is from a different experiment.
    """
    d = config_dict(cfg)
    for k in ignore:
        d.pop(k, None)
    # seed does define the experiment, so keep it even though it lives under runtime
    seed = getattr(getattr(cfg, "runtime", None), "seed", None)
    if seed is not None:
        d["_seed"] = seed
    return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:16]


def describe_env() -> str:
    amp = amp_config()
    return (
        f"[env] kaggle={on_kaggle()}  device={amp.device}  gpu={amp.name} "
        f"(sm_{amp.sm})  autocast={amp.dtype}  scaler={amp.use_scaler}"
    )


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/data/__init__.py
from src.data.hyperkvasir import HyperKvasirUnlabeled, HyperKvasirLabeled
from src.data.kvasir_seg import KvasirSegDataset
from src.data.transforms import (
    GPUAugment,
    TwoViewTransform,
    make_pretrain_transform,
    make_seg_transforms,
)

__all__ = [
    "HyperKvasirUnlabeled",
    "HyperKvasirLabeled",
    "KvasirSegDataset",
    "GPUAugment",
    "TwoViewTransform",
    "make_pretrain_transform",
    "make_seg_transforms",
]


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/data/transforms.py
"""View generation for pretraining and segmentation.

The split between CPU and GPU work here is a throughput decision, not a style one.
Kaggle gives 4 vCPUs. At the ~200 img/s a T4 x2 can push through SimCLR, the loader must
produce ~400 augmented 224px views per second; PIL colour-jitter plus Gaussian blur cost
~3 ms/view, which alone caps a 3-worker loader near 1000 views/s and in practice much
lower once JPEG decode is included. So:

    CPU workers:  decode -> RandomResizedCrop -> flip -> uint8 tensor   (cheap)
    GPU, batched: colour jitter -> grayscale -> blur -> solarize -> normalize

I-JEPA and MAE do not need any of this: the released i-jepa configs set
`use_horizontal_flip`, `use_color_distortion` and `use_gaussian_blur` all to False, so
RandomResizedCrop really is the only augmentation. That asymmetry between methods is
intentional and is documented as such in the thesis — equalising augmentation across
four objectives would misrepresent SimCLR and MoCo, whose augmentations *are* the method.
"""
from __future__ import annotations

import random
from typing import Callable, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# ------------------------------------------------------------------ pretraining


class TwoViewTransform:
    """Produce two independently augmented views of one image."""

    def __init__(self, view1: Callable, view2: Callable | None = None) -> None:
        self.view1 = view1
        self.view2 = view2 or view1

    def __call__(self, img):
        return self.view1(img), self.view2(img)


def make_pretrain_transform(
    img_size: int,
    crop_scale: Sequence[float] = (0.3, 1.0),
    horizontal_flip: bool = False,
    to_uint8: bool = False,
    mean: Sequence[float] = IMAGENET_MEAN,
    std: Sequence[float] = IMAGENET_STD,
):
    """CPU-side pretraining transform.

    :param to_uint8: return a uint8 CHW tensor instead of a normalised float tensor.
        Used for the contrastive methods, whose photometric augmentation and
        normalisation happen on the GPU in `GPUAugment`. uint8 also makes the
        worker->main-process transfer 4x smaller.
    """
    ops: list = [
        T.RandomResizedCrop(
            img_size, scale=tuple(crop_scale), interpolation=InterpolationMode.BICUBIC
        )
    ]
    if horizontal_flip:
        ops.append(T.RandomHorizontalFlip())
    if to_uint8:
        ops.append(T.PILToTensor())  # uint8 CHW, no scaling
    else:
        ops += [T.ToTensor(), T.Normalize(mean, std)]
    return T.Compose(ops)


class GPUAugment(nn.Module):
    """Batched photometric augmentation on uint8 CUDA tensors.

    Applied per-sample (each image in the batch gets its own random parameters) but
    vectorised where possible. Runs after the batch reaches the GPU, so it costs GPU
    time we have rather than CPU time we do not.
    """

    def __init__(
        self,
        color_jitter_strength: float = 0.4,
        color_distortion: bool = True,
        grayscale_p: float = 0.2,
        jitter_p: float = 0.8,
        gaussian_blur_p: float = 0.5,
        solarize_p: float = 0.0,
        horizontal_flip: bool = True,
        mean: Sequence[float] = IMAGENET_MEAN,
        std: Sequence[float] = IMAGENET_STD,
    ) -> None:
        super().__init__()
        s = color_jitter_strength
        self.brightness = 0.8 * s
        self.contrast = 0.8 * s
        self.saturation = 0.8 * s
        self.hue = 0.2 * s
        self.color_distortion = color_distortion
        self.grayscale_p = grayscale_p
        self.jitter_p = jitter_p
        self.blur_p = gaussian_blur_p
        self.solarize_p = solarize_p
        self.hflip = horizontal_flip
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))
        self.register_buffer("gray_w", torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1))

    @staticmethod
    def _rand(b: int, lo: float, hi: float, device) -> torch.Tensor:
        return torch.empty(b, 1, 1, 1, device=device).uniform_(lo, hi)

    def _blur(self, x: torch.Tensor) -> torch.Tensor:
        """Depthwise Gaussian blur with one shared sigma per batch.

        SimCLR draws sigma per-image; sharing it within a batch keeps this a single
        grouped conv instead of a Python loop, and the sigma still varies every step.
        """
        b, c, h, w = x.shape
        sigma = random.uniform(0.1, 2.0)
        k = max(3, int(0.1 * h) | 1)  # odd kernel, ~10% of image side, as in SimCLR
        coords = torch.arange(k, device=x.device, dtype=x.dtype) - (k - 1) / 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        g = g / g.sum()
        x = F.conv2d(x, g.view(1, 1, 1, k).expand(c, 1, 1, k), padding=(0, k // 2), groups=c)
        x = F.conv2d(x, g.view(1, 1, k, 1).expand(c, 1, k, 1), padding=(k // 2, 0), groups=c)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """uint8 or float [B,3,H,W] in [0,255] / [0,1] -> normalised float."""
        if x.dtype == torch.uint8:
            x = x.float().div_(255.0)
        b, device = x.shape[0], x.device

        if self.hflip:
            flip = torch.rand(b, device=device) < 0.5
            x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)

        if self.color_distortion:
            apply = (torch.rand(b, 1, 1, 1, device=device) < self.jitter_p).to(x.dtype)
            y = x
            y = y * (1 + apply * (self._rand(b, -self.brightness, self.brightness, device)))
            mean_c = y.mean(dim=(1, 2, 3), keepdim=True)
            y = mean_c + (y - mean_c) * (1 + apply * self._rand(b, -self.contrast, self.contrast, device))
            gray = (y * self.gray_w).sum(dim=1, keepdim=True)
            y = gray + (y - gray) * (1 + apply * self._rand(b, -self.saturation, self.saturation, device))
            # Hue is approximated by a channel-wise shift; a true HSV rotation costs a
            # full colour-space round-trip for a perceptually similar perturbation.
            y = y + apply * self._rand(b, -self.hue, self.hue, device)
            x = y.clamp_(0, 1)

            to_gray = (torch.rand(b, 1, 1, 1, device=device) < self.grayscale_p).to(x.dtype)
            gray = (x * self.gray_w).sum(dim=1, keepdim=True).expand_as(x)
            x = to_gray * gray + (1 - to_gray) * x

        if self.blur_p > 0 and random.random() < self.blur_p:
            x = self._blur(x)

        if self.solarize_p > 0:
            sol = (torch.rand(b, 1, 1, 1, device=device) < self.solarize_p).to(x.dtype)
            x = sol * torch.where(x < 0.5, x, 1.0 - x) + (1 - sol) * x

        return (x - self.mean) / self.std


# ---------------------------------------------------------------- segmentation


class SegTransform:
    """Joint image+mask transform. Every geometric op is applied to both.

    Validation and test use `train=False`, which is *only* resize + normalise. The
    earlier version of this project shared one augmented dataset between train and val
    via a `Subset`, so validation images were randomly flipped — making the val Dice
    noisy and the model-selection signal worse than it looks.
    """

    def __init__(
        self,
        img_size: int,
        train: bool,
        mean: Sequence[float] = IMAGENET_MEAN,
        std: Sequence[float] = IMAGENET_STD,
        scale_range: tuple[float, float] = (0.75, 1.25),
        rotation_deg: float = 90.0,
        color_jitter: float = 0.2,
    ) -> None:
        self.img_size = img_size
        self.train = train
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std = torch.tensor(std).view(3, 1, 1)
        self.scale_range = scale_range
        self.rotation_deg = rotation_deg
        self.jitter = T.ColorJitter(color_jitter, color_jitter, color_jitter, color_jitter / 4)

    def __call__(self, img, mask):
        import torchvision.transforms.functional as TF

        if self.train:
            if random.random() < 0.5:
                img, mask = TF.hflip(img), TF.hflip(mask)
            if random.random() < 0.5:
                img, mask = TF.vflip(img), TF.vflip(mask)
            if self.rotation_deg > 0 and random.random() < 0.5:
                angle = random.uniform(-self.rotation_deg, self.rotation_deg)
                img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            if self.scale_range and random.random() < 0.5:
                s = random.uniform(*self.scale_range)
                side = max(8, int(self.img_size * s))
                img = TF.resize(img, [side, side], InterpolationMode.BILINEAR)
                mask = TF.resize(mask, [side, side], InterpolationMode.NEAREST)

        img = TF.resize(img, [self.img_size, self.img_size], InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.img_size, self.img_size], InterpolationMode.NEAREST)

        if self.train:
            img = self.jitter(img)

        img = TF.to_tensor(img)
        img = (img - self.mean) / self.std
        mask = TF.to_tensor(mask)
        if mask.shape[0] > 1:  # some Kvasir-SEG masks are saved as RGB
            mask = mask[:1]
        mask = (mask > 0.5).float()
        return img, mask


def make_seg_transforms(img_size: int, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """(train_transform, eval_transform) pair."""
    return (
        SegTransform(img_size, train=True, mean=mean, std=std),
        SegTransform(img_size, train=False, mean=mean, std=std),
    )


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/data/hyperkvasir.py
"""HyperKvasir datasets — the SSL pretraining corpus.

The unlabeled split is 99,417 GI endoscopy frames with no annotations of any kind, which
is exactly what self-supervised pretraining wants and exactly what Kvasir-SEG's 1000
images cannot provide. The labeled split (10,662 images, 23 classes) is never trained
on; it is used only for k-NN and linear probing, which is how we get a representation
quality number before spending GPU hours on segmentation.

The file list is cached to disk on first scan. Walking ~100k files on Kaggle's network
filesystem takes ~40 s, and paying that once per worker per epoch is a real cost.
"""
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Callable, Sequence

from PIL import Image
from torch.utils.data import Dataset

from src.config import IMAGE_EXTS, resolve_dataset_dir

# Endoscopy frames are frequently large and occasionally slightly truncated; without
# this a single bad file aborts an 8-hour run at a random epoch.
Image.MAX_IMAGE_PIXELS = None
try:
    from PIL import ImageFile

    ImageFile.LOAD_TRUNCATED_IMAGES = True
except ImportError:
    pass


def scan_images(root: Path, cache: Path | None = None, exclude: set[str] | None = None) -> list[Path]:
    """Recursively list image files under `root`, sorted, with an optional path cache."""
    if cache is not None and cache.is_file():
        try:
            names = json.loads(cache.read_text())
            paths = [root / n for n in names]
            if paths and paths[0].exists():
                if exclude:
                    paths = [p for p in paths if p.name not in exclude]
                return paths
        except (json.JSONDecodeError, OSError):
            pass  # stale or corrupt cache; fall through to a fresh scan

    paths = sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    if cache is not None:
        try:
            cache.parent.mkdir(parents=True, exist_ok=True)
            cache.write_text(json.dumps([str(p.relative_to(root)) for p in paths]))
        except OSError:
            pass  # read-only /kaggle/input mount; caching is best-effort

    if exclude:
        paths = [p for p in paths if p.name not in exclude]
    return paths


class HyperKvasirUnlabeled(Dataset):
    """Unlabeled pretraining corpus. Returns a transformed image (or a view pair)."""

    def __init__(
        self,
        root: str | os.PathLike[str] | None = None,
        transform: Callable | None = None,
        exclude_file: str | os.PathLike[str] | None = None,
        cache_dir: str | os.PathLike[str] | None = None,
    ) -> None:
        self.root = Path(root) if root is not None else resolve_dataset_dir("hyperkvasir")
        self.transform = transform

        # Filenames removed by scripts/dedup_phash.py because they near-duplicate a
        # Kvasir-SEG image. Leaving them in would leak the downstream test set into
        # pretraining — the single most damaging objection this thesis can face.
        exclude: set[str] = set()
        if exclude_file is not None and Path(exclude_file).is_file():
            exclude = {
                line.strip()
                for line in Path(exclude_file).read_text().splitlines()
                if line.strip() and not line.startswith("#")
            }

        cache = Path(cache_dir) / "hyperkvasir_files.json" if cache_dir else None
        self.samples = scan_images(self.root, cache=cache, exclude=exclude)
        self.n_excluded = len(exclude)
        if not self.samples:
            raise FileNotFoundError(f"no images found under {self.root}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img = Image.open(self.samples[idx]).convert("RGB")
        return self.transform(img) if self.transform else img

    def describe(self) -> str:
        return (
            f"HyperKvasirUnlabeled: {len(self.samples)} images from {self.root}"
            + (f" ({self.n_excluded} excluded as Kvasir-SEG near-duplicates)" if self.n_excluded else "")
        )


class HyperKvasirLabeled(Dataset):
    """Labeled split, used only for k-NN / linear probing.

    Class label is the parent directory name, which is how HyperKvasir ships its
    `labeled-images/<anatomical-landmarks|pathological-findings>/<class>/` tree.
    """

    def __init__(
        self,
        root: str | os.PathLike[str] | None = None,
        transform: Callable | None = None,
        classes: Sequence[str] | None = None,
    ) -> None:
        self.root = Path(root) if root is not None else resolve_dataset_dir("hyperkvasir_labeled")
        self.transform = transform
        paths = scan_images(self.root)
        if not paths:
            raise FileNotFoundError(f"no images found under {self.root}")

        names = sorted({p.parent.name for p in paths}) if classes is None else list(classes)
        self.classes = names
        self.class_to_idx = {c: i for i, c in enumerate(names)}
        self.samples = [(p, self.class_to_idx[p.parent.name]) for p in paths if p.parent.name in self.class_to_idx]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return (self.transform(img) if self.transform else img), label


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/data/kvasir_seg.py
"""Kvasir-SEG: 1000 polyp images with binary masks.

Kaggle layout is `<root>/Kvasir-SEG/{images,masks}/*.jpg`, with the mask carrying the
same filename as its image.

Two things this dataset does that the metric code depends on:

* It can return the **native (H, W)** of each image. Every metric in the thesis is
  computed at native resolution — you upsample the *logits* back to the original size
  and threshold there. Computing Dice at 352x352 against a downsampled ground truth
  inflates it by roughly 1-2 points and is not comparable to published numbers.
* Train and eval use *different transform objects*, never a `Subset` of one augmented
  dataset, so validation is deterministic.
"""
from __future__ import annotations

import os
from pathlib import Path
from typing import Callable, Sequence

from PIL import Image
from torch.utils.data import Dataset

from src.config import IMAGE_EXTS, REPO_ROOT, resolve_dataset_dir

try:
    from PIL import ImageFile

    ImageFile.LOAD_TRUNCATED_IMAGES = True
except ImportError:
    pass


def find_image_and_mask_dirs(root: Path) -> tuple[Path, Path]:
    """Locate the images/ and masks/ pair under a Kvasir-SEG root."""
    for base in (root, root / "Kvasir-SEG", root / "kvasir-seg"):
        img_d, msk_d = base / "images", base / "masks"
        if img_d.is_dir() and msk_d.is_dir():
            return img_d, msk_d
    # Last resort: any descendant pair named images/ + masks/
    for cand in root.rglob("images"):
        if cand.is_dir() and (cand.parent / "masks").is_dir():
            return cand, cand.parent / "masks"
    raise FileNotFoundError(f"could not find images/ and masks/ under {root}")


def read_split(name: str, split: str, splits_dir: Path | None = None) -> list[str]:
    """Read `splits/<name>/<split>.txt` -> list of stems.

    Split files are committed to the repo so any run, on any machine, uses byte-identical
    splits. Regenerate them with `python -m src.data.splits`.
    """
    d = (splits_dir or REPO_ROOT / "splits") / name
    p = d / f"{split}.txt"
    if not p.is_file():
        raise FileNotFoundError(
            f"split file {p} not found. Generate it with:  python -m src.data.splits"
        )
    return [ln.strip() for ln in p.read_text().splitlines() if ln.strip() and not ln.startswith("#")]


class KvasirSegDataset(Dataset):
    def __init__(
        self,
        root: str | os.PathLike[str] | None = None,
        stems: Sequence[str] | None = None,
        transform: Callable | None = None,
        return_native_size: bool = False,
        label_fraction: float = 1.0,
        seed: int = 0,
    ) -> None:
        """
        :param stems: filename stems to include (from a split file). None = all.
        :param label_fraction: keep only this fraction, for the low-label ablation.
            Subsampling is seeded and *nested* — the 10% set is a subset of the 25% set —
            so the ablation curve is monotone in data, not confounded by which images
            happened to be drawn.
        """
        self.root = Path(root) if root is not None else resolve_dataset_dir("kvasir_seg")
        self.image_dir, self.mask_dir = find_image_and_mask_dirs(self.root)
        self.transform = transform
        self.return_native_size = return_native_size

        by_stem = {p.stem: p for p in sorted(self.image_dir.iterdir()) if p.suffix.lower() in IMAGE_EXTS}
        if stems is not None:
            missing = [s for s in stems if s not in by_stem]
            if missing:
                raise FileNotFoundError(
                    f"{len(missing)} stems from the split are not in {self.image_dir} "
                    f"(first few: {missing[:5]})"
                )
            selected = [by_stem[s] for s in stems]
        else:
            selected = list(by_stem.values())

        if label_fraction < 1.0:
            import random

            rng = random.Random(seed)
            order = list(range(len(selected)))
            rng.shuffle(order)
            keep = max(1, int(round(len(selected) * label_fraction)))
            selected = [selected[i] for i in sorted(order[:keep])]

        self.samples: list[tuple[Path, Path]] = []
        for img_p in selected:
            mask_p = self._find_mask(img_p.stem)
            if mask_p is None:
                raise FileNotFoundError(f"no mask for {img_p.name} in {self.mask_dir}")
            self.samples.append((img_p, mask_p))

        if not self.samples:
            raise RuntimeError(f"no image/mask pairs found under {self.root}")

    def _find_mask(self, stem: str) -> Path | None:
        for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
            p = self.mask_dir / f"{stem}{ext}"
            if p.is_file():
                return p
        for suffix in ("_mask", "_segmentation"):
            for ext in (".jpg", ".png"):
                p = self.mask_dir / f"{stem}{suffix}{ext}"
                if p.is_file():
                    return p
        return None

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_p, mask_p = self.samples[idx]
        img = Image.open(img_p).convert("RGB")
        mask = Image.open(mask_p).convert("L")
        native = (img.height, img.width)

        if self.transform is not None:
            img, mask = self.transform(img, mask)

        if self.return_native_size:
            return img, mask, native[0], native[1], idx
        return img, mask

    def stems(self) -> list[str]:
        return [p.stem for p, _ in self.samples]

    def native_mask_path(self, idx: int) -> Path:
        """Full-resolution mask path, for metrics computed at native resolution."""
        return self.samples[idx][1]

    def describe(self) -> str:
        return f"KvasirSeg: {len(self.samples)} pairs from {self.image_dir}"


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/data/splits.py
"""Generate Kvasir-SEG train/val/test splits.

Three properties the split must have, in decreasing order of how badly getting them
wrong would hurt:

1. **Group-aware.** Kvasir-SEG frames are extracted from colonoscopy *videos*, so the
   dataset contains near-duplicate views of the same polyp. If two frames of one polyp
   land in train and test, test Dice is measuring memorisation. We perceptual-hash every
   image, take connected components under a Hamming threshold, and keep whole components
   in one split.
2. **Stratified.** Polyp masks span roughly 0.8% to 62% of image area, and native
   resolutions range from 332x487 to 1920x1072. An unstratified 100-image test set can
   easily over-represent large, easy polyps and flatter every method equally.
3. **Committed to the repo.** The output `.txt` files are version-controlled so every
   run on every machine uses byte-identical splits.

We emit two split sets:
  * `800_100_100` — the primary. The test set is used exactly once, at the very end.
  * `880_120`     — the literature-standard split, so our numbers can be placed against
                    published Kvasir-SEG results. Note that the common practice of
                    selecting on those 120 *and* reporting on them is precisely the
                    leakage the primary split avoids.

Run:  python -m src.data.splits
"""
from __future__ import annotations

import argparse
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image

from src.config import REPO_ROOT, resolve_dataset_dir
from src.data.kvasir_seg import find_image_and_mask_dirs

HASH_SIZE = 8


def _dct_matrix(n: int) -> np.ndarray:
    k = np.arange(n).reshape(-1, 1)
    i = np.arange(n).reshape(1, -1)
    m = np.cos(np.pi * (2 * i + 1) * k / (2 * n))
    m[0] /= np.sqrt(2)
    return m * np.sqrt(2 / n)


def phash(img: Image.Image, hash_size: int = HASH_SIZE, highfreq_factor: int = 4) -> int:
    """Perceptual hash (DCT-based), returned as a 64-bit int.

    More robust than a difference hash to the brightness and scale changes that separate
    two frames of the same polyp, which is exactly the case we need to catch.
    """
    size = hash_size * highfreq_factor
    px = np.asarray(img.convert("L").resize((size, size), Image.Resampling.LANCZOS), dtype=np.float64)
    d = _dct_matrix(size)
    coeffs = d @ px @ d.T
    low = coeffs[:hash_size, :hash_size]
    med = np.median(low[1:, 1:])  # skip DC, which only encodes mean brightness
    bits = (low > med).flatten()
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


class UnionFind:
    def __init__(self, n: int) -> None:
        self.parent = list(range(n))

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


def group_near_duplicates(hashes: list[int], threshold: int = 6) -> list[int]:
    """Connected components under Hamming <= threshold. Returns a group id per item.

    O(n^2) at n=1000 is 500k integer XORs — under a second, so no need for an LSH index.
    """
    uf = UnionFind(len(hashes))
    for i in range(len(hashes)):
        for j in range(i + 1, len(hashes)):
            if hamming(hashes[i], hashes[j]) <= threshold:
                uf.union(i, j)
    roots = {}
    out = []
    for i in range(len(hashes)):
        r = uf.find(i)
        if r not in roots:
            roots[r] = len(roots)
        out.append(roots[r])
    return out


def scan_dataset(root: Path) -> list[dict]:
    """Per-image record: stem, phash, mask area fraction, native resolution."""
    image_dir, mask_dir = find_image_and_mask_dirs(root)
    recs = []
    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    for n, img_p in enumerate(images):
        if n % 200 == 0:
            print(f"[splits] hashing {n}/{len(images)} ...")
        img = Image.open(img_p)
        mask_p = next(
            (mask_dir / f"{img_p.stem}{e}" for e in (".jpg", ".png", ".jpeg") if (mask_dir / f"{img_p.stem}{e}").is_file()),
            None,
        )
        if mask_p is None:
            raise FileNotFoundError(f"no mask for {img_p.name}")
        m = np.asarray(Image.open(mask_p).convert("L"), dtype=np.uint8) > 127
        recs.append(
            {
                "stem": img_p.stem,
                "phash": phash(img),
                "area": float(m.mean()),
                "h": img.height,
                "w": img.width,
            }
        )
    return recs


def _strata(recs: list[dict]) -> list[str]:
    """Stratum label = mask-area quartile x resolution tercile."""
    areas = np.array([r["area"] for r in recs])
    pixels = np.array([r["h"] * r["w"] for r in recs])
    aq = np.digitize(areas, np.quantile(areas, [0.25, 0.5, 0.75]))
    rq = np.digitize(pixels, np.quantile(pixels, [1 / 3, 2 / 3]))
    return [f"a{a}r{r}" for a, r in zip(aq, rq)]


def make_splits(
    recs: list[dict],
    sizes: dict[str, int],
    seed: int = 0,
    dup_threshold: int = 6,
) -> tuple[dict[str, list[str]], dict]:
    """Assign whole duplicate-groups to splits, filling stratum quotas proportionally."""
    groups = group_near_duplicates([r["phash"] for r in recs], dup_threshold)
    strata = _strata(recs)
    for r, g, s in zip(recs, groups, strata):
        r["group"] = g
        r["stratum"] = s

    members: dict[int, list[dict]] = defaultdict(list)
    for r in recs:
        members[r["group"]].append(r)

    # A group's stratum is its most common member stratum.
    def group_stratum(gid: int) -> str:
        counts: dict[str, int] = defaultdict(int)
        for r in members[gid]:
            counts[r["stratum"]] += 1
        return max(counts.items(), key=lambda kv: (kv[1], kv[0]))[0]

    rng = np.random.RandomState(seed)
    gids = sorted(members)
    rng.shuffle(gids)

    n_total = len(recs)
    # Per-stratum quota for each split, proportional to that stratum's overall share.
    stratum_counts: dict[str, int] = defaultdict(int)
    for r in recs:
        stratum_counts[r["stratum"]] += 1
    quota = {
        split: {s: stratum_counts[s] * n / n_total for s in stratum_counts}
        for split, n in sizes.items()
    }

    assigned: dict[str, list[str]] = {k: [] for k in sizes}
    filled: dict[str, dict[str, float]] = {k: defaultdict(float) for k in sizes}
    counts = {k: 0 for k in sizes}

    # Largest groups first — they are the least flexible to place.
    for gid in sorted(gids, key=lambda g: -len(members[g])):
        s = group_stratum(gid)
        size = len(members[gid])
        # Prefer the split furthest below its quota for this stratum, then overall.
        best = min(
            sizes,
            key=lambda k: (
                counts[k] + size > sizes[k],  # hard cap first
                filled[k][s] - quota[k][s],
                counts[k] - sizes[k],
            ),
        )
        assigned[best] += [r["stem"] for r in members[gid]]
        filled[best][s] += size
        counts[best] += size

    stats = {
        "seed": seed,
        "dup_threshold": dup_threshold,
        "n_images": n_total,
        "n_groups": len(members),
        "largest_group": max(len(v) for v in members.values()),
        "multi_image_groups": sum(1 for v in members.values() if len(v) > 1),
        "counts": {k: len(v) for k, v in assigned.items()},
        "stratum_distribution": {
            k: dict(sorted(((s, int(c)) for s, c in filled[k].items()))) for k in assigned
        },
    }
    return {k: sorted(v) for k, v in assigned.items()}, stats


def write_splits(splits: dict[str, list[str]], stats: dict, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, stems in splits.items():
        (out_dir / f"{name}.txt").write_text("\n".join(stems) + "\n")
    (out_dir / "stats.json").write_text(json.dumps(stats, indent=2) + "\n")
    print(f"[splits] wrote {out_dir}: " + ", ".join(f"{k}={len(v)}" for k, v in splits.items()))


def main() -> None:
    ap = argparse.ArgumentParser(description="Generate Kvasir-SEG splits")
    ap.add_argument("--root", default=None, help="Kvasir-SEG root (auto-detected if omitted)")
    ap.add_argument("--out", default=str(REPO_ROOT / "splits"))
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--dup-threshold", type=int, default=6)
    args = ap.parse_args()

    root = Path(args.root) if args.root else resolve_dataset_dir("kvasir_seg")
    print(f"[splits] scanning {root}")
    recs = scan_dataset(root)
    print(f"[splits] {len(recs)} images hashed")

    out = Path(args.out)
    primary, stats = make_splits(
        recs, {"train": 800, "val": 100, "test": 100}, args.seed, args.dup_threshold
    )
    write_splits(primary, stats, out / "800_100_100")
    print(
        f"[splits] {stats['n_groups']} duplicate-groups, "
        f"{stats['multi_image_groups']} with >1 image, largest={stats['largest_group']}"
    )

    secondary, stats2 = make_splits(
        recs, {"train": 880, "val": 120}, args.seed, args.dup_threshold
    )
    secondary["test"] = list(secondary["val"])  # literature convention: val == test here
    write_splits(secondary, stats2, out / "880_120")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/masks/__init__.py
from src.masks.multiblock import MaskCollator
from src.masks.utils import apply_masks, repeat_interleave_batch

__all__ = ["MaskCollator", "apply_masks", "repeat_interleave_batch"]


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/masks/utils.py
"""Mask application helpers.

Ported verbatim in behaviour from reference/ijepa/src/masks/utils.py and
reference/ijepa/src/utils/tensors.py. Both functions stack along the *batch* dimension
rather than adding a mask axis, which is what makes the rest of the model mask-agnostic:
the encoder just sees a bigger batch of shorter sequences.
"""
from __future__ import annotations

import torch


def apply_masks(x: torch.Tensor, masks: list[torch.Tensor] | torch.Tensor) -> torch.Tensor:
    """Keep only the indexed patches, concatenating masks along the batch dim.

    :param x: [B, N, D]
    :param masks: list of M index tensors, each [B, K]
    :return: [M*B, K, D]
    """
    if not isinstance(masks, list):
        masks = [masks]
    out = []
    for m in masks:
        idx = m.unsqueeze(-1).expand(-1, -1, x.size(-1))
        out.append(torch.gather(x, dim=1, index=idx))
    return torch.cat(out, dim=0)


def repeat_interleave_batch(x: torch.Tensor, B: int, repeat: int) -> torch.Tensor:
    """Repeat each length-B block of the batch `repeat` times, keeping blocks contiguous.

    With 4 target masks and 1 context mask, `apply_masks` gives targets ordered
    [mask0(B), mask1(B), mask2(B), mask3(B)]. The predictor output is ordered the same
    way but with the context repeated per target. This aligns the two so `smooth_l1`
    compares the right pairs — get it wrong and the loss still decreases, just against
    the wrong targets.
    """
    N = len(x) // B
    return torch.cat(
        [torch.cat([x[i * B : (i + 1) * B] for _ in range(repeat)], dim=0) for i in range(N)],
        dim=0,
    )


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/masks/multiblock.py
"""I-JEPA multi-block masking.

This is the part of I-JEPA that actually carries the method, and it is where the
previous version of this project diverged most: it sampled **one** target block and used
its exact set-complement as context. The paper samples **four** target blocks and a
**separately sampled, large** context block (85-100% of the image) from which the union
of the targets is then removed. Predicting four scattered blocks from a large,
independently-chosen context is what forces semantic rather than local-texture features.

Ported from reference/ijepa/src/masks/multiblock.py, preserving its behaviour including
two quirks worth knowing about:

* `_sample_block_size` draws **one** random number and uses it for *both* the area scale
  and the aspect ratio, so the two are perfectly correlated within a call. That looks
  like a bug, but it is what the released models were trained with, so we keep it.
* Block *sizes* are drawn once per batch from a seeded generator (shared across ranks
  via a step counter, so DDP ranks agree), while block *positions* use the global RNG
  and therefore differ per image.

One **deliberate deviation** from the reference: it draws positions with
`torch.randint(0, height - h)`, whose exclusive upper bound means a block can never
touch the bottom or right edge of the grid. We use `height - h + 1`, so every valid
position is reachable. On ImageNet that bias is cosmetic; on endoscopy it is not, since
polyps sit against the frame edge often enough to matter.

The batch-shared size plus truncation to the batch-wide `min_keep` is what lets masks
collate into rectangular tensors with no padding — the reason this implementation needs
no `key_padding_mask` anywhere.
"""
from __future__ import annotations

import math
from multiprocessing import Value

import torch


class MaskCollator:
    """Collate function producing (images, enc_masks, pred_masks) per batch.

    Returned masks are lists of index tensors:
      * `enc_masks`:  `num_enc_masks` tensors of shape [B, K_enc]
      * `pred_masks`: `num_pred_masks` tensors of shape [B, K_pred]
    """

    def __init__(
        self,
        input_size: int | tuple[int, int] = 224,
        patch_size: int = 16,
        enc_mask_scale: tuple[float, float] = (0.85, 1.0),
        pred_mask_scale: tuple[float, float] = (0.15, 0.2),
        aspect_ratio: tuple[float, float] = (0.75, 1.5),
        num_enc_masks: int = 1,
        num_pred_masks: int = 4,
        min_keep: int = 10,
        allow_overlap: bool = False,
    ) -> None:
        if not isinstance(input_size, tuple):
            input_size = (input_size, input_size)
        self.patch_size = patch_size
        self.height = input_size[0] // patch_size
        self.width = input_size[1] // patch_size
        self.enc_mask_scale = enc_mask_scale
        self.pred_mask_scale = pred_mask_scale
        self.aspect_ratio = aspect_ratio
        self.nenc = num_enc_masks
        self.npred = num_pred_masks
        self.min_keep = min_keep
        self.allow_overlap = allow_overlap
        # Shared across dataloader workers so every worker advances the same counter;
        # the seed derived from it keeps block sizes consistent within a batch.
        self._itr_counter = Value("i", -1)
        self._validate_min_keep()

    def _block_area(self, scale_frac: float, aspect: float) -> int:
        """Patch count of the block the sampler would produce for these parameters."""
        max_keep = int(self.height * self.width * scale_frac)
        h = int(round(math.sqrt(max(1, max_keep) * aspect)))
        w = int(round(math.sqrt(max(1, max_keep) / aspect)))
        while h >= self.height:
            h -= 1
        while w >= self.width:
            w -= 1
        return max(1, h) * max(1, w)

    def _min_block_area(self, scale: tuple[float, float], aspect: tuple[float, float]) -> int:
        """Smallest block this sampler can produce.

        `_sample_block_size` draws a single random number and uses it for both the area
        scale and the aspect ratio, so the two are perfectly correlated and the minimum
        is not necessarily at either endpoint. Sweeping the draw is exact and cheap.
        """
        return min(
            self._block_area(
                scale[0] + r * (scale[1] - scale[0]), aspect[0] + r * (aspect[1] - aspect[0])
            )
            for r in (i / 200 for i in range(201))
        )

    def _validate_min_keep(self) -> None:
        """Fail loudly at construction if `min_keep` is unsatisfiable.

        `_sample_block_mask` loops until it finds a mask with more than `min_keep`
        patches. For target blocks there are no `acceptable_regions` to relax, so if the
        block size itself can never exceed `min_keep` the loop spins forever with no
        output — the worst possible failure on a metered GPU session. This turns that
        into an error at startup.
        """
        grid = f"{self.height}x{self.width}={self.height * self.width} patches"
        for name, scale, aspect in (
            ("pred_mask_scale", self.pred_mask_scale, self.aspect_ratio),
            ("enc_mask_scale", self.enc_mask_scale, (1.0, 1.0)),
        ):
            smallest = self._min_block_area(scale, aspect)
            if smallest <= self.min_keep:
                raise ValueError(
                    f"min_keep={self.min_keep} is unsatisfiable: on a {grid} grid the "
                    f"smallest block from {name}={tuple(scale)} is {smallest} patches, so "
                    f"mask sampling would loop forever. Either lower min_keep below "
                    f"{smallest}, raise the input resolution, or lower the patch size."
                )

    def step(self) -> int:
        i = self._itr_counter
        with i.get_lock():
            i.value += 1
            return i.value

    def set_epoch(self, epoch: int, iters_per_epoch: int) -> None:
        """Align the internal counter after a resume.

        Without this, session 2 would restart mask sampling from counter 0 and repeat
        session 1's exact mask sequence. Harmless for correctness, but it wastes the
        stochasticity the method depends on.
        """
        with self._itr_counter.get_lock():
            self._itr_counter.value = epoch * iters_per_epoch - 1

    def _sample_block_size(
        self,
        generator: torch.Generator,
        scale: tuple[float, float],
        aspect_ratio_scale: tuple[float, float],
    ) -> tuple[int, int]:
        rand = torch.rand(1, generator=generator).item()
        min_s, max_s = scale
        mask_scale = min_s + rand * (max_s - min_s)
        max_keep = int(self.height * self.width * mask_scale)
        min_ar, max_ar = aspect_ratio_scale
        aspect = min_ar + rand * (max_ar - min_ar)

        h = int(round(math.sqrt(max_keep * aspect)))
        w = int(round(math.sqrt(max_keep / aspect)))
        # Strictly smaller than the grid: `torch.randint(0, height - h)` below needs a
        # non-empty range, and a full-height block would leave no room to translate.
        while h >= self.height:
            h -= 1
        while w >= self.width:
            w -= 1
        return max(1, h), max(1, w)

    def _sample_block_mask(
        self,
        b_size: tuple[int, int],
        acceptable_regions: list[torch.Tensor] | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Sample one rectangular block, optionally constrained away from other blocks.

        `acceptable_regions` are 0/1 grids (the complements of already-sampled target
        blocks). Multiplying the candidate by them removes overlap. If that leaves fewer
        than `min_keep` patches after 20 tries, the constraint is relaxed by dropping one
        region — otherwise a large context block against four targets can loop forever.
        """
        h, w = b_size

        def constrain(mask: torch.Tensor, tries: int) -> None:
            assert acceptable_regions is not None
            n = max(int(len(acceptable_regions) - tries), 0)
            for k in range(n):
                mask *= acceptable_regions[k]

        tries = 0
        timeout = og_timeout = 20
        attempts = 0
        # Belt-and-braces bound. `_validate_min_keep` already rules out the case where no
        # mask can ever satisfy min_keep; this catches anything that slips through with a
        # diagnostic instead of an unbounded spin.
        max_attempts = og_timeout * (len(acceptable_regions or ()) + 2) + 200
        top = left = 0
        mask_idx = torch.empty(0, dtype=torch.long)
        valid = False
        while not valid:
            top = int(torch.randint(0, self.height - h + 1, (1,)).item())
            left = int(torch.randint(0, self.width - w + 1, (1,)).item())
            mask = torch.zeros((self.height, self.width), dtype=torch.int32)
            mask[top : top + h, left : left + w] = 1
            if acceptable_regions is not None:
                constrain(mask, tries)
            mask_idx = torch.nonzero(mask.flatten()).squeeze(-1)
            valid = len(mask_idx) > self.min_keep
            if not valid:
                attempts += 1
                if attempts > max_attempts:
                    raise RuntimeError(
                        f"could not sample a {h}x{w} block with more than min_keep="
                        f"{self.min_keep} patches on a {self.height}x{self.width} grid "
                        f"after {attempts} attempts (constrained by "
                        f"{len(acceptable_regions or ())} region(s)). Lower min_keep or "
                        "raise the input resolution."
                    )
                timeout -= 1
                if timeout == 0:
                    tries += 1
                    timeout = og_timeout

        complement = torch.ones((self.height, self.width), dtype=torch.int32)
        complement[top : top + h, left : left + w] = 0
        return mask_idx, complement

    def __call__(self, batch):
        """Collate a batch and attach one mask set to it."""
        collated = torch.utils.data.default_collate(batch)
        B = len(batch)

        g = torch.Generator()
        g.manual_seed(self.step())
        p_size = self._sample_block_size(g, self.pred_mask_scale, self.aspect_ratio)
        # Context blocks are square by construction: aspect (1, 1).
        e_size = self._sample_block_size(g, self.enc_mask_scale, (1.0, 1.0))

        collated_pred: list[list[torch.Tensor]] = []
        collated_enc: list[list[torch.Tensor]] = []
        min_keep_pred = self.height * self.width
        min_keep_enc = self.height * self.width

        for _ in range(B):
            masks_p, complements = [], []
            for _ in range(self.npred):
                mask, comp = self._sample_block_mask(p_size)
                masks_p.append(mask)
                complements.append(comp)
                min_keep_pred = min(min_keep_pred, len(mask))
            collated_pred.append(masks_p)

            acceptable = None if self.allow_overlap else complements
            masks_e = []
            for _ in range(self.nenc):
                mask, _ = self._sample_block_mask(e_size, acceptable_regions=acceptable)
                masks_e.append(mask)
                min_keep_enc = min(min_keep_enc, len(mask))
            collated_enc.append(masks_e)

        # Truncate every mask to the batch-wide minimum so they stack into rectangles.
        # This is how the reference avoids padding entirely.
        collated_pred = [[m[:min_keep_pred] for m in ms] for ms in collated_pred]
        collated_enc = [[m[:min_keep_enc] for m in ms] for ms in collated_enc]

        return (
            collated,
            torch.utils.data.default_collate(collated_enc),
            torch.utils.data.default_collate(collated_pred),
        )


def masks_to_grid(mask_idx: torch.Tensor, height: int, width: int) -> torch.Tensor:
    """Scatter a flat index tensor back to an [height, width] bool grid (for figures)."""
    grid = torch.zeros(height * width, dtype=torch.bool)
    grid[mask_idx.flatten()] = True
    return grid.view(height, width)


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/__init__.py
"""Model factory.

Every SSL method exposes the same surface so `src/engine/pretrain.py` can drive all four
without branching on the method name anywhere except loss computation:

    model.encoder            the ViT to export and later fine-tune
    model.target_encoder     EMA branch, or None
    model.checkpoint_modules()  -> {"model": self}
"""
from __future__ import annotations

import torch.nn as nn

from src.config import PretrainCfg

# `needs_two_views` drives the transform; `needs_masks` drives the collator.
METHOD_SPECS: dict[str, dict[str, bool]] = {
    "ijepa": {"needs_two_views": False, "needs_masks": True, "has_ema": True},
    "mae": {"needs_two_views": False, "needs_masks": False, "has_ema": False},
    "simclr": {"needs_two_views": True, "needs_masks": False, "has_ema": False},
    "mocov3": {"needs_two_views": True, "needs_masks": False, "has_ema": True},
}


def build_pretrain_model(cfg: PretrainCfg) -> nn.Module:
    m, method = cfg.model, cfg.method
    common = dict(
        arch=m.arch,
        img_size=m.img_size,
        patch_size=m.patch_size,
        drop_path_rate=m.drop_path_rate,
    )

    if method == "ijepa":
        from src.model.jepa import IJEPA

        return IJEPA(pred_emb_dim=cfg.jepa.pred_emb_dim, pred_depth=cfg.jepa.pred_depth, **common)

    if method == "mae":
        from src.model.mae import MAE

        return MAE(
            mask_ratio=cfg.mae.mask_ratio,
            decoder_embed_dim=cfg.mae.decoder_embed_dim,
            decoder_depth=cfg.mae.decoder_depth,
            decoder_num_heads=cfg.mae.decoder_num_heads,
            norm_pix_loss=cfg.mae.norm_pix_loss,
            **common,
        )

    if method == "simclr":
        from src.model.simclr import SimCLR

        c = cfg.contrastive
        return SimCLR(
            proj_hidden_dim=c.proj_hidden_dim,
            proj_out_dim=c.proj_out_dim,
            proj_num_layers=c.proj_num_layers,
            temperature=c.temperature,
            **common,
        )

    if method == "mocov3":
        from src.model.mocov3 import MoCoV3

        c = cfg.contrastive
        return MoCoV3(
            proj_hidden_dim=c.proj_hidden_dim,
            proj_out_dim=c.proj_out_dim,
            proj_num_layers=c.proj_num_layers,
            pred_hidden_dim=c.pred_hidden_dim,
            temperature=c.temperature,
            freeze_patch_embed=c.freeze_patch_embed,
            symmetric_loss=c.symmetric_loss,
            **common,
        )

    raise KeyError(f"unknown method {method!r}; known: {sorted(METHOD_SPECS)}")


__all__ = ["build_pretrain_model", "METHOD_SPECS"]


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/vit.py
"""Vision Transformer backbone.

Follows reference/ijepa/src/models/vision_transformer.py: no class token, fixed 2-D
sin-cos position embeddings, `fix_init_weight` residual rescaling, and a `masks`
argument that drops tokens *before* the blocks run (so the context encoder really does
cost less than a full forward pass).

Three deliberate changes from the reference, each for a concrete reason:

* **`F.scaled_dot_product_attention`** instead of an explicit softmax matmul. Same math,
  but it picks a memory-efficient kernel and cuts activation memory noticeably on a
  16 GB T4. The explicit path is kept behind `return_attention` for the attention-map
  figure, since SDPA does not expose the weights.
* **A correct `interpolate_pos_encoding`.** The reference version assumes a class token
  (`npatch = x.shape[1] - 1`) that these models do not have, so it mis-indexes. We need
  this working for real: segmentation fine-tunes at 352px, i.e. a 14x14 -> 22x22
  position-embedding stretch.
* **`return_intermediates`**, so the segmentation decoder can tap blocks {3,6,9,12}
  without a forward hook.
"""
from __future__ import annotations

import math
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.masks.utils import apply_masks

# --------------------------------------------------------------- position embeds


def get_1d_sincos_pos_embed_from_grid(embed_dim: int, pos: np.ndarray) -> np.ndarray:
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=float)
    omega /= embed_dim / 2.0
    omega = 1.0 / 10000**omega
    out = np.einsum("m,d->md", pos.reshape(-1), omega)
    return np.concatenate([np.sin(out), np.cos(out)], axis=1)


def get_2d_sincos_pos_embed(embed_dim: int, grid_size: int) -> np.ndarray:
    """[grid_size**2, embed_dim] 2-D sin-cos table. Width varies fastest."""
    grid_h = np.arange(grid_size, dtype=float)
    grid_w = np.arange(grid_size, dtype=float)
    grid = np.stack(np.meshgrid(grid_w, grid_h), axis=0).reshape([2, 1, grid_size, grid_size])
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])
    return np.concatenate([emb_h, emb_w], axis=1)


# ------------------------------------------------------------------ layers


def drop_path(x: torch.Tensor, drop_prob: float = 0.0, training: bool = False) -> torch.Tensor:
    if drop_prob == 0.0 or not training:
        return x
    keep = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    rand = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
    return x.div(keep) * rand.floor_()


class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0) -> None:
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return drop_path(x, self.drop_prob, self.training)

    def extra_repr(self) -> str:
        return f"drop_prob={self.drop_prob:.3f}"


class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim**-0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop_p = attn_drop
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        if return_attention:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            return attn.softmax(dim=-1)

        x = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.attn_drop_p if self.training else 0.0
        )
        x = x.transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class Block(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        drop=0.0,
        attn_drop=0.0,
        drop_path_rate=0.0,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), act_layer=act_layer, drop=drop)

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        if return_attention:
            return self.attn(self.norm1(x), return_attention=True)
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size**2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x).flatten(2).transpose(1, 2)


def trunc_normal_(tensor: torch.Tensor, mean=0.0, std=1.0, a=-2.0, b=2.0) -> torch.Tensor:
    return nn.init.trunc_normal_(tensor, mean=mean, std=std, a=a, b=b)


# ------------------------------------------------------------------ backbone


class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        depth: int = 12,
        num_heads: int = 12,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.0,
        norm_layer=partial(nn.LayerNorm, eps=1e-6),
        init_std: float = 0.02,
    ) -> None:
        super().__init__()
        self.num_features = self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.depth = depth
        self.patch_size = patch_size
        self.init_std = init_std

        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        # Fixed, not learned: sin-cos generalises to unseen grid sizes, which is what
        # makes the 224 -> 352 transfer at fine-tuning time well-posed.
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim), requires_grad=False)
        self.pos_embed.data.copy_(
            torch.from_numpy(get_2d_sincos_pos_embed(embed_dim, self.patch_embed.grid_size))
            .float()
            .unsqueeze(0)
        )

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList(
            [
                Block(
                    embed_dim,
                    num_heads,
                    mlp_ratio,
                    qkv_bias,
                    drop_rate,
                    attn_drop_rate,
                    dpr[i],
                    norm_layer=norm_layer,
                )
                for i in range(depth)
            ]
        )
        self.norm = norm_layer(embed_dim)

        self.apply(self._init_weights)
        self.fix_init_weight()

    def fix_init_weight(self) -> None:
        """Scale down deeper residual branches by 1/sqrt(2*layer_id).

        Keeps the residual stream's variance from growing with depth, which is what lets
        a 12-block ViT train without extra warmup tricks. Same as the reference.
        """

        def rescale(param: torch.Tensor, layer_id: int) -> None:
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def interpolate_pos_encoding(self, x: torch.Tensor) -> torch.Tensor:
        """Bicubically resize the position table to match x's token count.

        No class token to skip, unlike the reference implementation this is ported from.
        """
        npatch = x.shape[1]
        n = self.pos_embed.shape[1]
        if npatch == n:
            return self.pos_embed
        dim = self.pos_embed.shape[-1]
        src = int(round(math.sqrt(n)))
        dst = int(round(math.sqrt(npatch)))
        if src * src != n or dst * dst != npatch:
            raise ValueError(f"non-square token grids not supported: {n} -> {npatch}")
        pos = self.pos_embed.reshape(1, src, src, dim).permute(0, 3, 1, 2)
        pos = F.interpolate(pos, size=(dst, dst), mode="bicubic", align_corners=False)
        return pos.permute(0, 2, 3, 1).reshape(1, dst * dst, dim)

    def forward(
        self,
        x: torch.Tensor,
        masks: list[torch.Tensor] | torch.Tensor | None = None,
        return_intermediates: list[int] | None = None,
    ):
        """
        :param masks: index tensors of patches to keep. Applied after the position
            embedding, so masked-out tokens never enter the blocks at all.
        :param return_intermediates: 0-indexed block outputs to also return, for the
            segmentation feature pyramid.
        """
        if masks is not None and not isinstance(masks, list):
            masks = [masks]

        x = self.patch_embed(x)
        x = x + self.interpolate_pos_encoding(x)
        if masks is not None:
            x = apply_masks(x, masks)

        wanted = set(return_intermediates or [])
        intermediates: list[torch.Tensor] = []
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i in wanted:
                intermediates.append(x)

        x = self.norm(x)
        if return_intermediates:
            return x, intermediates
        return x

    def get_last_selfattention(self, x: torch.Tensor) -> torch.Tensor:
        """Attention weights of the final block — used by the attention-map figure."""
        x = self.patch_embed(x)
        x = x + self.interpolate_pos_encoding(x)
        for blk in self.blocks[:-1]:
            x = blk(x)
        return self.blocks[-1](x, return_attention=True)


# ------------------------------------------------------------------ factories

def vit_tiny(patch_size=16, **kw):
    return VisionTransformer(patch_size=patch_size, embed_dim=192, depth=12, num_heads=3, **kw)


def vit_small(patch_size=16, **kw):
    return VisionTransformer(patch_size=patch_size, embed_dim=384, depth=12, num_heads=6, **kw)


def vit_base(patch_size=16, **kw):
    return VisionTransformer(patch_size=patch_size, embed_dim=768, depth=12, num_heads=12, **kw)


def vit_large(patch_size=16, **kw):
    return VisionTransformer(patch_size=patch_size, embed_dim=1024, depth=24, num_heads=16, **kw)


VIT_FACTORY = {
    "vit_tiny": vit_tiny,
    "vit_small": vit_small,
    "vit_base": vit_base,
    "vit_large": vit_large,
}


def build_vit(arch: str, img_size: int, patch_size: int, **kw) -> VisionTransformer:
    if arch not in VIT_FACTORY:
        raise KeyError(f"unknown arch {arch!r}; known: {sorted(VIT_FACTORY)}")
    return VIT_FACTORY[arch](patch_size=patch_size, img_size=img_size, **kw)


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/predictor.py
"""I-JEPA predictor.

Takes the context tokens the encoder produced and predicts the *representations* of the
target blocks — never pixels. That is the whole point of JEPA versus MAE.

Ported from `VisionTransformerPredictor` in
reference/ijepa/src/models/vision_transformer.py. The pieces the previous version of
this project was missing, and why each matters:

* **`predictor_embed` / `predictor_proj`.** The predictor runs in its own narrower width
  (192 here vs the encoder's 384) with a Linear in and out. Forcing the two widths equal
  instead, as the old code did, makes the predictor as expensive as the encoder and
  removes the bottleneck that stops it from learning an identity shortcut.
* **Predictor position embeddings added to the context tokens** (`x += apply_masks(...)`).
  Without them the predictor cannot tell *where* the context patches came from, so it
  cannot know the spatial relationship between context and target — which is the only
  signal it has.
* **Its own sin-cos position table**, separate from the encoder's.
"""
from __future__ import annotations

import math
from functools import partial

import torch
import torch.nn as nn

from src.masks.utils import apply_masks, repeat_interleave_batch
from src.model.vit import Block, get_2d_sincos_pos_embed, trunc_normal_


class VisionTransformerPredictor(nn.Module):
    def __init__(
        self,
        num_patches: int,
        embed_dim: int = 768,
        predictor_embed_dim: int = 384,
        depth: int = 6,
        num_heads: int = 12,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.0,
        norm_layer=partial(nn.LayerNorm, eps=1e-6),
        init_std: float = 0.02,
    ) -> None:
        super().__init__()
        self.init_std = init_std
        self.predictor_embed = nn.Linear(embed_dim, predictor_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_embed_dim))

        self.predictor_pos_embed = nn.Parameter(
            torch.zeros(1, num_patches, predictor_embed_dim), requires_grad=False
        )
        self.predictor_pos_embed.data.copy_(
            torch.from_numpy(
                get_2d_sincos_pos_embed(predictor_embed_dim, int(round(num_patches**0.5)))
            )
            .float()
            .unsqueeze(0)
        )

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.predictor_blocks = nn.ModuleList(
            [
                Block(
                    predictor_embed_dim,
                    num_heads,
                    mlp_ratio,
                    qkv_bias,
                    drop_rate,
                    attn_drop_rate,
                    dpr[i],
                    norm_layer=norm_layer,
                )
                for i in range(depth)
            ]
        )
        self.predictor_norm = norm_layer(predictor_embed_dim)
        self.predictor_proj = nn.Linear(predictor_embed_dim, embed_dim, bias=True)

        trunc_normal_(self.mask_token, std=self.init_std)
        self.apply(self._init_weights)
        self.fix_init_weight()

    def fix_init_weight(self) -> None:
        def rescale(param: torch.Tensor, layer_id: int) -> None:
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.predictor_blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(
        self,
        x: torch.Tensor,
        masks_x: list[torch.Tensor],
        masks: list[torch.Tensor],
    ) -> torch.Tensor:
        """
        :param x: encoder output for the context tokens, [n_enc*B, K_ctx, embed_dim]
        :param masks_x: the context mask indices (to look up their positions)
        :param masks: the target mask indices (what to predict)
        :return: predicted target representations, [n_pred*n_enc*B, K_tgt, embed_dim]
        """
        if not isinstance(masks_x, list):
            masks_x = [masks_x]
        if not isinstance(masks, list):
            masks = [masks]

        B = len(x) // len(masks_x)

        x = self.predictor_embed(x)

        # Tell the predictor where the context patches live in the image.
        x_pos = self.predictor_pos_embed.repeat(B, 1, 1)
        x = x + apply_masks(x_pos, masks_x)
        _, n_ctxt, _ = x.shape

        # Query tokens: a shared learned token plus the position of what to predict.
        pos = self.predictor_pos_embed.repeat(B, 1, 1)
        pos = apply_masks(pos, masks)
        pos = repeat_interleave_batch(pos, B, repeat=len(masks_x))
        pred_tokens = self.mask_token.repeat(pos.size(0), pos.size(1), 1) + pos

        # One copy of the context per target block, so each target attends to all of it.
        x = x.repeat(len(masks), 1, 1)
        x = torch.cat([x, pred_tokens], dim=1)

        for blk in self.predictor_blocks:
            x = blk(x)
        x = self.predictor_norm(x)

        return self.predictor_proj(x[:, n_ctxt:])


def vit_predictor(**kwargs) -> VisionTransformerPredictor:
    return VisionTransformerPredictor(mlp_ratio=4, qkv_bias=True, **kwargs)


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/jepa.py
"""I-JEPA: encoder + predictor + EMA target encoder.

The objective, in one line: encode a large *context* block, predict the representations
of four disjoint *target* blocks, and compare against an exponential-moving-average copy
of the same encoder run on the full image.

Two details here are load-bearing and both were missing from the earlier scaffold:

1. **`F.layer_norm` on the target encoder output, before masking.** Without it the model
   can trivially minimise the loss by shrinking the target representations toward a
   constant — representation collapse. The LayerNorm removes the scale degree of freedom
   the collapse would exploit. This is the single most important line in the file.
2. **The EMA update runs in fp32, outside autocast.** With momentum 0.9995 the update
   term is `(1-m) = 5e-4` times a parameter of order 1e-2. In fp16 that rounds to zero,
   so the target encoder silently stops tracking and the loss flatlines at a plausible
   value. Nothing errors; the run is simply meaningless. Hence the explicit no_grad +
   float path.
"""
from __future__ import annotations

import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.masks.utils import apply_masks, repeat_interleave_batch
from src.model.predictor import VisionTransformerPredictor
from src.model.vit import VisionTransformer, build_vit


class IJEPA(nn.Module):
    def __init__(
        self,
        arch: str = "vit_small",
        img_size: int = 224,
        patch_size: int = 16,
        pred_emb_dim: int = 192,
        pred_depth: int = 6,
        drop_path_rate: float = 0.0,
    ) -> None:
        super().__init__()
        self.encoder: VisionTransformer = build_vit(
            arch, img_size=img_size, patch_size=patch_size, drop_path_rate=drop_path_rate
        )
        self.predictor = VisionTransformerPredictor(
            num_patches=self.encoder.patch_embed.num_patches,
            embed_dim=self.encoder.embed_dim,
            predictor_embed_dim=pred_emb_dim,
            depth=pred_depth,
            num_heads=self.encoder.num_heads,
        )
        # Target encoder starts as an exact copy and is only ever updated by EMA.
        self.target_encoder: VisionTransformer = copy.deepcopy(self.encoder)
        for p in self.target_encoder.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target_encoder(self, m: float) -> None:
        """EMA: target <- m * target + (1 - m) * online.

        Runs in fp32 regardless of the surrounding autocast context — see module
        docstring for why that is not optional.
        """
        for q, k in zip(self.encoder.parameters(), self.target_encoder.parameters()):
            k.data.mul_(m).add_(q.detach().data.to(k.dtype), alpha=1.0 - m)

    def forward_target(
        self, imgs: torch.Tensor, masks_pred: list[torch.Tensor], n_enc: int
    ) -> torch.Tensor:
        with torch.no_grad():
            h = self.target_encoder(imgs)
            h = F.layer_norm(h, (h.size(-1),))  # anti-collapse; see module docstring
            B = len(h)
            h = apply_masks(h, masks_pred)
            return repeat_interleave_batch(h, B, repeat=n_enc)

    def forward_context(
        self,
        imgs: torch.Tensor,
        masks_enc: list[torch.Tensor],
        masks_pred: list[torch.Tensor],
    ) -> torch.Tensor:
        z = self.encoder(imgs, masks_enc)
        return self.predictor(z, masks_enc, masks_pred)

    def forward(
        self,
        imgs: torch.Tensor,
        masks_enc: list[torch.Tensor],
        masks_pred: list[torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.forward_target(imgs, masks_pred, n_enc=len(masks_enc))
        z = self.forward_context(imgs, masks_enc, masks_pred)
        return z, h

    @staticmethod
    def loss(z: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """Smooth L1 in fp32.

        The loss is order 1e-2 and fp16 has ~3 decimal digits there, so accumulating it
        in half precision throws away most of the gradient signal.
        """
        return F.smooth_l1_loss(z.float(), h.float())

    def checkpoint_modules(self) -> dict[str, nn.Module]:
        return {"model": self}


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/mae.py
"""Masked Autoencoder (He et al., 2021) — the most important baseline here.

I-JEPA's central claim is that predicting in *latent* space beats predicting in *pixel*
space, because pixel reconstruction forces the encoder to keep high-frequency detail that
carries no semantics. MAE is the pixel-space arm of exactly that comparison, so if only
one baseline survives a budget cut, this is the one to keep.

Architecture matches the paper: asymmetric, with the encoder seeing only the 25% visible
patches and a shallow narrow decoder reconstructing all of them. Decoder width is 256 =
0.67 x 384, preserving MAE's own 512/768 decoder/encoder ratio for ViT-B rather than
copying the absolute 512.

Budget caveat to state in the thesis: MAE is known to keep improving out to 800-1600
epochs, while contrastive methods saturate earlier. A fixed 100-epoch budget therefore
*structurally disadvantages* MAE. That is a limitation to disclose, not to hide.
"""
from __future__ import annotations

from functools import partial

import torch
import torch.nn as nn

from src.model.vit import (
    Block,
    VisionTransformer,
    build_vit,
    get_2d_sincos_pos_embed,
    trunc_normal_,
)


class MAE(nn.Module):
    def __init__(
        self,
        arch: str = "vit_small",
        img_size: int = 224,
        patch_size: int = 16,
        in_chans: int = 3,
        mask_ratio: float = 0.75,
        decoder_embed_dim: int = 256,
        decoder_depth: int = 8,
        decoder_num_heads: int = 8,
        norm_pix_loss: bool = True,
        drop_path_rate: float = 0.0,
    ) -> None:
        super().__init__()
        self.encoder: VisionTransformer = build_vit(
            arch, img_size=img_size, patch_size=patch_size, drop_path_rate=drop_path_rate
        )
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.mask_ratio = mask_ratio
        self.norm_pix_loss = norm_pix_loss

        embed_dim = self.encoder.embed_dim
        num_patches = self.encoder.patch_embed.num_patches
        grid = self.encoder.patch_embed.grid_size

        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(
            torch.zeros(1, num_patches, decoder_embed_dim), requires_grad=False
        )
        self.decoder_pos_embed.data.copy_(
            torch.from_numpy(get_2d_sincos_pos_embed(decoder_embed_dim, grid)).float().unsqueeze(0)
        )
        norm_layer = partial(nn.LayerNorm, eps=1e-6)
        self.decoder_blocks = nn.ModuleList(
            [
                Block(decoder_embed_dim, decoder_num_heads, 4.0, qkv_bias=True, norm_layer=norm_layer)
                for _ in range(decoder_depth)
            ]
        )
        self.decoder_norm = norm_layer(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size**2 * in_chans, bias=True)

        trunc_normal_(self.mask_token, std=0.02)
        self.decoder_blocks.apply(self._init_weights)
        self._init_weights(self.decoder_embed)
        self._init_weights(self.decoder_pred)

    @staticmethod
    def _init_weights(m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    # ------------------------------------------------------------ patch helpers

    def patchify(self, imgs: torch.Tensor) -> torch.Tensor:
        """[B,C,H,W] -> [B, L, p*p*C]"""
        p, c = self.patch_size, self.in_chans
        b, _, h, w = imgs.shape
        gh, gw = h // p, w // p
        x = imgs.reshape(b, c, gh, p, gw, p)
        x = torch.einsum("nchpwq->nhwpqc", x)
        return x.reshape(b, gh * gw, p * p * c)

    def unpatchify(self, x: torch.Tensor) -> torch.Tensor:
        """[B, L, p*p*C] -> [B,C,H,W]"""
        p, c = self.patch_size, self.in_chans
        b, ln, _ = x.shape
        g = int(round(ln**0.5))
        x = x.reshape(b, g, g, p, p, c)
        x = torch.einsum("nhwpqc->nchpwq", x)
        return x.reshape(b, c, g * p, g * p)

    # ---------------------------------------------------------------- masking

    def random_masking(self, x: torch.Tensor, mask_ratio: float):
        """Per-sample random shuffle, keep the first (1-mask_ratio) fraction.

        Returns (kept tokens, binary mask with 1 = removed, ids_restore) — the restore
        indices are what let the decoder put mask tokens back in the right places.
        """
        b, ln, d = x.shape
        keep = int(ln * (1 - mask_ratio))
        noise = torch.rand(b, ln, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        ids_keep = ids_shuffle[:, :keep]
        x_masked = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, d))

        mask = torch.ones(b, ln, device=x.device)
        mask[:, :keep] = 0
        mask = torch.gather(mask, 1, ids_restore)
        return x_masked, mask, ids_restore

    # ---------------------------------------------------------------- forward

    def forward_encoder(self, imgs: torch.Tensor, mask_ratio: float):
        enc = self.encoder
        x = enc.patch_embed(imgs)
        x = x + enc.interpolate_pos_encoding(x)
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        for blk in enc.blocks:
            x = blk(x)
        return enc.norm(x), mask, ids_restore

    def forward_decoder(self, x: torch.Tensor, ids_restore: torch.Tensor) -> torch.Tensor:
        x = self.decoder_embed(x)
        b, _, d = x.shape
        n_missing = ids_restore.shape[1] - x.shape[1]
        mask_tokens = self.mask_token.expand(b, n_missing, -1)
        x = torch.cat([x, mask_tokens], dim=1)
        x = torch.gather(x, 1, ids_restore.unsqueeze(-1).expand(-1, -1, d))  # unshuffle
        x = x + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        return self.decoder_pred(self.decoder_norm(x))

    def loss(self, imgs: torch.Tensor, pred: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """MSE on removed patches only, in fp32.

        `norm_pix_loss` normalises each target patch by its own mean/std, which the paper
        found materially improves representation quality — it stops the loss being
        dominated by locally bright or high-contrast patches.
        """
        target = self.patchify(imgs).float()
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1.0e-6) ** 0.5
        loss = (pred.float() - target) ** 2
        loss = loss.mean(dim=-1)
        return (loss * mask).sum() / mask.sum().clamp(min=1)

    def forward(self, imgs: torch.Tensor, mask_ratio: float | None = None):
        mr = self.mask_ratio if mask_ratio is None else mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mr)
        pred = self.forward_decoder(latent, ids_restore)
        return self.loss(imgs, pred, mask), pred, mask

    # MAE has no EMA target encoder — the exported weights are the online encoder.
    target_encoder = None

    def checkpoint_modules(self) -> dict[str, nn.Module]:
        return {"model": self}


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/heads.py
"""Projection and prediction heads shared by the contrastive baselines."""
from __future__ import annotations

import torch
import torch.nn as nn


def build_mlp(
    in_dim: int,
    hidden_dim: int,
    out_dim: int,
    num_layers: int = 3,
    use_bn: bool = True,
    last_bn: bool = False,
) -> nn.Sequential:
    """MLP head with BatchNorm between layers.

    BatchNorm in the projector is not incidental — SimCLR and MoCo v3 both report clear
    drops without it. `last_bn` (an affine-free BN on the output) is MoCo v3's variant.
    """
    layers: list[nn.Module] = []
    d = in_dim
    for i in range(num_layers - 1):
        layers.append(nn.Linear(d, hidden_dim, bias=not use_bn))
        if use_bn:
            layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU(inplace=True))
        d = hidden_dim
    layers.append(nn.Linear(d, out_dim, bias=not last_bn))
    if last_bn:
        layers.append(nn.BatchNorm1d(out_dim, affine=False))
    return nn.Sequential(*layers)


class PooledBackbone(nn.Module):
    """ViT + mean pooling over patch tokens.

    These models have no class token, so the sequence has to be reduced some other way.
    Mean pooling is the standard choice and is what the segmentation decoder's features
    are consistent with.
    """

    def __init__(self, encoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder

    @property
    def embed_dim(self) -> int:
        return self.encoder.embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x).mean(dim=1)


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/simclr.py
"""SimCLR (Chen et al., 2020) with a ViT backbone.

Two augmented views per image, NT-Xent over the batch. Everything expensive about this
method is structural: both views run through the encoder *with gradients*, so a step
costs roughly twice a supervised step — about 1.8x I-JEPA and 3.6x MAE.

Two implementation points that are easy to get wrong and silent when you do:

* **Embeddings are all-gathered across ranks with gradient flow** (`gather_with_grad`).
  Plain `dist.all_gather` detaches, which would quietly reduce the negatives each rank
  sees from `global_batch - 1` to `per_gpu_batch - 1` — at global batch 512 on 2 GPUs,
  255 instead of 511. The loss still trains; it is just a weaker method than reported.
* **The logits are computed in fp32.** Under fp16 autocast, `exp()` on the similarity
  matrix overflows to inf and the loss becomes NaN or silently degenerate. This is the
  single most common half-precision bug in contrastive code.

Budget caveat for the thesis: SimCLR benefits substantially from batches far larger than
512. Holding global batch equal across all four methods is the right *fairness* call
(equal optimisation budget) but it does disadvantage SimCLR, and that should be
disclosed rather than buried.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model.heads import PooledBackbone, build_mlp
from src.model.vit import build_vit
from src.utils.ddp import gather_with_grad, get_rank, get_world_size


class SimCLR(nn.Module):
    def __init__(
        self,
        arch: str = "vit_small",
        img_size: int = 224,
        patch_size: int = 16,
        proj_hidden_dim: int = 2048,
        proj_out_dim: int = 256,
        proj_num_layers: int = 3,
        temperature: float = 0.1,
        drop_path_rate: float = 0.0,
    ) -> None:
        super().__init__()
        self.encoder = build_vit(
            arch, img_size=img_size, patch_size=patch_size, drop_path_rate=drop_path_rate
        )
        self.backbone = PooledBackbone(self.encoder)
        self.projector = build_mlp(
            self.encoder.embed_dim, proj_hidden_dim, proj_out_dim, proj_num_layers
        )
        self.temperature = temperature

    target_encoder = None

    def embed(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.projector(self.backbone(x)), dim=-1)

    def nt_xent(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """NT-Xent over the *global* batch, computed in fp32."""
        with torch.autocast(device_type=z1.device.type, enabled=False):
            z1, z2 = z1.float(), z2.float()
            local_n = z1.shape[0]

            g1, g2 = gather_with_grad(z1), gather_with_grad(z2)
            z = torch.cat([g1, g2], dim=0)  # [2N, D]
            n = g1.shape[0]

            sim = (z @ z.T) / self.temperature
            sim.fill_diagonal_(float("-inf"))  # never contrast a sample with itself

            # Positive for row i in [0,N) is row i+N, and vice versa.
            targets = torch.cat(
                [torch.arange(n, 2 * n, device=z.device), torch.arange(0, n, device=z.device)]
            )

            # Only this rank's rows contribute to the loss; the rest are negatives whose
            # gradients arrive through the gather's backward. Averaging every row on every
            # rank would count each sample world_size times.
            rank, world = get_rank(), get_world_size()
            if world > 1:
                idx = torch.cat(
                    [
                        torch.arange(rank * local_n, (rank + 1) * local_n, device=z.device),
                        torch.arange(n + rank * local_n, n + (rank + 1) * local_n, device=z.device),
                    ]
                )
                sim, targets = sim[idx], targets[idx]

            return F.cross_entropy(sim, targets)

    def forward(self, views: tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
        v1, v2 = views
        return self.nt_xent(self.embed(v1), self.embed(v2))

    def checkpoint_modules(self) -> dict[str, nn.Module]:
        return {"model": self}


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/mocov3.py
"""MoCo v3 (Chen et al., 2021) with a ViT backbone.

The fairest contrastive comparison for I-JEPA, because it also uses an EMA target
encoder — so any difference between the two is attributable to the *objective*
(contrastive instance discrimination vs latent block prediction) rather than to the
presence or absence of a momentum branch.

Also the most expensive method in the study: two full views through the query encoder
*with* gradients, plus two more through the momentum encoder without, plus a symmetrised
loss. About 2.4x I-JEPA and 4.8x MAE per image.

Three details that matter:

* **Frozen random patch embedding.** The paper's own fix for ViT training instability:
  the patch-embed conv is initialised randomly and never updated. Without it MoCo v3 on
  a ViT is prone to loss spikes and partial collapse, and fp16 makes that worse.
* **The momentum update runs in fp32, outside autocast.** With m -> 1.0 the update term
  `(1-m)` falls below fp16 resolution relative to the parameter magnitude, so the key
  encoder silently stops tracking the query encoder and the loss plateaus at a plausible
  value. Nothing errors. The `momentum_update` method below is deliberately not
  autocast-wrapped.
* **InfoNCE in fp32**, for the same overflow reason as SimCLR.
"""
from __future__ import annotations

import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model.heads import PooledBackbone, build_mlp
from src.model.vit import build_vit
from src.utils.ddp import gather_with_grad


class MoCoV3(nn.Module):
    def __init__(
        self,
        arch: str = "vit_small",
        img_size: int = 224,
        patch_size: int = 16,
        proj_hidden_dim: int = 2048,
        proj_out_dim: int = 256,
        proj_num_layers: int = 3,
        pred_hidden_dim: int = 2048,
        temperature: float = 0.2,
        freeze_patch_embed: bool = True,
        symmetric_loss: bool = True,
        drop_path_rate: float = 0.0,
    ) -> None:
        super().__init__()
        self.encoder = build_vit(
            arch, img_size=img_size, patch_size=patch_size, drop_path_rate=drop_path_rate
        )
        embed_dim = self.encoder.embed_dim
        self.backbone = PooledBackbone(self.encoder)
        self.projector = build_mlp(
            embed_dim, proj_hidden_dim, proj_out_dim, proj_num_layers, last_bn=True
        )
        # Asymmetric predictor on the query branch only — this asymmetry is what stops
        # the two branches collapsing onto a constant.
        self.predictor = build_mlp(proj_out_dim, pred_hidden_dim, proj_out_dim, 2, last_bn=False)

        self.target_encoder = copy.deepcopy(self.encoder)
        self.target_projector = copy.deepcopy(self.projector)
        for p in list(self.target_encoder.parameters()) + list(self.target_projector.parameters()):
            p.requires_grad = False
        self.target_backbone = PooledBackbone(self.target_encoder)

        self.temperature = temperature
        self.symmetric_loss = symmetric_loss

        if freeze_patch_embed:
            for p in self.encoder.patch_embed.parameters():
                p.requires_grad = False
            for p in self.target_encoder.patch_embed.parameters():
                p.requires_grad = False

    @torch.no_grad()
    def momentum_update(self, m: float) -> None:
        """key <- m * key + (1 - m) * query, in fp32. See module docstring."""
        for q, k in zip(self.encoder.parameters(), self.target_encoder.parameters()):
            k.data.mul_(m).add_(q.detach().data.to(k.dtype), alpha=1.0 - m)
        for q, k in zip(self.projector.parameters(), self.target_projector.parameters()):
            k.data.mul_(m).add_(q.detach().data.to(k.dtype), alpha=1.0 - m)
        for q, k in zip(self.projector.buffers(), self.target_projector.buffers()):
            k.data.copy_(q.data)  # BatchNorm running stats are copied, not averaged

    def _query(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.predictor(self.projector(self.backbone(x))), dim=-1)

    @torch.no_grad()
    def _key(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.target_projector(self.target_backbone(x)), dim=-1)

    def infonce(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        """Contrast each query against all gathered keys, in fp32."""
        with torch.autocast(device_type=q.device.type, enabled=False):
            q = q.float()
            k = gather_with_grad(k.float())
            logits = (q @ k.T) / self.temperature
            # Positives sit on the diagonal of this rank's slice of the gathered keys.
            from src.utils.ddp import get_rank

            offset = get_rank() * q.shape[0]
            labels = torch.arange(q.shape[0], device=q.device) + offset
            # 2*tau matches the paper's scaling, which keeps the loss magnitude
            # comparable across temperature choices.
            return F.cross_entropy(logits, labels) * (2 * self.temperature)

    def forward(self, views: tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
        v1, v2 = views
        q1, q2 = self._query(v1), self._query(v2)
        k1, k2 = self._key(v1), self._key(v2)
        if self.symmetric_loss:
            return self.infonce(q1, k2) + self.infonce(q2, k1)
        return self.infonce(q1, k2)

    def checkpoint_modules(self) -> dict[str, nn.Module]:
        return {"model": self}


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/simple_fpn.py
"""ViTDet-style simple feature pyramid.

SegFormer's all-MLP head expects a 4-scale feature pyramid, but a plain ViT is
single-scale: every block outputs tokens at stride 16. The ViTDet paper's answer, which
we use here, is to build the pyramid from *one* backbone by resampling features taken
from several blocks:

    block  3  -> ConvTranspose x4 -> stride 4
    block  6  -> ConvTranspose x2 -> stride 8
    block  9  -> identity         -> stride 16
    block 12  -> MaxPool x2       -> stride 32

This is worth being precise about in the write-up: calling the result "SegFormer" without
qualification is inaccurate, because SegFormer's own encoder (MiT) is genuinely
hierarchical. The honest description is "SegFormer-style all-MLP decoder on a ViT simple
feature pyramid".
"""
from __future__ import annotations

import torch
import torch.nn as nn


class SimpleFeaturePyramid(nn.Module):
    """Turn a list of [B, N, D] token tensors into feature maps at strides {4,8,16,32}."""

    def __init__(self, embed_dim: int, out_dim: int | None = None) -> None:
        super().__init__()
        out_dim = out_dim or embed_dim
        d = embed_dim

        # LayerNorm over channels for the 2-D maps. nn.LayerNorm on NCHW would normalise
        # the wrong axes, so we use GroupNorm(1, C), which is the same statistic.
        def norm(c: int) -> nn.Module:
            return nn.GroupNorm(1, c)

        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(d, d // 2, 2, stride=2),
            norm(d // 2),
            nn.GELU(),
            nn.ConvTranspose2d(d // 2, d // 4, 2, stride=2),
        )
        self.up2 = nn.Sequential(nn.ConvTranspose2d(d, d // 2, 2, stride=2))
        self.id1 = nn.Identity()
        self.down2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.out_channels = [d // 4, d // 2, d, d]
        self.out_dim = out_dim

    @staticmethod
    def tokens_to_map(x: torch.Tensor) -> torch.Tensor:
        """[B, N, D] -> [B, D, H, W] for a square token grid.

        `.contiguous()` is not decorative: `transpose` leaves a non-contiguous view, and
        autograd's backward for the following reshape then tries a `view` on a gradient
        whose strides straddle two subspaces, which raises. Forcing the copy here makes
        the backward well-defined.
        """
        b, n, d = x.shape
        g = int(round(n**0.5))
        if g * g != n:
            raise ValueError(f"expected a square token grid, got {n} tokens")
        return x.transpose(1, 2).contiguous().reshape(b, d, g, g)

    def forward(self, features: list[torch.Tensor]) -> list[torch.Tensor]:
        if len(features) != 4:
            raise ValueError(f"expected 4 intermediate features, got {len(features)}")
        f1, f2, f3, f4 = (self.tokens_to_map(f) for f in features)
        return [self.up4(f1), self.up2(f2), self.id1(f3), self.down2(f4)]


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/segformer_head.py
"""SegFormer all-MLP decoder (Xie et al., NeurIPS 2021).

Deliberately minimal: unify every pyramid level to one width with a 1x1 Linear, upsample
all of them to the finest stride, concatenate, fuse with one more MLP, and predict. No
attention, no heavy conv stack — about 2.4M parameters on top of a ViT-S encoder, which
is what makes "lightweight transformer-based decoder" an accurate description.

The same head, with the same initialisation, is used for all five encoders (I-JEPA, MAE,
SimCLR, MoCo v3, random init). That identity is the entire basis of the comparison, so it
is enforced in code rather than left to discipline.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model.simple_fpn import SimpleFeaturePyramid
from src.model.vit import VisionTransformer, build_vit


class MLPProject(nn.Module):
    """1x1 projection implemented as a Linear over flattened spatial positions."""

    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        x = x.flatten(2).transpose(1, 2)
        # See SimpleFeaturePyramid.tokens_to_map: reshape-after-transpose needs an
        # explicit copy or the backward pass hits a non-contiguous view.
        return self.proj(x).transpose(1, 2).contiguous().reshape(b, -1, h, w)


class SegFormerHead(nn.Module):
    def __init__(
        self,
        in_channels: list[int],
        embed_dim: int = 256,
        num_classes: int = 1,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.projections = nn.ModuleList([MLPProject(c, embed_dim) for c in in_channels])
        self.fuse = nn.Sequential(
            nn.Conv2d(embed_dim * len(in_channels), embed_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
        )
        self.dropout = nn.Dropout2d(dropout)
        self.classifier = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, features: list[torch.Tensor]) -> torch.Tensor:
        target_size = features[0].shape[2:]  # finest level, stride 4
        outs = []
        for proj, f in zip(self.projections, features):
            y = proj(f)
            if y.shape[2:] != target_size:
                y = F.interpolate(y, size=target_size, mode="bilinear", align_corners=False)
            outs.append(y)
        x = self.fuse(torch.cat(outs, dim=1))
        return self.classifier(self.dropout(x))


class ViTSegFormer(nn.Module):
    """ViT encoder + simple feature pyramid + SegFormer all-MLP head."""

    def __init__(
        self,
        arch: str = "vit_small",
        img_size: int = 352,
        patch_size: int = 16,
        fpn_layers: tuple[int, int, int, int] = (2, 5, 8, 11),
        decoder_embed_dim: int = 256,
        num_classes: int = 1,
        drop_path_rate: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder: VisionTransformer = build_vit(
            arch, img_size=img_size, patch_size=patch_size, drop_path_rate=drop_path_rate
        )
        self.fpn_layers = list(fpn_layers)
        depth = self.encoder.depth
        bad = [i for i in self.fpn_layers if not 0 <= i < depth]
        if bad:
            raise ValueError(f"fpn_layers {bad} out of range for a {depth}-block encoder")
        self.pyramid = SimpleFeaturePyramid(self.encoder.embed_dim)
        self.head = SegFormerHead(self.pyramid.out_channels, decoder_embed_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        _, intermediates = self.encoder(x, return_intermediates=self.fpn_layers)
        logits = self.head(self.pyramid(intermediates))
        return F.interpolate(logits, size=size, mode="bilinear", align_corners=False)

    # ------------------------------------------------------------- pretrained

    def load_pretrained_encoder(self, ckpt_path: str, strict_report: bool = True) -> dict:
        """Load exported SSL encoder weights.

        Position embeddings are a fixed sin-cos buffer regenerated at construction for
        the current image size, so a 224-pretrained checkpoint transfers to a 352 model
        without touching them — `interpolate_pos_encoding` handles any residual mismatch
        at forward time.
        """
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt.get("encoder", ckpt)
        state = {k: v for k, v in state.items() if not k.endswith("pos_embed")}

        # `strict=False` forgives missing/unexpected keys but still hard-errors on a
        # shape mismatch, and PyTorch's message names a tensor rather than the cause.
        # Catch it here and say what actually differs.
        own = self.encoder.state_dict()
        shape_mismatch = {
            k: (tuple(v.shape), tuple(own[k].shape))
            for k, v in state.items()
            if k in own and v.shape != own[k].shape
        }
        if shape_mismatch:
            hint = ""
            if any("patch_embed" in k for k in shape_mismatch):
                got, want = next(v for k, v in shape_mismatch.items() if "patch_embed" in k)
                hint = (
                    f" The patch embedding differs ({got[-1]}px vs {want[-1]}px), so the "
                    f"checkpoint was pretrained with patch_size={got[-1]} but this model "
                    f"uses patch_size={want[-1]}."
                )
            raise RuntimeError(
                f"encoder weights are incompatible with this model: "
                f"{len(shape_mismatch)} tensor(s) differ in shape, e.g. "
                f"{list(shape_mismatch.items())[:2]}.{hint} "
                f"Rebuild the segmentation model with the pretraining arch/patch_size, "
                f"or point --pretrained-ckpt at a matching encoder. ({ckpt_path})"
            )

        missing, unexpected = self.encoder.load_state_dict(state, strict=False)
        real_missing = [k for k in missing if not k.endswith("pos_embed")]
        report = {
            "source": ckpt_path,
            "source_module": ckpt.get("source_module"),
            "run_id": ckpt.get("run_id"),
            "loaded": len(state),
            "missing": real_missing,
            "unexpected": list(unexpected),
        }
        if strict_report and (real_missing or unexpected):
            raise RuntimeError(
                f"encoder weights do not match the model: missing={real_missing[:5]} "
                f"unexpected={list(unexpected)[:5]}. Check that arch/patch_size agree "
                f"with the pretraining config. ({ckpt_path})"
            )
        return report

    def decoder_modules(self) -> list[nn.Module]:
        return [self.pyramid, self.head]

    def param_groups(self, enc_lr: float, dec_lr: float, layer_decay: float = 0.75) -> list[dict]:
        return layerwise_param_groups(
            self.encoder, self.decoder_modules(), enc_lr, dec_lr, layer_decay
        )


def layerwise_param_groups(
    encoder: VisionTransformer,
    decoder_modules: list[nn.Module],
    enc_lr: float,
    dec_lr: float,
    layer_decay: float = 0.75,
) -> list[dict]:
    """Layer-wise LR decay for the encoder, flat LR for the decoder.

    Earlier blocks encode generic structure and are worth preserving; later blocks need
    to adapt to the task. `lr_scale` is what `ScheduleSet.apply` multiplies the scheduled
    LR by, so the decay survives the cosine schedule. Because the schedule's reference LR
    is `enc_lr`, the decoder's scale is `dec_lr / enc_lr` — every group's live LR is
    therefore `scheduled_lr * lr_scale`, and the `lr` field below is just its value at
    step 0.

    Groups are built from *all* parameters, not from `p.requires_grad`. Building them
    from `requires_grad` while a module is frozen produces an empty group, and unfreezing
    later then does nothing because those tensors were never registered with the
    optimizer — a silent no-op that is easy to ship and hard to notice.
    """
    depth = encoder.depth
    groups: list[dict] = []
    seen: set[int] = set()

    def split(named):
        decay, no_decay = [], []
        for n, p in named:
            if id(p) in seen or not p.requires_grad:
                continue
            seen.add(id(p))
            (no_decay if (p.ndim <= 1 or n.endswith(".bias")) else decay).append(p)
        return decay, no_decay

    def add(params, scale, wd_exclude):
        if params:
            groups.append(
                {
                    "params": params,
                    "lr": enc_lr * scale,
                    "lr_scale": scale,
                    **({"WD_exclude": True, "weight_decay": 0.0} if wd_exclude else {}),
                }
            )

    def emit(named, scale):
        d, nd = split(named)
        add(d, scale, False)
        add(nd, scale, True)

    # Patch embedding behaves as layer 0 — the most generic, so the most decayed.
    emit(list(encoder.patch_embed.named_parameters()), layer_decay**depth)
    for i, blk in enumerate(encoder.blocks):
        emit(list(blk.named_parameters()), layer_decay ** (depth - i - 1))
    emit(list(encoder.norm.named_parameters()), 1.0)

    dec_scale = (dec_lr / enc_lr) if enc_lr else 1.0
    dec_named: list = []
    for m in decoder_modules:
        dec_named += list(m.named_parameters())
    emit(dec_named, dec_scale)
    return groups


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/model/unet.py
"""ViT-encoder UNet decoder — kept purely as a decoder ablation.

The primary decoder is the SegFormer all-MLP head in `segformer_head.py`. This one exists
so the thesis can report a decoder-ablation row: does the encoder ranking (I-JEPA vs MAE
vs SimCLR vs MoCo v3) survive a change of decoder? If it does, the result is about the
*representations*, which is the claim being made. That is worth one extra table row.

It exposes the same interface as `ViTSegFormer` (`load_pretrained_encoder`,
`param_groups`) so `segment.py` can swap decoders without branching.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model.vit import VisionTransformer, build_vit


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch: int, out_ch: int, kernel: int = 3) -> None:
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel, padding=kernel // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int) -> None:
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.fuse = ConvBNReLU(in_ch // 2 + skip_ch, out_ch)
        self.refine = ConvBNReLU(out_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor | None = None) -> torch.Tensor:
        x = self.up(x)
        if skip is not None:
            if skip.shape[-2:] != x.shape[-2:]:
                skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.refine(self.fuse(x))


class ViTUNet(nn.Module):
    def __init__(
        self,
        arch: str = "vit_small",
        img_size: int = 352,
        patch_size: int = 16,
        skip_layers: tuple[int, ...] = (2, 5, 8),
        num_classes: int = 1,
        drop_path_rate: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder: VisionTransformer = build_vit(
            arch, img_size=img_size, patch_size=patch_size, drop_path_rate=drop_path_rate
        )
        d = self.encoder.embed_dim
        self.skip_layers = list(skip_layers)

        # Skips are consumed deepest-first, so reverse the (ascending) block order.
        dims = [d, d // 2, d // 4, d // 8]
        self.up_blocks = nn.ModuleList(
            [
                UpBlock(dims[i], d if i < len(self.skip_layers) else 0, dims[i + 1])
                for i in range(len(dims) - 1)
            ]
        )
        self.head = nn.Conv2d(dims[-1], num_classes, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        bottleneck, skips = self.encoder(x, return_intermediates=self.skip_layers)
        b, n, d = bottleneck.shape
        g = int(round(n**0.5))
        y = bottleneck.transpose(1, 2).reshape(b, d, g, g)
        maps = [s.transpose(1, 2).reshape(b, d, g, g) for s in skips][::-1]

        for i, blk in enumerate(self.up_blocks):
            y = blk(y, maps[i] if i < len(maps) else None)
        return F.interpolate(self.head(y), size=size, mode="bilinear", align_corners=False)

    def load_pretrained_encoder(self, ckpt_path: str, strict_report: bool = True) -> dict:
        from src.model.segformer_head import ViTSegFormer

        return ViTSegFormer.load_pretrained_encoder(self, ckpt_path, strict_report)

    def decoder_modules(self) -> list[nn.Module]:
        return [self.up_blocks, self.head]

    def param_groups(self, enc_lr: float, dec_lr: float, layer_decay: float = 0.75) -> list[dict]:
        from src.model.segformer_head import layerwise_param_groups

        return layerwise_param_groups(
            self.encoder, self.decoder_modules(), enc_lr, dec_lr, layer_decay
        )


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/utils/schedulers.py
"""Closed-form LR / weight-decay / EMA-momentum schedules.

The official i-jepa implementation uses stateful scheduler *objects* whose `.step()`
advances an internal counter, and on resume it replays them with

    for _ in range(start_epoch * ipe):
        scheduler.step(); wd_scheduler.step(); next(momentum_scheduler)

That works, but it makes the schedule position implicit state that must be replayed
exactly. On Kaggle, where a run is chopped into 2-3 sessions, we want the schedule to be
a pure function of `global_step` so a resume cannot drift. These functions produce
numerically identical values to the reference implementation when called with
`step = scheduler._step`.

Reference: reference/ijepa/src/utils/schedulers.py
"""
from __future__ import annotations

import math


def warmup_cosine_lr(
    step: int,
    warmup_steps: int,
    total_steps: int,
    start_lr: float,
    ref_lr: float,
    final_lr: float = 0.0,
) -> float:
    """Linear warmup start_lr -> ref_lr, then cosine ref_lr -> final_lr.

    Mirrors `WarmupCosineSchedule`, whose `T_max` is `total_steps - warmup_steps`.
    """
    if step < warmup_steps:
        progress = step / max(1, warmup_steps)
        return start_lr + progress * (ref_lr - start_lr)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(1.0, progress)
    lr = final_lr + (ref_lr - final_lr) * 0.5 * (1.0 + math.cos(math.pi * progress))
    return max(final_lr, lr)


def cosine_wd(step: int, total_steps: int, ref_wd: float, final_wd: float = 0.0) -> float:
    """Cosine weight decay, ref_wd -> final_wd.

    Note the direction: i-jepa *increases* WD over training (0.04 -> 0.4), so the
    clamp has to work in both directions. Mirrors `CosineWDSchedule`.
    """
    progress = min(1.0, step / max(1, total_steps))
    wd = final_wd + (ref_wd - final_wd) * 0.5 * (1.0 + math.cos(math.pi * progress))
    return max(final_wd, wd) if final_wd <= ref_wd else min(final_wd, wd)


def linear_momentum(step: int, total_steps: int, ema: tuple[float, float]) -> float:
    """EMA momentum ramp, ema[0] -> ema[1] linearly.

    i-jepa builds this as a generator over `int(ipe * num_epochs * ipe_scale) + 1`
    values; this is the same sequence indexed directly. The multiply-then-divide order
    is kept as-is so the values are bitwise identical to the reference generator.
    """
    lo, hi = ema
    step = min(step, total_steps)
    return lo + step * (hi - lo) / max(1, total_steps)


def cosine_momentum(step: int, total_steps: int, m: tuple[float, float]) -> float:
    """Cosine momentum ramp, used by MoCo v3 (0.99 -> 1.0)."""
    lo, hi = m
    progress = min(1.0, step / max(1, total_steps))
    return hi - (hi - lo) * 0.5 * (1.0 + math.cos(math.pi * progress))


class ScheduleSet:
    """Bundles the three schedules for one run and applies them to an optimizer.

    Holds no mutable step state of its own — `step` is always passed in — so a
    checkpoint only needs `global_step` to restore the schedule exactly.
    """

    def __init__(
        self,
        iters_per_epoch: int,
        epochs: int,
        warmup_epochs: int,
        start_lr: float,
        ref_lr: float,
        final_lr: float,
        weight_decay: float,
        final_weight_decay: float,
        ema: tuple[float, float],
        ipe_scale: float = 1.0,
    ) -> None:
        self.iters_per_epoch = iters_per_epoch
        self.total_steps = max(1, int(ipe_scale * epochs * iters_per_epoch))
        self.warmup_steps = int(warmup_epochs * iters_per_epoch)
        self.start_lr = start_lr
        self.ref_lr = ref_lr
        self.final_lr = final_lr
        self.ref_wd = weight_decay
        self.final_wd = final_weight_decay
        self.ema = tuple(ema)

    def lr(self, step: int) -> float:
        return warmup_cosine_lr(
            step, self.warmup_steps, self.total_steps, self.start_lr, self.ref_lr, self.final_lr
        )

    def wd(self, step: int) -> float:
        return cosine_wd(step, self.total_steps, self.ref_wd, self.final_wd)

    def momentum(self, step: int) -> float:
        return linear_momentum(step, self.total_steps, self.ema)

    def apply(self, optimizer, step: int) -> tuple[float, float]:
        """Write lr/wd into the optimizer's param groups. Returns (lr, wd).

        Groups flagged `WD_exclude` (biases and 1-D params such as LayerNorm weights)
        keep weight_decay=0 — decaying them measurably hurts ViT training and the
        reference implementation excludes them too.
        """
        lr, wd = self.lr(step), self.wd(step)
        for group in optimizer.param_groups:
            group["lr"] = lr * group.get("lr_scale", 1.0)
            if not group.get("WD_exclude", False):
                group["weight_decay"] = wd
        return lr, wd


def param_groups_with_wd_exclusion(*modules, base_lr: float = 0.0) -> list[dict]:
    """Split parameters into decayed / not-decayed groups.

    Biases and any 1-D parameter (LayerNorm weights, the mask token, class tokens) are
    excluded from weight decay, matching `init_opt` in reference/ijepa/src/helper.py.
    """
    decay, no_decay = [], []
    for module in modules:
        if module is None:
            continue
        for name, p in module.named_parameters():
            if not p.requires_grad:
                continue
            (no_decay if (p.ndim == 1 or name.endswith(".bias")) else decay).append(p)
    groups: list[dict] = []
    if decay:
        groups.append({"params": decay, "lr": base_lr})
    if no_decay:
        groups.append({"params": no_decay, "lr": base_lr, "WD_exclude": True, "weight_decay": 0.0})
    return groups


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/utils/checkpoint.py
"""Resumable checkpointing.

A 100-epoch pretraining run does not fit in one Kaggle session, so every run is chopped
into 2-3 sessions and *must* resume exactly. The previous version of this project saved
only encoder weights, which is not resumable: you lose AdamW's second-moment estimates
(thousands of steps to re-warm), the GradScaler's loss scale (~2000 steps), and the
position in the LR/WD/EMA schedules.

Two invariants worth stating because violating either is silent, not loud:

* **Atomic writes.** A session killed mid-`torch.save` leaves a truncated file. Since
  the next session loads exactly that file, one badly-timed kill would otherwise cost
  the whole run. We write to `.tmp` and `os.replace`, which is atomic on POSIX.
* **Refuse, don't warn, on config drift.** Resuming a checkpoint whose config differs
  changes `total_steps`, which shifts every schedule. That produces a run that trains
  fine and means nothing. The config hash mismatch raises.
"""
from __future__ import annotations

import os
import time
from pathlib import Path
from typing import Any

import torch

from src.utils.ddp import get_world_size, is_main, unwrap

FORMAT_VERSION = 2


def _state(obj: Any) -> Any:
    if obj is None:
        return None
    if isinstance(obj, torch.nn.Module):
        return unwrap(obj).state_dict()
    return obj.state_dict()


def build_checkpoint(
    *,
    run_id: str,
    config_hash: str,
    resolved_cfg: dict[str, Any],
    modules: dict[str, Any],
    optimizer: Any,
    scaler: Any,
    global_step: int,
    epoch: int,
    total_steps: int,
    base_seed: int,
    loss_history: list[float],
    wall_clock_s: float,
    sessions: list[dict[str, Any]],
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Assemble the full resume state.

    `epoch` is the *next* epoch to run, so a checkpoint written after finishing epoch 40
    carries epoch=41 and resuming needs no off-by-one reasoning at the call site.
    """
    ckpt: dict[str, Any] = {
        "format_version": FORMAT_VERSION,
        "run_id": run_id,
        "config_hash": config_hash,
        "resolved_cfg": resolved_cfg,
        "modules": {k: _state(v) for k, v in modules.items()},
        "optimizer": _state(optimizer),
        "scaler": _state(scaler),
        "global_step": int(global_step),
        "epoch": int(epoch),
        "total_steps": int(total_steps),
        "base_seed": int(base_seed),
        "world_size": get_world_size(),
        "loss_history": list(loss_history),
        "wall_clock_s": float(wall_clock_s),
        "sessions": list(sessions),
    }
    if extra:
        ckpt.update(extra)
    return ckpt


def save_checkpoint(ckpt: dict[str, Any], path: str | os.PathLike[str]) -> Path:
    """Atomically write a checkpoint. Rank 0 only; other ranks return the path."""
    path = Path(path)
    if not is_main():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(ckpt, tmp)
    os.replace(tmp, path)
    return path


def find_checkpoint(candidates: list[str | os.PathLike[str]]) -> Path | None:
    """First readable checkpoint from an ordered candidate list.

    Order matters: the mid-run Kaggle Dataset push comes first, the previous kernel
    version's committed output second (it is at most one session stale), the local
    working directory last (only valid within the current session).
    """
    for c in candidates:
        p = Path(c)
        if p.is_file() and p.stat().st_size > 0:
            return p
    return None


class ResumeMismatch(RuntimeError):
    """The checkpoint cannot be resumed from. Fatal — see `RunIdMismatch` for the
    benign case."""


class RunIdMismatch(ResumeMismatch):
    """The checkpoint belongs to a different run entirely.

    Distinct from config drift on purpose. A foreign checkpoint just means "not mine" —
    the caller should ignore it and start fresh. Config drift *within* the same run_id
    means the experiment definition changed under a resume, which silently shifts every
    schedule, and that must stop the run.
    """


def load_checkpoint(
    path: str | os.PathLike[str],
    *,
    expect_run_id: str,
    expect_config_hash: str,
    modules: dict[str, Any],
    optimizer: Any = None,
    scaler: Any = None,
    map_location: str = "cpu",
    strict: bool = True,
) -> dict[str, Any]:
    """Restore state in place and return the checkpoint's bookkeeping fields.

    Raises `ResumeMismatch` if the checkpoint belongs to a different run or config.
    That is deliberate — the caller should either start fresh or fix the config, never
    silently continue from an incompatible state.
    """
    ckpt = torch.load(path, map_location=map_location, weights_only=False)

    if ckpt.get("format_version") != FORMAT_VERSION:
        raise ResumeMismatch(
            f"checkpoint format v{ckpt.get('format_version')} != v{FORMAT_VERSION} ({path})"
        )
    if ckpt.get("run_id") != expect_run_id:
        raise RunIdMismatch(
            f"checkpoint is for run {ckpt.get('run_id')!r}, not {expect_run_id!r} ({path})"
        )
    if ckpt.get("config_hash") != expect_config_hash:
        raise ResumeMismatch(
            f"config drift: checkpoint hash {ckpt.get('config_hash')} != {expect_config_hash}. "
            "Something in the experiment config changed, which would shift the LR/WD/EMA "
            f"schedules. Start a new run_id or restore the original config. ({path})"
        )

    saved = ckpt.get("modules", {})
    for name, module in modules.items():
        if module is None:
            continue
        if name not in saved or saved[name] is None:
            if strict:
                raise ResumeMismatch(f"checkpoint has no state for module {name!r} ({path})")
            continue
        target = unwrap(module) if isinstance(module, torch.nn.Module) else module
        target.load_state_dict(saved[name])

    if optimizer is not None and ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scaler is not None and ckpt.get("scaler") is not None:
        scaler.load_state_dict(ckpt["scaler"])

    return ckpt


def session_record(start_time: float, epochs_done: int, accelerator: str) -> dict[str, Any]:
    """One row of the `sessions` audit trail.

    Worth keeping: it is where the thesis's compute table comes from, and it makes a
    mixed-hardware run (some sessions T4 x2, some P100) visible instead of invisible.
    """
    return {
        "start_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(start_time)),
        "end_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "duration_s": round(time.time() - start_time, 1),
        "epochs": epochs_done,
        "accelerator": accelerator,
        "world_size": get_world_size(),
    }


def _extract_prefix(state: dict[str, Any], prefix: str) -> dict[str, Any]:
    """Pull out `prefix.*` keys with the prefix stripped."""
    n = len(prefix)
    return {k[n:]: v for k, v in state.items() if k.startswith(prefix)}


def export_encoder(
    ckpt_path: str | os.PathLike[str],
    out_path: str | os.PathLike[str],
    prefer_target: bool = True,
) -> Path:
    """Strip a training checkpoint down to publishable encoder weights (~88 MB).

    Every SSL model here saves its whole state under `modules["model"]`, so the encoder
    is recovered by prefix rather than by a separate key. We prefer the *target* (EMA)
    encoder where one exists (I-JEPA, MoCo v3): the EMA weights are what those papers
    evaluate and they transfer better than the online encoder. MAE and SimCLR have no
    target branch and fall through to `encoder.`.
    """
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("modules", {}).get("model")
    if state is None:
        raise KeyError(f"checkpoint has no modules['model'] state: {ckpt_path}")

    source = ""
    enc: dict[str, Any] = {}
    if prefer_target:
        enc = _extract_prefix(state, "target_encoder.")
        source = "target_encoder"
    if not enc:
        enc = _extract_prefix(state, "encoder.")
        source = "encoder"
    if not enc:
        raise KeyError(f"no 'encoder.' or 'target_encoder.' keys in checkpoint {ckpt_path}")

    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "encoder": enc,
            "source_module": source,
            "run_id": ckpt.get("run_id"),
            "config_hash": ckpt.get("config_hash"),
            "resolved_cfg": ckpt.get("resolved_cfg"),
            "epoch": ckpt.get("epoch"),
            "global_step": ckpt.get("global_step"),
            "wall_clock_s": ckpt.get("wall_clock_s"),
        },
        out,
    )
    return out


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/utils/ddp.py
"""Minimal DDP helpers for Kaggle.

Kaggle gives you T4 x2 (world_size 2) or a single P100. There is no torchrun and no
srun, so a run is launched with `torch.multiprocessing.spawn` from inside the notebook
and the rendezvous env vars are set by hand.

Single-GPU and CPU are first-class: every helper degrades to a no-op so the same engine
code runs unchanged on a Mac for smoke tests.
"""
from __future__ import annotations

import os
from contextlib import contextmanager
from typing import Any

import torch
import torch.distributed as dist


def is_dist() -> bool:
    return dist.is_available() and dist.is_initialized()


def get_rank() -> int:
    return dist.get_rank() if is_dist() else 0


def get_world_size() -> int:
    return dist.get_world_size() if is_dist() else 1


def is_main() -> bool:
    return get_rank() == 0


def setup(rank: int, world_size: int, backend: str | None = None, port: str = "29517") -> None:
    """Initialise the process group. Safe to call with world_size=1 (does nothing)."""
    if world_size <= 1:
        return
    os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
    os.environ.setdefault("MASTER_PORT", port)
    if backend is None:
        backend = "nccl" if torch.cuda.is_available() else "gloo"
    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)
    if torch.cuda.is_available():
        torch.cuda.set_device(rank)


def cleanup() -> None:
    if is_dist():
        dist.barrier()
        dist.destroy_process_group()


def barrier() -> None:
    if is_dist():
        dist.barrier()


def all_reduce_mean(value: torch.Tensor | float) -> float:
    """Average a scalar across ranks. Used for logging, not for gradients."""
    if not is_dist():
        return float(value)
    t = value.detach().clone() if isinstance(value, torch.Tensor) else torch.tensor(float(value))
    t = t.to(torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
    dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return float(t.item() / dist.get_world_size())


class GatherWithGrad(torch.autograd.Function):
    """all_gather that propagates gradients back to the local shard.

    Plain `dist.all_gather` detaches — using it for contrastive embeddings would give
    each rank gradients only from its own slice, quietly reducing SimCLR's effective
    negative count from (global_batch - 1) back to (per_gpu_batch - 1). This is the
    standard fix and it is why SimCLR here really does see 511 negatives at
    global_batch 512, not 255.
    """

    @staticmethod
    def forward(ctx: Any, x: torch.Tensor) -> torch.Tensor:  # type: ignore[override]
        if not is_dist():
            return x
        ctx.rank = dist.get_rank()
        ctx.batch = x.shape[0]
        out = [torch.zeros_like(x) for _ in range(dist.get_world_size())]
        dist.all_gather(out, x.contiguous())
        return torch.cat(out, dim=0)

    @staticmethod
    def backward(ctx: Any, grad: torch.Tensor):  # type: ignore[override]
        if not is_dist():
            return grad
        dist.all_reduce(grad, op=dist.ReduceOp.SUM)
        start = ctx.rank * ctx.batch
        return grad[start : start + ctx.batch]


def gather_with_grad(x: torch.Tensor) -> torch.Tensor:
    return GatherWithGrad.apply(x)


def unwrap(model: torch.nn.Module) -> torch.nn.Module:
    """Strip DistributedDataParallel so state_dict keys have no `module.` prefix."""
    return model.module if hasattr(model, "module") else model


@contextmanager
def main_process_first():
    """Let rank 0 run a block (e.g. a download) before the others enter it."""
    if not is_dist():
        yield
        return
    if not is_main():
        dist.barrier()
    yield
    if is_main():
        dist.barrier()


def epoch_seed(base_seed: int, epoch: int) -> int:
    """Deterministic per-epoch seed.

    We derive the data order from (base_seed, epoch) rather than checkpointing and
    replaying RNG state. Bitwise RNG restoration across a DDP restart is fragile —
    worker count, cuDNN autotune and kernel selection all perturb it — whereas this
    reproduces the exact shuffle and augmentation stream for epoch k regardless of
    which session runs it.
    """
    return (base_seed * 100_003 + epoch) % (2**31 - 1)


def seed_everything(seed: int, deterministic: bool = False) -> None:
    import random

    import numpy as np

    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        # Shapes are fixed for the whole run (masks are truncated to min_keep), so
        # autotuning pays off and never re-triggers.
        torch.backends.cudnn.benchmark = True


def worker_init_fn(worker_id: int, base: int = 0) -> None:
    import random

    import numpy as np

    s = (base + worker_id) % (2**31 - 1)
    random.seed(s)
    np.random.seed(s % (2**32))
    torch.manual_seed(s)


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/utils/kaggle_io.py
"""Persisting state between Kaggle sessions.

`/kaggle/working` is wiped when a session ends, so a checkpoint that only lives there is
lost the moment the 9-hour cap hits. Three ways out, and they are not equivalent:

* **A Kaggle Dataset we own, versioned from inside the kernel.** Push-based, so it can
  happen *mid-run*. A crash at hour 7.4 costs one push interval, not the session.
* **The previous kernel version's committed output** (`kernel_sources` pointing at the
  notebook itself). Free and automatic, but only produced when a commit *succeeds* — and
  a run that exceeds the time limit fails, taking its output with it. Good mirror,
  unreliable primary.
* **Kaggle Models.** Built for release artefacts; too heavy for a 400 MB blob rewritten
  every 45 minutes. Used here only for the four final encoders.

So: dataset push for hot resume state, kernel output as a free mirror, and a session
guard that exits *cleanly* before the platform kills us so the commit succeeds.

Credentials come from Kaggle Secrets (Add-ons -> Secrets), never a committed
`kaggle.json`.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import time
from pathlib import Path

from src.config import on_kaggle
from src.utils.ddp import is_main


def load_kaggle_credentials(verbose: bool = True) -> bool:
    """Populate KAGGLE_USERNAME / KAGGLE_KEY from Kaggle Secrets.

    Returns False if credentials are unavailable — callers should degrade to local-only
    checkpointing rather than crash, since a run without dataset pushes is still a run.
    """
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]

        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
        return True
    except Exception as e:  # noqa: BLE001
        if verbose:
            print(
                f"[kaggle_io] no credentials ({e}). Checkpoints stay in /kaggle/working "
                "only and will be lost when the session ends. Add KAGGLE_USERNAME and "
                "KAGGLE_KEY under Add-ons -> Secrets to enable cross-session resume."
            )
        return False


def _run(cmd: list[str], timeout: float = 900.0) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)


class CheckpointPusher:
    """Versions a directory into a Kaggle Dataset on a time-based cadence.

    Cadence is time-based rather than epoch-based on purpose: epochs range from 2.7 min
    (MAE) to 12.8 min (MoCo v3), so "every N epochs" would mean wildly different
    exposure to a crash across methods. ~45 min caps the loss uniformly at ~10% of a
    session for roughly 4% overhead.
    """

    def __init__(
        self,
        slug: str,
        staging_dir: str | os.PathLike[str],
        title: str | None = None,
        push_minutes: float = 45.0,
        enabled: bool = True,
    ) -> None:
        self.slug = slug
        self.dir = Path(staging_dir)
        self.title = title or (slug.split("/")[-1] if slug else "checkpoints")
        self.push_interval = push_minutes * 60.0
        self.last_push = time.time()
        self.enabled = bool(enabled and slug and on_kaggle() and is_main())
        if self.enabled:
            self.enabled = load_kaggle_credentials()
        if self.enabled:
            self.dir.mkdir(parents=True, exist_ok=True)
            self._write_metadata()

    def _write_metadata(self) -> None:
        (self.dir / "dataset-metadata.json").write_text(
            json.dumps(
                {
                    "title": self.title,
                    "id": self.slug,
                    "licenses": [{"name": "CC0-1.0"}],
                },
                indent=2,
            )
        )

    def due(self) -> bool:
        return self.enabled and (time.time() - self.last_push) >= self.push_interval

    def push(self, message: str, force: bool = False) -> bool:
        """Create a new dataset version. Returns True on success.

        Never raises: a failed push (flaky network, quota) must not kill a training run
        that is otherwise healthy. The next push attempt will carry the same state.
        """
        if not self.enabled or (not force and not self.due()):
            return False
        self.last_push = time.time()
        cmd = [
            "kaggle", "datasets", "version",
            "-p", str(self.dir),
            "-m", message[:500],
            "--dir-mode", "zip",
            "--quiet",
        ]
        try:
            r = _run(cmd)
        except subprocess.TimeoutExpired:
            print("[kaggle_io] push timed out; continuing training")
            return False
        if r.returncode != 0:
            err = (r.stderr or r.stdout).strip().splitlines()
            if err and "does not exist" in err[-1].lower():
                print(
                    f"[kaggle_io] dataset {self.slug} does not exist yet. Create it once "
                    f"with:  kaggle datasets create -p {self.dir} --dir-mode zip --private"
                )
            else:
                print(f"[kaggle_io] push failed: {err[-1] if err else r.returncode}")
            return False
        print(f"[kaggle_io] pushed {self.slug}: {message}")
        return True


def resume_candidates(
    filename: str,
    ckpt_slug: str = "",
    kernel_slug: str = "",
    working_dir: str | os.PathLike[str] = "/kaggle/working",
) -> list[Path]:
    """Ordered places to look for a resume checkpoint, most-recent-first."""
    out: list[Path] = []
    if ckpt_slug:
        out.append(Path("/kaggle/input") / ckpt_slug.split("/")[-1] / filename)
    if kernel_slug:
        base = Path("/kaggle/input") / kernel_slug.split("/")[-1]
        out += [base / filename, base / "ckpt" / filename]
    out.append(Path(working_dir) / filename)
    out.append(Path(working_dir) / "ckpt" / filename)
    return out


class SessionGuard:
    """Stops training before Kaggle kills the session.

    Exiting cleanly matters more than it sounds: a notebook that runs past the platform
    limit is marked *failed*, and a failed commit produces no output — so the free
    `kernel_sources` mirror disappears exactly when it would have been most useful.
    Break out of the loop, save, push, and let the commit finish.
    """

    def __init__(self, hours: float = 7.5) -> None:
        self.start = time.time()
        self.limit = hours * 3600.0

    @property
    def elapsed(self) -> float:
        return time.time() - self.start

    @property
    def remaining(self) -> float:
        return max(0.0, self.limit - self.elapsed)

    def expired(self, margin_s: float = 0.0) -> bool:
        return self.elapsed + margin_s >= self.limit

    def would_exceed(self, next_epoch_s: float) -> bool:
        """True if another epoch probably would not finish before the guard fires."""
        return self.elapsed + next_epoch_s >= self.limit

    def summary(self) -> str:
        h, rem = divmod(int(self.elapsed), 3600)
        return f"{h}h{rem // 60:02d}m elapsed, {self.remaining / 3600:.1f}h before guard"


def stage_file(src: str | os.PathLike[str], staging_dir: str | os.PathLike[str]) -> Path:
    """Copy a file into the push staging directory (same-filesystem move if possible)."""
    src, staging = Path(src), Path(staging_dir)
    staging.mkdir(parents=True, exist_ok=True)
    dst = staging / src.name
    if src.resolve() == dst.resolve():
        return dst
    shutil.copy2(src, dst)
    return dst


def append_metrics(path: str | os.PathLike[str], record: dict) -> None:
    """Append one JSON line of metrics.

    Every figure script reads these files rather than a checkpoint, so figures can be
    iterated on a laptop with no GPU and no 400 MB download.
    """
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "a") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/engine/pretrain.py
"""Self-supervised pretraining engine, shared by I-JEPA, MAE, SimCLR and MoCo v3.

One engine for all four methods is what makes the comparison a comparison: the
optimiser, schedules, precision policy, data pipeline, checkpointing and logging are
literally the same code path, so the only thing that differs is the objective.

Kaggle shapes several decisions here:

* A run is chopped into 2-3 sessions by the ~9h cap, so the loop is resumable and the
  session guard exits *cleanly* before the platform kills us (a killed notebook is
  marked failed and its output — the free checkpoint mirror — is discarded).
* T4 is sm_75: fp16 tensor cores, no bf16 hardware. Precision comes from the capability
  gate in `src.config`, never from an assumption.
* Global batch is pinned; per-GPU batch is derived. Otherwise a session that lands on a
  1-GPU P100 instead of T4 x2 would silently halve the global batch and shift every
  schedule.
"""
from __future__ import annotations

import argparse
import json
import math
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, DistributedSampler
from tqdm import tqdm

from src.config import (
    PretrainCfg,
    amp_config,
    build_pretrain_cfg,
    config_dict,
    config_hash,
    default_output_dir,
    derive_batch,
    on_kaggle,
    resolve_dataset_dir,
)
from src.data.hyperkvasir import HyperKvasirUnlabeled
from src.data.transforms import GPUAugment, TwoViewTransform, make_pretrain_transform
from src.masks.multiblock import MaskCollator
from src.model import METHOD_SPECS, build_pretrain_model
from src.utils import ddp
from src.utils.checkpoint import (
    ResumeMismatch,
    RunIdMismatch,
    build_checkpoint,
    export_encoder,
    find_checkpoint,
    load_checkpoint,
    save_checkpoint,
    session_record,
)
from src.utils.kaggle_io import (
    CheckpointPusher,
    SessionGuard,
    append_metrics,
    resume_candidates,
    stage_file,
)
from src.utils.schedulers import ScheduleSet, cosine_momentum, param_groups_with_wd_exclusion

CKPT_NAME = "latest.pt"


# ------------------------------------------------------------------ data


def build_dataloader(cfg: PretrainCfg, per_gpu_batch: int, data_root: Path | None = None):
    spec = METHOD_SPECS[cfg.method]
    a, m = cfg.aug, cfg.model

    if spec["needs_two_views"]:
        # CPU produces uint8 crops; colour jitter / blur / normalise happen on the GPU.
        view = make_pretrain_transform(
            m.img_size, a.crop_scale, horizontal_flip=False, to_uint8=True
        )
        transform = TwoViewTransform(view)
    else:
        transform = make_pretrain_transform(
            m.img_size, a.crop_scale, a.horizontal_flip, to_uint8=False, mean=a.mean, std=a.std
        )

    exclude_file = Path(__file__).resolve().parents[2] / "splits" / "pretrain_excluded.txt"
    dataset = HyperKvasirUnlabeled(
        root=data_root,
        transform=transform,
        exclude_file=exclude_file if exclude_file.is_file() else None,
        cache_dir=default_output_dir(),
    )
    if ddp.is_main():
        print(f"[data] {dataset.describe()}")

    collate = None
    if spec["needs_masks"]:
        k = cfg.mask
        collate = MaskCollator(
            input_size=m.img_size,
            patch_size=m.patch_size,
            enc_mask_scale=tuple(k.enc_mask_scale),
            pred_mask_scale=tuple(k.pred_mask_scale),
            aspect_ratio=tuple(k.aspect_ratio),
            num_enc_masks=k.num_enc_masks,
            num_pred_masks=k.num_pred_masks,
            min_keep=k.min_keep,
            allow_overlap=k.allow_overlap,
        )

    sampler = (
        DistributedSampler(dataset, shuffle=True, drop_last=True, seed=cfg.runtime.seed)
        if ddp.is_dist()
        else None
    )
    r = cfg.runtime
    loader = DataLoader(
        dataset,
        batch_size=per_gpu_batch,
        shuffle=(sampler is None),
        sampler=sampler,
        num_workers=r.num_workers,
        pin_memory=r.pin_memory,
        drop_last=True,
        collate_fn=collate,
        persistent_workers=r.persistent_workers and r.num_workers > 0,
        prefetch_factor=r.prefetch_factor if r.num_workers > 0 else None,
    )
    return dataset, loader, sampler, collate


def build_gpu_augment(cfg: PretrainCfg, device: str) -> GPUAugment | None:
    if not METHOD_SPECS[cfg.method]["needs_two_views"]:
        return None
    a = cfg.aug
    return GPUAugment(
        color_jitter_strength=a.color_jitter_strength,
        color_distortion=a.color_distortion,
        gaussian_blur_p=0.5 if a.gaussian_blur else 0.0,
        solarize_p=0.2 if a.solarize else 0.0,
        horizontal_flip=a.horizontal_flip,
        mean=a.mean,
        std=a.std,
    ).to(device)


# ------------------------------------------------------------------ one step


def forward_loss(cfg: PretrainCfg, model: nn.Module, batch, device: str, gpu_aug) -> torch.Tensor:
    """Dispatch to the method's objective. The only method-specific code in the loop."""
    core = ddp.unwrap(model)
    method = cfg.method

    if method == "ijepa":
        imgs, masks_enc, masks_pred = batch
        imgs = imgs.to(device, non_blocking=True)
        masks_enc = [u.to(device, non_blocking=True) for u in masks_enc]
        masks_pred = [u.to(device, non_blocking=True) for u in masks_pred]
        z, h = core(imgs, masks_enc, masks_pred)
        return core.loss(z, h)

    if method == "mae":
        imgs = batch.to(device, non_blocking=True)
        loss, _, _ = core(imgs)
        return loss

    # simclr / mocov3
    v1, v2 = batch
    v1 = gpu_aug(v1.to(device, non_blocking=True))
    v2 = gpu_aug(v2.to(device, non_blocking=True))
    return core((v1, v2))


# ------------------------------------------------------------------ main loop


def pretrain(cfg: PretrainCfg, rank: int = 0, world_size: int = 1, data_root: Path | None = None) -> Path:
    ddp.setup(rank, world_size)
    amp = amp_config()
    device = f"cuda:{rank}" if amp.device == "cuda" else amp.device
    if amp.device == "cuda":
        torch.cuda.set_device(rank)
    ddp.seed_everything(cfg.runtime.seed + rank)

    spec = METHOD_SPECS[cfg.method]
    o = cfg.optim

    # Contrastive losses are batch-coupled: gradient accumulation does NOT reproduce a
    # larger batch for InfoNCE the way it does for a per-sample loss. So the global batch
    # can only be preserved by having the right world size.
    if spec["needs_two_views"] and o.accum_steps != 1:
        raise ValueError(
            f"{cfg.method}: accum_steps must be 1. InfoNCE is computed over the batch, so "
            "accumulation would silently shrink the negative set rather than emulate a "
            "larger batch. Run on the world size that divides global_batch instead."
        )
    per_gpu = derive_batch(o.global_batch, world_size, o.accum_steps)

    if ddp.is_main():
        print(f"[pretrain] {cfg.run_id}")
        print(f"[pretrain] device={amp.name} sm_{amp.sm} autocast={amp.dtype} scaler={amp.use_scaler}")
        print(
            f"[pretrain] global_batch={o.global_batch} = per_gpu {per_gpu} "
            f"x world {world_size} x accum {o.accum_steps}"
        )

    dataset, loader, sampler, collate = build_dataloader(cfg, per_gpu, data_root)
    gpu_aug = build_gpu_augment(cfg, device)

    model = build_pretrain_model(cfg).to(device)
    if world_size > 1:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[rank] if amp.device == "cuda" else None,
            # MoCo v3 freezes the patch embedding and MAE's decoder is only partly used
            # on some steps; DDP would otherwise error on unused parameters.
            find_unused_parameters=spec["has_ema"] or cfg.method == "mocov3",
        )
    core = ddp.unwrap(model)

    iters_per_epoch = len(loader) // o.accum_steps
    sched = ScheduleSet(
        iters_per_epoch=iters_per_epoch,
        epochs=o.epochs,
        warmup_epochs=o.warmup_epochs,
        start_lr=o.start_lr,
        ref_lr=o.ref_lr,
        final_lr=o.final_lr,
        weight_decay=o.weight_decay,
        final_weight_decay=o.final_weight_decay,
        ema=tuple(o.ema),
        ipe_scale=o.ipe_scale,
    )
    optimizer = torch.optim.AdamW(
        param_groups_with_wd_exclusion(core, base_lr=o.ref_lr), betas=(0.9, 0.95)
    )
    scaler = torch.amp.GradScaler("cuda", enabled=amp.use_scaler)

    out_dir = default_output_dir()
    # One checkpoint directory per run, so concurrent or sequential runs of different
    # methods never contend for the same file. Metrics stay in one shared file (each
    # record carries its method) so the figures can read every run from one place.
    ckpt_dir = out_dir / "ckpt" / cfg.run_id
    ckpt_path = ckpt_dir / CKPT_NAME
    metrics_path = out_dir / "ckpt" / "metrics.jsonl"
    cfg_hash = config_hash(cfg)
    resolved = config_dict(cfg)

    # ------------------------------------------------------------ resume
    start_epoch, global_step, loss_history, wall_clock, sessions = 0, 0, [], 0.0, []
    found = find_checkpoint(
        resume_candidates(CKPT_NAME, cfg.runtime.ckpt_dataset_slug, working_dir=str(ckpt_dir))
    )
    if found is not None:
        try:
            ck = load_checkpoint(
                found,
                expect_run_id=cfg.run_id,
                expect_config_hash=cfg_hash,
                modules=core.checkpoint_modules(),
                optimizer=optimizer,
                scaler=scaler,
                map_location=device,
            )
            start_epoch = ck["epoch"]
            global_step = ck["global_step"]
            loss_history = ck["loss_history"]
            wall_clock = ck["wall_clock_s"]
            sessions = ck["sessions"]
            if ddp.is_main():
                print(
                    f"[resume] {found} -> epoch {start_epoch}/{o.epochs}, step {global_step}, "
                    f"{wall_clock / 3600:.2f} GPU-h already spent"
                )
        except RunIdMismatch as e:
            # Someone else's checkpoint (e.g. a different method sharing the mounted
            # Kaggle dataset). Not an error — just not ours.
            if ddp.is_main():
                print(f"[resume] ignoring foreign checkpoint: {e}")
        except ResumeMismatch as e:
            # Same run, different config: the schedules would silently shift. Stop.
            if ddp.is_main():
                print(f"[resume] refusing to resume: {e}")
            raise

    if start_epoch >= o.epochs:
        if ddp.is_main():
            print(f"[pretrain] already complete ({start_epoch}/{o.epochs} epochs)")
            _finalise(cfg, ckpt_path, out_dir)
        ddp.cleanup()
        return ckpt_path

    pusher = CheckpointPusher(
        cfg.runtime.ckpt_dataset_slug,
        ckpt_dir,
        push_minutes=cfg.runtime.ckpt_push_minutes,
        enabled=bool(cfg.runtime.ckpt_dataset_slug),
    )
    guard = SessionGuard(cfg.runtime.session_guard_hours)
    session_start = time.time()

    def write_ckpt(epoch: int, message: str, push: bool = False) -> None:
        ck = build_checkpoint(
            run_id=cfg.run_id,
            config_hash=cfg_hash,
            resolved_cfg=resolved,
            modules=core.checkpoint_modules(),
            optimizer=optimizer,
            scaler=scaler,
            global_step=global_step,
            epoch=epoch,
            total_steps=sched.total_steps,
            base_seed=cfg.runtime.seed,
            loss_history=loss_history,
            wall_clock_s=wall_clock + (time.time() - session_start),
            sessions=sessions + [session_record(session_start, epoch - start_epoch, amp.name)],
        )
        save_checkpoint(ck, ckpt_path)
        if push and pusher.enabled:
            stage_file(ckpt_path, ckpt_dir)
            pusher.push(message, force=True)

    # ------------------------------------------------------------ train
    nan_streak = 0
    epochs_done = 0

    for epoch in range(start_epoch, o.epochs):
        model.train()
        if sampler is not None:
            sampler.set_epoch(epoch)
        if collate is not None:
            collate.set_epoch(epoch, iters_per_epoch)
        # Data order is a pure function of (base_seed, epoch), so resuming at epoch k
        # reproduces exactly the stream an uninterrupted run would have seen.
        ddp.seed_everything(ddp.epoch_seed(cfg.runtime.seed, epoch) + rank)

        t_epoch = time.time()
        running, n_batches = 0.0, 0
        pbar = (
            tqdm(loader, desc=f"{cfg.method} ep {epoch + 1}/{o.epochs}", disable=not ddp.is_main())
            if ddp.is_main()
            else loader
        )

        optimizer.zero_grad(set_to_none=True)
        for it, batch in enumerate(pbar):
            lr, wd = sched.apply(optimizer, global_step)

            with torch.autocast(
                device_type="cuda" if amp.device == "cuda" else "cpu",
                dtype=amp.dtype or torch.float32,
                enabled=amp.enabled,
            ):
                loss = forward_loss(cfg, model, batch, device, gpu_aug)
                loss = loss / o.accum_steps

            if not torch.isfinite(loss):
                nan_streak += 1
                optimizer.zero_grad(set_to_none=True)
                if nan_streak >= 20:
                    raise RuntimeError(
                        f"loss non-finite for {nan_streak} consecutive steps at epoch "
                        f"{epoch + 1}. Reload the last checkpoint and halve the LR "
                        f"(currently {lr:.2e})."
                    )
                continue
            nan_streak = 0

            scaler.scale(loss).backward()

            if (it + 1) % o.accum_steps == 0:
                if o.grad_clip > 0:
                    scaler.unscale_(optimizer)  # required before clipping
                    torch.nn.utils.clip_grad_norm_(core.parameters(), o.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                # EMA / momentum update, fp32 and outside autocast. See model docstrings:
                # in fp16 the (1-m) term rounds away and the target silently stops tracking.
                if cfg.method == "ijepa":
                    core.update_target_encoder(sched.momentum(global_step))
                elif cfg.method == "mocov3":
                    # MoCo v3 ramps its momentum on a *cosine*, not linearly like i-jepa.
                    core.momentum_update(
                        cosine_momentum(
                            global_step, sched.total_steps, tuple(cfg.contrastive.moco_momentum)
                        )
                    )
                global_step += 1

            running += float(loss.detach()) * o.accum_steps
            n_batches += 1
            if ddp.is_main() and isinstance(pbar, tqdm) and it % 10 == 0:
                pbar.set_postfix(loss=f"{running / max(1, n_batches):.4f}", lr=f"{lr:.2e}")

        epoch_loss = ddp.all_reduce_mean(running / max(1, n_batches))
        epoch_time = time.time() - t_epoch
        loss_history.append(epoch_loss)
        epochs_done += 1

        if ddp.is_main():
            print(
                f"[{cfg.method} ep {epoch + 1}/{o.epochs}] loss={epoch_loss:.4f} "
                f"lr={sched.lr(global_step):.2e} wd={sched.wd(global_step):.3f} "
                f"{epoch_time / 60:.1f} min/epoch | {guard.summary()}"
            )
            append_metrics(
                metrics_path,
                {
                    "run_id": cfg.run_id,
                    "method": cfg.method,
                    "epoch": epoch + 1,
                    "loss": epoch_loss,
                    "lr": sched.lr(global_step),
                    "wd": sched.wd(global_step),
                    "momentum": (
                        sched.momentum(global_step)
                        if cfg.method == "ijepa"
                        else cosine_momentum(
                            global_step, sched.total_steps, tuple(cfg.contrastive.moco_momentum)
                        )
                        if cfg.method == "mocov3"
                        else None
                    ),
                    "global_step": global_step,
                    "samples_seen": global_step * o.global_batch,
                    "epoch_time_s": round(epoch_time, 1),
                    "accelerator": amp.name,
                    "world_size": world_size,
                },
            )
            write_ckpt(epoch + 1, f"{cfg.run_id} ep={epoch + 1} loss={epoch_loss:.4f}", push=pusher.due())

        # Stop before Kaggle stops us, so the commit succeeds and its output survives.
        if guard.would_exceed(epoch_time):
            if ddp.is_main():
                print(
                    f"[guard] {guard.summary()}; next epoch (~{epoch_time / 60:.1f} min) would "
                    f"overrun. Stopping cleanly at epoch {epoch + 1}/{o.epochs}."
                )
                write_ckpt(epoch + 1, f"{cfg.run_id} guard-exit ep={epoch + 1}", push=True)
            break

    ddp.barrier()
    if ddp.is_main():
        done_epochs = min(start_epoch + epochs_done, o.epochs)
        write_ckpt(done_epochs, f"{cfg.run_id} session end", push=True)
        # Completion is decided by epochs finished, not by how the loop exited: the
        # guard can legitimately fire on the very last epoch, and that is still a
        # finished run that deserves its exported encoder.
        if done_epochs >= o.epochs:
            _finalise(cfg, ckpt_path, out_dir)
        else:
            print(
                f"[pretrain] {o.epochs - done_epochs} epochs remaining — "
                "re-run this notebook to continue."
            )
    ddp.cleanup()
    return ckpt_path


def _finalise(cfg: PretrainCfg, ckpt_path: Path, out_dir: Path) -> None:
    """Export publishable encoder weights once a run completes."""
    weights = out_dir / "weights" / f"{cfg.method}_{cfg.model.arch}_{cfg.model.img_size}.pt"
    export_encoder(ckpt_path, weights)
    print(f"[pretrain] run complete. Encoder exported to {weights} ({weights.stat().st_size / 1e6:.0f} MB)")


# ------------------------------------------------------------------ launcher


def _worker(rank: int, world_size: int, cfg: PretrainCfg, data_root: str | None) -> None:
    pretrain(cfg, rank, world_size, Path(data_root) if data_root else None)


def run(cfg: PretrainCfg, data_root: Path | None = None, world_size: int | None = None) -> None:
    """Launch training, spawning one process per GPU when there is more than one."""
    if world_size is None:
        world_size = torch.cuda.device_count() if torch.cuda.is_available() else 1

    spec = METHOD_SPECS[cfg.method]
    if spec["needs_two_views"] and on_kaggle() and world_size != 2:
        raise RuntimeError(
            f"{cfg.method} needs world_size 2 to keep global_batch={cfg.optim.global_batch} "
            f"with accum_steps=1, but found {world_size} GPU(s). Kaggle assigned a single-GPU "
            "accelerator (usually P100). Set Session options -> Accelerator -> GPU T4 x2 and "
            "re-run; the checkpoint is safe. (I-JEPA and MAE can absorb this with accum_steps, "
            "contrastive losses cannot.)"
        )

    if world_size > 1:
        torch.multiprocessing.spawn(
            _worker, args=(world_size, cfg, str(data_root) if data_root else None), nprocs=world_size
        )
    else:
        pretrain(cfg, 0, 1, data_root)


def main() -> None:
    ap = argparse.ArgumentParser(description="SSL pretraining")
    ap.add_argument("--method", required=True, choices=sorted(METHOD_SPECS))
    ap.add_argument("--config", default=None)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--global-batch", type=int, default=None)
    ap.add_argument("--accum-steps", type=int, default=None)
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--img-size", type=int, default=None)
    ap.add_argument("--patch-size", type=int, default=None)
    ap.add_argument("--arch", default=None)
    ap.add_argument("--num-workers", type=int, default=None)
    ap.add_argument("--data-root", default=None)
    ap.add_argument("--ckpt-slug", default=None, help="Kaggle dataset slug for cross-session resume")
    ap.add_argument("--guard-hours", type=float, default=None)
    ap.add_argument("--world-size", type=int, default=None)
    args = ap.parse_args()

    overrides: dict = {"optim": {}, "model": {}, "runtime": {}}
    if args.epochs is not None:
        overrides["optim"]["epochs"] = args.epochs
    if args.global_batch is not None:
        overrides["optim"]["global_batch"] = args.global_batch
    if args.accum_steps is not None:
        overrides["optim"]["accum_steps"] = args.accum_steps
    if args.lr is not None:
        overrides["optim"]["ref_lr"] = args.lr
    if args.img_size is not None:
        overrides["model"]["img_size"] = args.img_size
    if args.patch_size is not None:
        overrides["model"]["patch_size"] = args.patch_size
    if args.arch is not None:
        overrides["model"]["arch"] = args.arch
    if args.seed is not None:
        overrides["runtime"]["seed"] = args.seed
    if args.num_workers is not None:
        overrides["runtime"]["num_workers"] = args.num_workers
    if args.ckpt_slug is not None:
        overrides["runtime"]["ckpt_dataset_slug"] = args.ckpt_slug
    if args.guard_hours is not None:
        overrides["runtime"]["session_guard_hours"] = args.guard_hours
    overrides = {k: v for k, v in overrides.items() if v}

    cfg = build_pretrain_cfg(args.method, args.config, overrides)
    root = Path(args.data_root) if args.data_root else None
    if root is None and not on_kaggle():
        try:
            root = resolve_dataset_dir("hyperkvasir")
        except FileNotFoundError:
            pass
    run(cfg, root, args.world_size)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/engine/segment.py
"""Segmentation fine-tuning on Kvasir-SEG.

Runs identically for all five encoders (I-JEPA, MAE, SimCLR, MoCo v3, random init) —
same decoder, same initialisation, same schedule, same augmentation, same splits. Only
the encoder weights differ. That is the entire experiment.

Protocol decisions enforced here rather than left to discipline, because each is a place
where segmentation papers routinely leak:

* **Model selection uses val, never test.** The checkpoint is chosen by best val Dice
  with early stopping. `evaluate_test` refuses to run unless `training_complete` is set,
  so the common error of reporting `max over epochs of test Dice` is not reachable by
  accident.
* **Val and test transforms have no augmentation.** They are separate dataset objects
  with their own eval transform, not a `Subset` of the augmented training set.
* **Test metrics are computed at native resolution** via `src.eval.metrics`.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.config import (
    SegCfg,
    amp_config,
    build_seg_cfg,
    config_dict,
    config_hash,
    default_output_dir,
    resolve_dataset_dir,
)
from src.data.kvasir_seg import KvasirSegDataset, read_split
from src.data.transforms import make_seg_transforms
from src.eval.metrics import aggregate, dice_from_logits, evaluate_dataset
from src.model.segformer_head import ViTSegFormer
from src.utils import ddp
from src.utils.kaggle_io import append_metrics
from src.utils.schedulers import ScheduleSet


# ------------------------------------------------------------------ loss


class DiceBCELoss(nn.Module):
    """Dice + BCE. Dice handles the class imbalance (polyps are a small minority of
    pixels), BCE keeps the gradient well-behaved where Dice saturates."""

    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5, eps: float = 1e-6) -> None:
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.eps = eps

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # fp32: BCE-with-logits is numerically delicate under fp16 autocast.
        logits, target = logits.float(), target.float()
        bce = F.binary_cross_entropy_with_logits(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = 1 - ((2 * inter + self.eps) / (denom + self.eps)).mean()
        return self.bce_weight * bce + self.dice_weight * dice


# ------------------------------------------------------------------ model


def build_seg_model(cfg: SegCfg) -> nn.Module:
    m = cfg.model
    if cfg.decoder == "segformer":
        return ViTSegFormer(
            arch=m.arch,
            img_size=m.img_size,
            patch_size=m.patch_size,
            fpn_layers=tuple(cfg.fpn_layers),
            decoder_embed_dim=cfg.decoder_embed_dim,
            drop_path_rate=m.drop_path_rate,
        )
    if cfg.decoder == "unet":
        from src.model.unet import ViTUNet

        return ViTUNet(
            arch=m.arch,
            img_size=m.img_size,
            patch_size=m.patch_size,
            skip_layers=tuple(cfg.fpn_layers[:3]),
        )
    raise KeyError(f"unknown decoder {cfg.decoder!r}")


def resolve_encoder_ckpt(cfg: SegCfg, weights_dir: Path) -> Path | None:
    """Find the exported encoder for this arm. `random` deliberately returns None."""
    if cfg.encoder == "random":
        return None
    if cfg.pretrained_ckpt:
        p = Path(cfg.pretrained_ckpt)
        if not p.is_file():
            raise FileNotFoundError(f"pretrained_ckpt not found: {p}")
        return p
    pattern = f"{cfg.encoder}_{cfg.model.arch}_*.pt"
    matches = sorted(weights_dir.glob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"no exported encoder matching {weights_dir / pattern}. Run pretraining for "
            f"'{cfg.encoder}' first, or pass --pretrained-ckpt."
        )
    return matches[-1]


# ------------------------------------------------------------------ data


def build_seg_loaders(cfg: SegCfg, root: Path | None = None):
    train_tf, eval_tf = make_seg_transforms(cfg.model.img_size)
    root = root or resolve_dataset_dir("kvasir_seg")

    ds_train = KvasirSegDataset(
        root,
        stems=read_split(cfg.split, "train"),
        transform=train_tf,
        label_fraction=cfg.label_fraction,
        seed=cfg.runtime.seed,
    )
    # Separate dataset objects with the eval transform — never a Subset of the augmented
    # training set, which would randomly flip validation images.
    ds_val = KvasirSegDataset(root, stems=read_split(cfg.split, "val"), transform=eval_tf)
    ds_test = KvasirSegDataset(root, stems=read_split(cfg.split, "test"), transform=eval_tf)

    dl_train = DataLoader(
        ds_train,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.runtime.num_workers,
        pin_memory=True,
        drop_last=len(ds_train) > cfg.batch_size,
        persistent_workers=cfg.runtime.num_workers > 0,
    )
    dl_val = DataLoader(
        ds_val, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.runtime.num_workers
    )
    return (ds_train, ds_val, ds_test), (dl_train, dl_val)


# ------------------------------------------------------------------ train


def segment(cfg: SegCfg, data_root: Path | None = None) -> dict:
    amp = amp_config()
    device = "cuda:0" if amp.device == "cuda" else amp.device
    ddp.seed_everything(cfg.runtime.seed)

    out_dir = default_output_dir()
    run_dir = out_dir / "seg" / cfg.run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    (ds_train, ds_val, ds_test), (dl_train, dl_val) = build_seg_loaders(cfg, data_root)
    print(f"[segment] {cfg.run_id}")
    print(f"[segment] train={len(ds_train)} val={len(ds_val)} test={len(ds_test)} split={cfg.split}")

    model = build_seg_model(cfg).to(device)
    weights_dir = out_dir / "weights"
    ckpt = resolve_encoder_ckpt(cfg, weights_dir)
    if ckpt is not None:
        report = model.load_pretrained_encoder(str(ckpt))
        print(f"[segment] encoder <- {ckpt.name} (from {report['source_module']}, run {report['run_id']})")
    else:
        print("[segment] encoder: random initialisation (control arm)")

    n_enc = sum(p.numel() for p in model.encoder.parameters())
    n_dec = sum(p.numel() for p in model.parameters()) - n_enc
    print(f"[segment] params: encoder {n_enc / 1e6:.2f}M  decoder {n_dec / 1e6:.2f}M")

    criterion = DiceBCELoss(cfg.bce_weight, cfg.dice_weight)
    optimizer = torch.optim.AdamW(
        model.param_groups(cfg.enc_lr, cfg.dec_lr, cfg.layer_decay), betas=(0.9, 0.999)
    )
    iters = max(1, len(dl_train))
    sched = ScheduleSet(
        iters_per_epoch=iters,
        epochs=cfg.epochs,
        warmup_epochs=cfg.warmup_epochs,
        start_lr=cfg.enc_lr * 0.1,
        ref_lr=cfg.enc_lr,
        final_lr=cfg.enc_lr * 0.01,
        weight_decay=cfg.weight_decay,
        final_weight_decay=cfg.weight_decay,
        ema=(0.0, 0.0),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=amp.use_scaler)

    best_dice, best_epoch, patience = -1.0, -1, 0
    best_path = run_dir / "best.pt"
    metrics_path = run_dir / "metrics.jsonl"
    step = 0
    t0 = time.time()

    for epoch in range(cfg.epochs):
        model.train()
        running, n = 0.0, 0
        for imgs, masks in tqdm(dl_train, desc=f"seg ep {epoch + 1}/{cfg.epochs}", leave=False):
            sched.apply(optimizer, step)
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda" if amp.device == "cuda" else "cpu",
                dtype=amp.dtype or torch.float32,
                enabled=amp.enabled,
            ):
                loss = criterion(model(imgs), masks)

            scaler.scale(loss).backward()
            if cfg.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.detach())
            n += 1
            step += 1

        # -- validation: no augmentation, deterministic
        model.eval()
        val_dice, val_loss, vb = 0.0, 0.0, 0
        with torch.no_grad():
            for imgs, masks in dl_val:
                imgs, masks = imgs.to(device), masks.to(device)
                with torch.autocast(
                    device_type="cuda" if amp.device == "cuda" else "cpu",
                    dtype=amp.dtype or torch.float32,
                    enabled=amp.enabled,
                ):
                    logits = model(imgs)
                val_loss += float(criterion(logits, masks))
                val_dice += dice_from_logits(logits, masks)
                vb += 1
        val_dice /= max(1, vb)
        val_loss /= max(1, vb)
        train_loss = running / max(1, n)
        improved = val_dice > best_dice

        print(
            f"[seg ep {epoch + 1}/{cfg.epochs}] train_loss={train_loss:.4f} "
            f"val_loss={val_loss:.4f} val_dice={val_dice:.4f}" + ("  <- best" if improved else "")
        )
        append_metrics(
            metrics_path,
            {
                "run_id": cfg.run_id,
                "encoder": cfg.encoder,
                "decoder": cfg.decoder,
                "seed": cfg.runtime.seed,
                "label_fraction": cfg.label_fraction,
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_dice": val_dice,
            },
        )

        if improved:
            best_dice, best_epoch, patience = val_dice, epoch + 1, 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch + 1,
                    "val_dice": val_dice,
                    "config_hash": config_hash(cfg),
                    "resolved_cfg": config_dict(cfg),
                },
                best_path,
            )
        else:
            patience += 1
            if patience >= cfg.early_stop_patience:
                print(f"[segment] early stop at epoch {epoch + 1} (best {best_dice:.4f} @ {best_epoch})")
                break

    train_minutes = (time.time() - t0) / 60
    print(
        f"[segment] training done in {train_minutes:.1f} min; "
        f"best val Dice {best_dice:.4f} @ epoch {best_epoch}"
    )

    summary = evaluate_test(cfg, model, ds_test, best_path, device, amp, run_dir, training_complete=True)
    summary.update(
        {
            "best_val_dice": best_dice,
            "best_epoch": best_epoch,
            "train_minutes": round(train_minutes, 1),
            "encoder_params": n_enc,
            "decoder_params": n_dec,
        }
    )
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return summary


def evaluate_test(
    cfg: SegCfg,
    model: nn.Module,
    ds_test,
    best_path: Path,
    device: str,
    amp,
    run_dir: Path,
    training_complete: bool = False,
) -> dict:
    """Score the held-out test set. Refuses to run mid-training.

    The guard is the point: it makes 'peek at test each epoch and report the best' an
    error rather than a temptation.
    """
    if not training_complete:
        raise RuntimeError(
            "evaluate_test called before training finished. The test set is scored exactly "
            "once, on the best-val checkpoint. Use the validation loader for any "
            "during-training signal."
        )
    if best_path.is_file():
        model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])

    results = evaluate_dataset(
        model,
        ds_test,
        device=device,
        amp_dtype=amp.dtype,
        save_predictions=run_dir / "test_predictions.npz",
    )
    (run_dir / "test_per_image.json").write_text(
        json.dumps([r.as_dict() for r in results], indent=2) + "\n"
    )
    summary = aggregate(results)
    summary.update(
        {
            "run_id": cfg.run_id,
            "encoder": cfg.encoder,
            "decoder": cfg.decoder,
            "seed": cfg.runtime.seed,
            "label_fraction": cfg.label_fraction,
            "split": cfg.split,
        }
    )
    print(
        f"[test] Dice={summary['dice']:.4f} IoU={summary['iou']:.4f} "
        f"HD95={summary['hd95']:.1f}px failure_rate={summary['failure_rate']:.2%} "
        f"(n={summary['n']}, hd95 substituted {summary['hd95_substituted']}x)"
    )
    return summary


def main() -> None:
    ap = argparse.ArgumentParser(description="Segmentation fine-tuning on Kvasir-SEG")
    ap.add_argument("--encoder", default=None, help="ijepa|mae|simclr|mocov3|random")
    ap.add_argument("--decoder", default=None, choices=["segformer", "unet"])
    ap.add_argument("--config", default=None)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--batch-size", type=int, default=None)
    ap.add_argument("--img-size", type=int, default=None)
    ap.add_argument("--patch-size", type=int, default=None)
    ap.add_argument("--arch", default=None)
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--label-fraction", type=float, default=None)
    ap.add_argument("--split", default=None)
    ap.add_argument("--pretrained-ckpt", default=None)
    ap.add_argument("--num-workers", type=int, default=None)
    ap.add_argument("--data-root", default=None)
    args = ap.parse_args()

    ov: dict = {"model": {}, "runtime": {}}
    for key, val in [
        ("encoder", args.encoder),
        ("decoder", args.decoder),
        ("epochs", args.epochs),
        ("batch_size", args.batch_size),
        ("label_fraction", args.label_fraction),
        ("split", args.split),
        ("pretrained_ckpt", args.pretrained_ckpt),
    ]:
        if val is not None:
            ov[key] = val
    if args.img_size is not None:
        ov["model"]["img_size"] = args.img_size
    if args.patch_size is not None:
        ov["model"]["patch_size"] = args.patch_size
    if args.arch is not None:
        ov["model"]["arch"] = args.arch
    if args.seed is not None:
        ov["runtime"]["seed"] = args.seed
    if args.num_workers is not None:
        ov["runtime"]["num_workers"] = args.num_workers
    ov = {k: v for k, v in ov.items() if v != {}}

    cfg = build_seg_cfg(args.config, ov)
    segment(cfg, Path(args.data_root) if args.data_root else None)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/eval/__init__.py
from src.eval.metrics import (
    aggregate,
    binary_metrics,
    evaluate_dataset,
    hd95,
)

__all__ = ["aggregate", "binary_metrics", "evaluate_dataset", "hd95"]


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/eval/metrics.py
"""Segmentation metrics, computed at native resolution.

Three conventions are fixed here on purpose, because each is a place where published
polyp-segmentation numbers quietly disagree with each other:

1. **Native resolution.** Predictions are made at 352x352, then the *logits* are
   bilinearly upsampled to the image's original H x W and thresholded there, and compared
   against the original mask. Thresholding first and upsampling the binary mask, or
   comparing at 352 against a downsampled ground truth, inflates Dice by roughly 1-2
   points and is not comparable to the literature.

2. **Per-image mean, not dataset-aggregated.** Dice computed by pooling all pixels across
   the test set differs substantially from the mean of per-image Dice, because large
   polyps dominate the pooled version. The per-image mean is the clinically meaningful
   one and is this thesis's primary endpoint. Both are reported so the difference is
   visible.

3. **An explicit empty-prediction convention for HD95.** If the prediction is empty and
   the ground truth is not, the Hausdorff distance is undefined. Silently dropping those
   cases as NaN biases the metric toward methods that fail *completely* rather than
   partially — exactly backwards. We substitute the image diagonal and report how many
   times that happened.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

EPS = 1e-7


@dataclass
class ImageResult:
    stem: str
    dice: float
    iou: float
    precision: float
    recall: float
    hd95: float
    hd95_substituted: bool
    gt_area: float
    pred_area: float
    height: int
    width: int
    extra: dict = field(default_factory=dict)

    def as_dict(self) -> dict:
        return asdict(self)


def binary_metrics(pred: np.ndarray, gt: np.ndarray) -> dict[str, float]:
    """Dice / IoU / precision / recall for one boolean image pair.

    When both masks are empty the prediction is perfect, so Dice and IoU are 1.0. That
    case does not arise in Kvasir-SEG (every image contains a polyp) but the convention
    matters for the external CVC-ClinicDB evaluation.
    """
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = float(np.logical_and(pred, gt).sum())
    fp = float(np.logical_and(pred, ~gt).sum())
    fn = float(np.logical_and(~pred, gt).sum())

    if tp + fp + fn == 0:
        return {"dice": 1.0, "iou": 1.0, "precision": 1.0, "recall": 1.0}

    return {
        "dice": 2 * tp / (2 * tp + fp + fn + EPS),
        "iou": tp / (tp + fp + fn + EPS),
        "precision": tp / (tp + fp + EPS) if (tp + fp) > 0 else 0.0,
        "recall": tp / (tp + fn + EPS) if (tp + fn) > 0 else 0.0,
    }


def _surface_distances(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Distances from every boundary pixel of `a` to the nearest boundary pixel of `b`."""
    from scipy.ndimage import binary_erosion, distance_transform_edt

    a_border = a ^ binary_erosion(a)
    b_border = b ^ binary_erosion(b)
    if not a_border.any() or not b_border.any():
        return np.array([])
    dt = distance_transform_edt(~b_border)
    return dt[a_border]


def hd95(pred: np.ndarray, gt: np.ndarray, diagonal: float) -> tuple[float, bool]:
    """95th-percentile symmetric Hausdorff distance in pixels.

    Returns (value, substituted). `substituted` is True when one mask was empty and the
    image diagonal was used instead — see module docstring.
    """
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any():
        return 0.0, False
    if not pred.any() or not gt.any():
        return float(diagonal), True

    try:
        d1 = _surface_distances(pred, gt)
        d2 = _surface_distances(gt, pred)
    except ImportError:
        return float("nan"), False
    if d1.size == 0 or d2.size == 0:
        return float(diagonal), True
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95))), False


def logits_to_native_mask(
    logits: torch.Tensor, height: int, width: int, threshold: float = 0.5
) -> np.ndarray:
    """Upsample logits to native resolution, then threshold. Order matters — see module docstring."""
    if logits.ndim == 3:
        logits = logits.unsqueeze(0)
    up = F.interpolate(logits.float(), size=(height, width), mode="bilinear", align_corners=False)
    return (torch.sigmoid(up)[0, 0] > threshold).cpu().numpy()


@torch.no_grad()
def evaluate_dataset(
    model: torch.nn.Module,
    dataset,
    device: str = "cuda",
    amp_dtype: torch.dtype | None = None,
    threshold: float = 0.5,
    compute_hd95: bool = True,
    save_predictions: Path | None = None,
) -> list[ImageResult]:
    """Run the model over a dataset and score every image at its native resolution.

    Batch size is 1 because each image has a different native size; the cost is
    negligible (100 test images) and it keeps the resolution handling unambiguous.
    """
    from PIL import Image

    model.eval()
    results: list[ImageResult] = []
    preds_to_save: dict[str, np.ndarray] = {}

    for idx in range(len(dataset)):
        img, _mask = dataset[idx][:2]
        img_t = img.unsqueeze(0).to(device)

        with torch.autocast(
            device_type="cuda" if str(device).startswith("cuda") else "cpu",
            dtype=amp_dtype or torch.float32,
            enabled=amp_dtype is not None,
        ):
            logits = model(img_t)

        # Ground truth is read from disk at full resolution, never from the resized
        # tensor the model saw.
        gt_path = dataset.native_mask_path(idx)
        gt = np.asarray(Image.open(gt_path).convert("L"), dtype=np.uint8) > 127
        h, w = gt.shape

        pred = logits_to_native_mask(logits, h, w, threshold)
        m = binary_metrics(pred, gt)
        diag = float(np.hypot(h, w))
        hd, sub = hd95(pred, gt, diag) if compute_hd95 else (float("nan"), False)

        stem = dataset.samples[idx][0].stem
        results.append(
            ImageResult(
                stem=stem,
                dice=m["dice"],
                iou=m["iou"],
                precision=m["precision"],
                recall=m["recall"],
                hd95=hd,
                hd95_substituted=sub,
                gt_area=float(gt.mean()),
                pred_area=float(pred.mean()),
                height=h,
                width=w,
            )
        )
        if save_predictions is not None:
            preds_to_save[stem] = np.packbits(pred)  # 8x smaller than bool

    if save_predictions is not None and preds_to_save:
        save_predictions.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            save_predictions,
            shapes=np.array([[r.height, r.width] for r in results]),
            stems=np.array([r.stem for r in results]),
            **preds_to_save,
        )
    return results


def aggregate(results: list[ImageResult]) -> dict[str, float]:
    """Summary statistics over a set of per-image results."""
    if not results:
        return {}
    d = np.array([r.dice for r in results])
    i = np.array([r.iou for r in results])
    p = np.array([r.precision for r in results])
    rc = np.array([r.recall for r in results])
    hd = np.array([r.hd95 for r in results], dtype=float)

    return {
        "n": len(results),
        "dice": float(d.mean()),
        "dice_std": float(d.std(ddof=1)) if len(d) > 1 else 0.0,
        "dice_median": float(np.median(d)),
        "iou": float(i.mean()),
        "precision": float(p.mean()),
        "recall": float(rc.mean()),
        "hd95": float(np.nanmean(hd)),
        "hd95_substituted": int(sum(r.hd95_substituted for r in results)),
        # The most clinically interpretable single number: how often the model
        # essentially missed the polyp.
        "failure_rate": float((d < 0.5).mean()),
    }


def dice_from_logits(logits: torch.Tensor, target: torch.Tensor, threshold: float = 0.5) -> float:
    """Fast in-training Dice at model resolution, for the val-selection signal only.

    Deliberately separate from the native-resolution path above: this one runs every
    epoch on the GPU and only needs to rank checkpoints, not to be reported.
    """
    pred = (torch.sigmoid(logits) > threshold).float()
    target = target.float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    denom = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return float(((2 * inter + EPS) / (denom + EPS)).mean())


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/eval/stats.py
"""Statistical comparison of the encoder arms.

The design choices here are the ones a thesis committee will ask about, so each is
justified in place:

* **Unit of analysis is the test image (n=100), paired across methods** — not the seed.
  Seeds give n=5, which is hopelessly underpowered: with 5 samples per arm you cannot
  detect anything smaller than a very large effect. Per-image pairing also removes
  image difficulty as a nuisance variable, which is the dominant source of variance in
  polyp segmentation.

* **Wilcoxon signed-rank, not a paired t-test.** Per-image Dice is bounded on [0,1],
  strongly left-skewed, and has a point mass near 0 for completely-missed polyps. The
  t-test's normality assumption is visibly violated; the sign-rank test only needs
  symmetry of the *differences*, which is far weaker.

* **Holm-Bonferroni**, not raw p-values, because we make one planned comparison per
  baseline. Holm is uniformly more powerful than Bonferroni at the same family-wise
  error rate, so there is no reason to use the latter.

* **A BCa bootstrap CI on the effect size, always.** A significant p-value on a +0.004
  Dice difference is not a result. The interval is what makes that visible.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np


@dataclass
class Comparison:
    method_a: str
    method_b: str
    n: int
    median_diff: float
    mean_diff: float
    ci_low: float
    ci_high: float
    statistic: float
    p_raw: float
    p_adjusted: float
    significant: bool
    n_better: int
    n_worse: int
    n_tied: int

    def as_dict(self) -> dict:
        return asdict(self)


# ------------------------------------------------------------------ bootstrap


def bca_ci(
    x: np.ndarray,
    statistic=np.median,
    alpha: float = 0.05,
    n_boot: int = 10_000,
    seed: int = 0,
) -> tuple[float, float]:
    """Bias-corrected and accelerated bootstrap confidence interval.

    BCa rather than the percentile bootstrap because the sampling distribution of a
    median of skewed paired differences is itself skewed, and the percentile interval is
    then noticeably off-centre. BCa corrects for both the bias and the skew.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 2:
        return (float("nan"), float("nan"))

    rng = np.random.RandomState(seed)
    theta_hat = float(statistic(x))
    boot = np.array([statistic(x[rng.randint(0, n, n)]) for _ in range(n_boot)])

    # z0: bias correction, from the fraction of bootstrap replicates below the estimate.
    prop = float(np.mean(boot < theta_hat))
    if prop <= 0 or prop >= 1:  # degenerate; fall back to percentile
        return float(np.percentile(boot, 100 * alpha / 2)), float(
            np.percentile(boot, 100 * (1 - alpha / 2))
        )
    z0 = _norm_ppf(prop)

    # a: acceleration, from the jackknife skewness.
    jack = np.array([statistic(np.delete(x, i)) for i in range(n)])
    jack_mean = jack.mean()
    num = float(((jack_mean - jack) ** 3).sum())
    den = float(6.0 * (((jack_mean - jack) ** 2).sum() ** 1.5))
    a = num / den if den != 0 else 0.0

    def adjust(p: float) -> float:
        z = _norm_ppf(p)
        return _norm_cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))

    lo = float(np.percentile(boot, 100 * adjust(alpha / 2)))
    hi = float(np.percentile(boot, 100 * adjust(1 - alpha / 2)))
    return lo, hi


def _norm_cdf(z: float) -> float:
    from math import erf, sqrt

    return 0.5 * (1.0 + erf(z / sqrt(2.0)))


def _norm_ppf(p: float) -> float:
    try:
        from scipy.stats import norm

        return float(norm.ppf(p))
    except ImportError:
        # Acklam's rational approximation; accurate to ~1e-9, plenty for CI endpoints.
        if not 0 < p < 1:
            return float("inf") if p >= 1 else float("-inf")
        a = [-3.969683028665376e01, 2.209460984245205e02, -2.759285104469687e02,
             1.383577518672690e02, -3.066479806614716e01, 2.506628277459239e00]
        b = [-5.447609879822406e01, 1.615858368580409e02, -1.556989798598866e02,
             6.680131188771972e01, -1.328068155288572e01]
        c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e00,
             -2.549732539343734e00, 4.374664141464968e00, 2.938163982698783e00]
        d = [7.784695709041462e-03, 3.224671290700398e-01, 2.445134137142996e00,
             3.754408661907416e00]
        pl, ph = 0.02425, 1 - 0.02425
        if p < pl:
            q = (-2 * np.log(p)) ** 0.5
            return float((((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) /
                         ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1))
        if p > ph:
            q = (-2 * np.log(1 - p)) ** 0.5
            return float(-(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) /
                         ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1))
        q = p - 0.5
        r = q * q
        return float((((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q /
                     (((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1))


# ------------------------------------------------------------------ tests


def wilcoxon(diff: np.ndarray) -> tuple[float, float]:
    """Two-sided Wilcoxon signed-rank test. Returns (statistic, p-value)."""
    try:
        from scipy.stats import wilcoxon as _w

        nz = diff[diff != 0]
        if len(nz) == 0:
            return 0.0, 1.0
        stat, p = _w(diff, zero_method="wilcox", alternative="two-sided")
        return float(stat), float(p)
    except ImportError:
        # Normal approximation with tie correction — adequate at n=100.
        nz = diff[diff != 0]
        n = len(nz)
        if n == 0:
            return 0.0, 1.0
        order = np.argsort(np.abs(nz))
        ranks = np.empty(n, dtype=float)
        ranks[order] = np.arange(1, n + 1)
        w_plus = ranks[nz > 0].sum()
        mu = n * (n + 1) / 4
        sigma = (n * (n + 1) * (2 * n + 1) / 24) ** 0.5
        z = (w_plus - mu) / sigma if sigma > 0 else 0.0
        return float(w_plus), float(2 * (1 - _norm_cdf(abs(z))))


def holm_bonferroni(pvals: list[float], alpha: float = 0.05) -> tuple[list[float], list[bool]]:
    """Holm step-down adjustment. Returns (adjusted p-values, reject flags)."""
    m = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * pvals[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted.tolist(), [bool(a <= alpha) for a in adjusted]


# ------------------------------------------------------------------ pipeline


def load_per_image_dice(seg_dir: Path) -> dict[str, dict[str, dict[str, float]]]:
    """Read every run's per-image test results.

    Returns {encoder: {seed: {stem: dice}}} from `outputs/seg/*/test_per_image.json`,
    keyed by the encoder and seed recorded in each run's summary.json.
    """
    out: dict[str, dict[str, dict[str, float]]] = {}
    for run in sorted(Path(seg_dir).glob("*/test_per_image.json")):
        summary_p = run.parent / "summary.json"
        if not summary_p.is_file():
            continue
        summary = json.loads(summary_p.read_text())
        # The low-label arms are a separate experiment; keep them out of the main table.
        if summary.get("label_fraction", 1.0) != 1.0:
            continue
        enc = summary["encoder"]
        seed = str(summary.get("seed", 0))
        records = json.loads(run.read_text())
        out.setdefault(enc, {})[seed] = {r["stem"]: r["dice"] for r in records}
    return out


def aggregate_over_seeds(per_seed: dict[str, dict[str, float]]) -> dict[str, float]:
    """Mean per-image Dice across seeds -> one value per image.

    This is the aggregation step that turns 5 noisy runs into one paired observation per
    image, which is what the n=100 test then operates on.
    """
    stems = set.intersection(*(set(d) for d in per_seed.values())) if per_seed else set()
    return {s: float(np.mean([d[s] for d in per_seed.values()])) for s in sorted(stems)}


def compare_all(
    dice_by_encoder: dict[str, dict[str, float]],
    reference: str = "ijepa",
    alpha: float = 0.05,
    n_boot: int = 10_000,
    seed: int = 0,
) -> list[Comparison]:
    """Paired comparison of `reference` against every other arm."""
    if reference not in dice_by_encoder:
        raise KeyError(f"reference {reference!r} not among {sorted(dice_by_encoder)}")

    others = [k for k in sorted(dice_by_encoder) if k != reference]
    raw: list[Comparison] = []
    for other in others:
        stems = sorted(set(dice_by_encoder[reference]) & set(dice_by_encoder[other]))
        a = np.array([dice_by_encoder[reference][s] for s in stems])
        b = np.array([dice_by_encoder[other][s] for s in stems])
        diff = a - b
        stat, p = wilcoxon(diff)
        lo, hi = bca_ci(diff, np.median, alpha, n_boot, seed)
        raw.append(
            Comparison(
                method_a=reference,
                method_b=other,
                n=len(stems),
                median_diff=float(np.median(diff)),
                mean_diff=float(diff.mean()),
                ci_low=lo,
                ci_high=hi,
                statistic=stat,
                p_raw=p,
                p_adjusted=p,
                significant=False,
                n_better=int((diff > 0).sum()),
                n_worse=int((diff < 0).sum()),
                n_tied=int((diff == 0).sum()),
            )
        )

    adj, reject = holm_bonferroni([c.p_raw for c in raw], alpha)
    for c, a_, r in zip(raw, adj, reject):
        c.p_adjusted = a_
        c.significant = r
    return raw


def run(seg_dir: Path, out_path: Path, reference: str = "ijepa") -> list[Comparison]:
    by_encoder = load_per_image_dice(seg_dir)
    if not by_encoder:
        raise FileNotFoundError(f"no completed segmentation runs under {seg_dir}")

    dice = {enc: aggregate_over_seeds(per_seed) for enc, per_seed in by_encoder.items()}
    for enc, per_seed in by_encoder.items():
        print(f"[stats] {enc}: {len(per_seed)} seed(s), {len(dice[enc])} test images")

    comparisons = compare_all(dice, reference=reference)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps([c.as_dict() for c in comparisons], indent=2) + "\n")

    print(f"\n{'comparison':28s} {'median Δ':>10s} {'95% BCa CI':>22s} {'p_adj':>10s}")
    for c in comparisons:
        star = " *" if c.significant else "  "
        print(
            f"{c.method_a + ' vs ' + c.method_b:28s} {c.median_diff:+10.4f} "
            f"[{c.ci_low:+.4f}, {c.ci_high:+.4f}] {c.p_adjusted:10.4g}{star}"
        )
    print("\n* = significant at alpha=0.05 after Holm-Bonferroni")
    return comparisons


def main() -> None:
    import argparse

    from src.config import default_output_dir

    ap = argparse.ArgumentParser(description="Paired statistical comparison of encoders")
    ap.add_argument("--seg-dir", default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--reference", default="ijepa")
    args = ap.parse_args()

    base = default_output_dir()
    run(
        Path(args.seg_dir) if args.seg_dir else base / "seg",
        Path(args.out) if args.out else base / "results" / "comparisons.json",
        args.reference,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/eval/tables.py
"""Result tables, emitted as both Markdown (to read) and LaTeX booktabs (to paste).

Five tables, matching the thesis structure:

  T1  main results       Dice/IoU/Prec/Rec/HD95/failure-rate, mean +- std over seeds,
                         Holm-adjusted p vs the reference arm
  T2  comparability      the 880/120 split alongside published numbers, clearly marked
                         as cited rather than reproduced
  T3  fairness           every hyperparameter held constant across the four methods
  T4  ablations          decoder swap, low-label regime
  T5  compute            GPU-hours, params, GMACs, FPS per method

Everything reads from `outputs/seg/*/summary.json` and `outputs/ckpt/metrics.jsonl`, so
tables regenerate on a laptop with no GPU and no checkpoints.
"""
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import numpy as np

DISPLAY = {
    "ijepa": "I-JEPA (ours)",
    "mae": "MAE",
    "simclr": "SimCLR",
    "mocov3": "MoCo v3",
    "random": "Random init",
    "imagenet": "ImageNet sup. (ref.)",
}
ORDER = ["ijepa", "mae", "simclr", "mocov3", "random", "imagenet"]


def _fmt(v: float, nd: int = 4) -> str:
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{nd}f}"


def markdown_table(headers: list[str], rows: list[list[str]], align: str = "l") -> str:
    sep = ["---"] + ["---:"] * (len(headers) - 1) if align == "l" else ["---"] * len(headers)
    out = ["| " + " | ".join(headers) + " |", "| " + " | ".join(sep) + " |"]
    out += ["| " + " | ".join(r) + " |" for r in rows]
    return "\n".join(out)


def latex_table(headers: list[str], rows: list[list[str]], caption: str, label: str) -> str:
    spec = "l" + "r" * (len(headers) - 1)
    esc = lambda s: s.replace("_", r"\_").replace("%", r"\%")  # noqa: E731
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{esc(caption)}}}",
        rf"\label{{{label}}}",
        rf"\begin{{tabular}}{{{spec}}}",
        r"\toprule",
        " & ".join(esc(h) for h in headers) + r" \\",
        r"\midrule",
    ]
    lines += [" & ".join(r) + r" \\" for r in rows]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


def load_summaries(seg_dir: Path) -> list[dict]:
    out = []
    for p in sorted(Path(seg_dir).glob("*/summary.json")):
        try:
            out.append(json.loads(p.read_text()))
        except json.JSONDecodeError:
            print(f"[tables] skipping unreadable {p}")
    return out


def group_by_encoder(summaries: list[dict], **filters) -> dict[str, list[dict]]:
    groups: dict[str, list[dict]] = defaultdict(list)
    for s in summaries:
        if all(s.get(k) == v for k, v in filters.items()):
            groups[s["encoder"]].append(s)
    return groups


def _mean_std(vals: list[float]) -> tuple[float, float]:
    a = np.array(vals, dtype=float)
    return float(a.mean()), float(a.std(ddof=1)) if len(a) > 1 else 0.0


def table_main(
    summaries: list[dict],
    comparisons: list[dict] | None = None,
    reference: str = "ijepa",
) -> tuple[list[str], list[list[str]]]:
    """T1: the headline table."""
    groups = group_by_encoder(summaries, label_fraction=1.0, decoder="segformer", split="800_100_100")
    p_by_method = {c["method_b"]: c for c in (comparisons or [])}

    metrics = ["dice", "iou", "precision", "recall"]
    best = {m: max((np.mean([s[m] for s in g]) for g in groups.values()), default=0.0) for m in metrics}

    headers = ["Encoder", "Dice", "mIoU", "Prec.", "Rec.", "HD95 (px)", "Fail. rate", "p (Holm)"]
    rows = []
    for enc in ORDER:
        if enc not in groups:
            continue
        g = groups[enc]
        cells = [DISPLAY.get(enc, enc)]
        for m in metrics:
            mu, sd = _mean_std([s[m] for s in g])
            cell = f"{mu:.4f}" + (f" ± {sd:.4f}" if sd > 0 else "")
            if abs(mu - best[m]) < 1e-9:
                cell = f"**{cell}**"
            cells.append(cell)
        hd_mu, hd_sd = _mean_std([s["hd95"] for s in g])
        cells.append(f"{hd_mu:.1f}" + (f" ± {hd_sd:.1f}" if hd_sd > 0 else ""))
        fr_mu, _ = _mean_std([s["failure_rate"] for s in g])
        cells.append(f"{fr_mu:.1%}")

        if enc == reference:
            cells.append("--")
        elif enc in p_by_method:
            c = p_by_method[enc]
            cells.append(f"{c['p_adjusted']:.3g}" + ("†" if c["significant"] else ""))
        else:
            cells.append("--")
        rows.append(cells)
    return headers, rows


def table_fairness(cfgs: dict[str, dict]) -> tuple[list[str], list[list[str]]]:
    """T3: what was held constant. The table that makes the comparison defensible."""
    headers = ["Held constant", "Value"]
    if not cfgs:
        return headers, []
    any_cfg = next(iter(cfgs.values()))
    m, o = any_cfg.get("model", {}), any_cfg.get("optim", {})
    corpus_note = "HyperKvasir unlabeled, deduplicated against Kvasir-SEG"
    rows = [
        ["Backbone", f"{m.get('arch', 'vit_small')}/{m.get('patch_size', 16)}"],
        ["Input resolution", f"{m.get('img_size', 224)} x {m.get('img_size', 224)}"],
        ["Stochastic depth", str(m.get("drop_path_rate", 0.0))],
        ["Pretraining corpus", corpus_note],
        ["Epochs", str(o.get("epochs", 100))],
        ["Global batch", str(o.get("global_batch", 512))],
        ["Samples seen", f"{o.get('epochs', 100)} x corpus size (identical for all methods)"],
        ["Optimizer", "AdamW, betas (0.9, 0.95), warmup + cosine"],
        ["Warmup epochs", str(o.get("warmup_epochs", 10))],
        ["Gradient clip", str(o.get("grad_clip", 3.0))],
        ["Precision", "fp16 AMP + GradScaler (losses in fp32)"],
        ["Decoder", "SegFormer all-MLP on ViT simple feature pyramid"],
        ["Fine-tune recipe", "352 px, AdamW, layer decay 0.75, Dice+BCE, 100 ep"],
        ["Splits", "800/100/100, group-aware, stratified, committed to repo"],
    ]
    # Peak LR is intentionally NOT constant — see below.
    rows.append(
        [
            "Peak LR (NOT held constant)",
            ", ".join(
                f"{k}: {v.get('optim', {}).get('ref_lr', '?')}" for k, v in sorted(cfgs.items())
            )
            + " — each method's published value under its own batch-scaling rule",
        ]
    )
    return headers, rows


def table_compute(pretrain_metrics: dict[str, list[dict]], summaries: list[dict]) -> tuple[list[str], list[list[str]]]:
    """T5: what it cost."""
    headers = ["Method", "Epochs", "min/epoch", "GPU-h", "Encoder params", "Decoder params"]
    enc_p = {s["encoder"]: s.get("encoder_params") for s in summaries if s.get("encoder_params")}
    dec_p = {s["encoder"]: s.get("decoder_params") for s in summaries if s.get("decoder_params")}

    rows = []
    for method in ORDER:
        recs = pretrain_metrics.get(method)
        if not recs:
            continue
        epochs = max(r["epoch"] for r in recs)
        times = [r["epoch_time_s"] for r in recs if r.get("epoch_time_s")]
        mins = np.mean(times) / 60 if times else float("nan")
        gpu_h = sum(times) / 3600 if times else float("nan")
        rows.append(
            [
                DISPLAY.get(method, method),
                str(epochs),
                _fmt(mins, 1),
                _fmt(gpu_h, 1),
                f"{enc_p.get(method, 0) / 1e6:.2f}M" if enc_p.get(method) else "--",
                f"{dec_p.get(method, 0) / 1e6:.2f}M" if dec_p.get(method) else "--",
            ]
        )
    return headers, rows


def table_low_label(summaries: list[dict]) -> tuple[list[str], list[list[str]]]:
    """T4a: Dice vs fraction of training labels — usually the strongest result."""
    fractions = sorted({s.get("label_fraction", 1.0) for s in summaries})
    headers = ["Encoder"] + [f"{f:.0%} labels" for f in fractions]
    rows = []
    for enc in ORDER:
        cells = [DISPLAY.get(enc, enc)]
        found = False
        for f in fractions:
            vals = [
                s["dice"]
                for s in summaries
                if s["encoder"] == enc and s.get("label_fraction", 1.0) == f
            ]
            if vals:
                found = True
                mu, sd = _mean_std(vals)
                cells.append(f"{mu:.4f}" + (f" ± {sd:.4f}" if sd > 0 else ""))
            else:
                cells.append("--")
        if found:
            rows.append(cells)
    return headers, rows


def table_decoder_ablation(summaries: list[dict]) -> tuple[list[str], list[list[str]]]:
    """T4b: does the encoder ranking survive a decoder swap?"""
    decoders = sorted({s.get("decoder", "segformer") for s in summaries})
    headers = ["Encoder"] + [d for d in decoders]
    rows = []
    for enc in ORDER:
        cells = [DISPLAY.get(enc, enc)]
        found = False
        for d in decoders:
            vals = [
                s["dice"]
                for s in summaries
                if s["encoder"] == enc
                and s.get("decoder") == d
                and s.get("label_fraction", 1.0) == 1.0
            ]
            if vals:
                found = True
                mu, sd = _mean_std(vals)
                cells.append(f"{mu:.4f}" + (f" ± {sd:.4f}" if sd > 0 else ""))
            else:
                cells.append("--")
        if found:
            rows.append(cells)
    return headers, rows


def build_all(out_dir: Path, seg_dir: Path, ckpt_dir: Path, results_dir: Path) -> None:
    summaries = load_summaries(seg_dir)
    if not summaries:
        print(f"[tables] no summaries under {seg_dir}; run segmentation first")

    comparisons = []
    comp_path = results_dir / "comparisons.json"
    if comp_path.is_file():
        comparisons = json.loads(comp_path.read_text())

    pretrain_metrics: dict[str, list[dict]] = defaultdict(list)
    metrics_file = ckpt_dir / "metrics.jsonl"
    if metrics_file.is_file():
        for line in metrics_file.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                pretrain_metrics[r["method"]].append(r)

    cfgs: dict[str, dict] = {}
    from src.config import REPO_ROOT

    for method in ["ijepa", "mae", "simclr", "mocov3"]:
        p = REPO_ROOT / "configs" / f"pretrain_{method}.yaml"
        if p.is_file():
            import yaml

            cfgs[method] = yaml.safe_load(p.read_text())

    specs = [
        ("t1_main", "Polyp segmentation on Kvasir-SEG (800/100/100 split, mean +- std over 5 seeds). "
                    "Bold = best; dagger = significant vs I-JEPA after Holm-Bonferroni.",
         *table_main(summaries, comparisons)),
        ("t3_fairness", "Experimental variables held constant across all four SSL methods.",
         *table_fairness(cfgs)),
        ("t4a_low_label", "Dice under reduced label budgets.", *table_low_label(summaries)),
        ("t4b_decoder", "Decoder ablation: SegFormer head vs ViT-UNet.",
         *table_decoder_ablation(summaries)),
        ("t5_compute", "Compute cost per pretraining method.",
         *table_compute(pretrain_metrics, summaries)),
    ]

    out_dir.mkdir(parents=True, exist_ok=True)
    md_parts = []
    for name, caption, headers, rows in specs:
        if not rows:
            continue
        md_parts.append(f"### {caption}\n\n{markdown_table(headers, rows)}\n")
        latex = latex_table(
            headers, [[c.replace("**", "").replace("±", r"$\pm$").replace("†", r"$^\dagger$") for c in r] for r in rows],
            caption, f"tab:{name}",
        )
        (out_dir / f"{name}.tex").write_text(latex + "\n")
        print(f"[tables] wrote {out_dir / f'{name}.tex'}")

    (out_dir / "all_tables.md").write_text("\n".join(md_parts) + "\n")
    print(f"[tables] wrote {out_dir / 'all_tables.md'}")
    if md_parts:
        print("\n" + "\n".join(md_parts))


def main() -> None:
    import argparse

    from src.config import default_output_dir

    ap = argparse.ArgumentParser(description="Generate result tables")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    base = default_output_dir()
    out = Path(args.out) if args.out else base / "results" / "tables"
    build_all(out, base / "seg", base / "ckpt", base / "results")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/eval/probe.py
"""Frozen-feature evaluation: k-NN and linear probing on HyperKvasir-labelled.

The cheapest signal about representation quality that exists — no decoder, no
fine-tuning, ~20 minutes for all four encoders. Run it as soon as pretraining finishes
and *before* committing GPU hours to segmentation: if an encoder's k-NN accuracy is at
chance, something is wrong with the pretraining run and no amount of fine-tuning will
hide it.

Note this uses the labelled HyperKvasir split (10,662 images, 23 classes), which is never
trained on anywhere in this project. It is a diagnostic, not a headline result.

    python -m src.eval.probe --encoders ijepa mae simclr mocov3 random
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from src.config import amp_config, default_output_dir, resolve_dataset_dir
from src.data.hyperkvasir import HyperKvasirLabeled
from src.data.transforms import make_pretrain_transform
from src.model.vit import build_vit


@torch.no_grad()
def extract_features(
    encoder: torch.nn.Module, loader: DataLoader, device: str, amp
) -> tuple[np.ndarray, np.ndarray]:
    """Mean-pooled patch tokens for every image. No augmentation, model in eval mode."""
    encoder.eval()
    feats, labels = [], []
    for i, (imgs, y) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        with torch.autocast(
            device_type="cuda" if str(device).startswith("cuda") else "cpu",
            dtype=amp.dtype or torch.float32,
            enabled=amp.enabled,
        ):
            f = encoder(imgs).mean(dim=1)
        feats.append(f.float().cpu())
        labels.append(y)
        if i % 20 == 0:
            print(f"[probe]   batch {i}/{len(loader)}")
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()


def knn_accuracy(
    train_x: np.ndarray,
    train_y: np.ndarray,
    test_x: np.ndarray,
    test_y: np.ndarray,
    k: int = 20,
    temperature: float = 0.07,
) -> float:
    """Weighted k-NN on L2-normalised features, as used by DINO/MoCo evaluations."""
    tr = F.normalize(torch.from_numpy(train_x), dim=1)
    te = F.normalize(torch.from_numpy(test_x), dim=1)
    tr_y = torch.from_numpy(train_y)
    n_classes = int(tr_y.max()) + 1

    correct = 0
    for start in range(0, len(te), 256):
        chunk = te[start : start + 256]
        sim = chunk @ tr.T
        top_sim, top_idx = sim.topk(min(k, tr.shape[0]), dim=1)
        weights = (top_sim / temperature).exp()
        votes = torch.zeros(chunk.shape[0], n_classes)
        votes.scatter_add_(1, tr_y[top_idx], weights)
        correct += int((votes.argmax(dim=1).numpy() == test_y[start : start + 256]).sum())
    return correct / len(te)


def linear_probe(
    train_x: np.ndarray,
    train_y: np.ndarray,
    test_x: np.ndarray,
    test_y: np.ndarray,
    epochs: int = 100,
    lr: float = 1e-3,
    device: str = "cpu",
) -> float:
    """Logistic regression on frozen features, trained with AdamW."""
    mu, sd = train_x.mean(0, keepdims=True), train_x.std(0, keepdims=True) + 1e-6
    tx = torch.from_numpy((train_x - mu) / sd).float().to(device)
    ty = torch.from_numpy(train_y).long().to(device)
    vx = torch.from_numpy((test_x - mu) / sd).float().to(device)

    clf = torch.nn.Linear(tx.shape[1], int(train_y.max()) + 1).to(device)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for _ in range(epochs):
        opt.zero_grad(set_to_none=True)
        F.cross_entropy(clf(tx), ty).backward()
        opt.step()
        sched.step()
    with torch.no_grad():
        pred = clf(vx).argmax(dim=1).cpu().numpy()
    return float((pred == test_y).mean())


def load_encoder(weights_dir: Path, encoder: str, arch: str, img_size: int, patch_size: int):
    """Build a ViT and load exported SSL weights. `random` returns an untrained model."""
    model = build_vit(arch, img_size=img_size, patch_size=patch_size)
    if encoder == "random":
        return model, None
    matches = sorted(weights_dir.glob(f"{encoder}_{arch}_*.pt"))
    if not matches:
        raise FileNotFoundError(f"no exported encoder for {encoder!r} under {weights_dir}")
    ckpt = torch.load(matches[-1], map_location="cpu", weights_only=False)
    state = {k: v for k, v in ckpt["encoder"].items() if not k.endswith("pos_embed")}
    missing, unexpected = model.load_state_dict(state, strict=False)
    real_missing = [k for k in missing if not k.endswith("pos_embed")]
    if real_missing or unexpected:
        raise RuntimeError(f"{encoder}: mismatched weights (missing={real_missing[:3]})")
    return model, matches[-1]


def main() -> None:
    ap = argparse.ArgumentParser(description="k-NN and linear probe on frozen features")
    ap.add_argument("--encoders", nargs="+", default=["ijepa", "mae", "simclr", "mocov3", "random"])
    ap.add_argument("--arch", default="vit_small")
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--patch-size", type=int, default=16)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--num-workers", type=int, default=3)
    ap.add_argument("--k", type=int, default=20)
    ap.add_argument("--train-frac", type=float, default=0.8)
    ap.add_argument("--data-root", default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--save-embeddings", action="store_true", help="also write a UMAP projection")
    args = ap.parse_args()

    amp = amp_config()
    device = "cuda:0" if amp.device == "cuda" else amp.device
    base = default_output_dir()
    out_path = Path(args.out) if args.out else base / "results" / "probe.json"

    root = Path(args.data_root) if args.data_root else resolve_dataset_dir("hyperkvasir_labeled")
    tf = make_pretrain_transform(args.img_size, crop_scale=(1.0, 1.0), horizontal_flip=False)
    ds = HyperKvasirLabeled(root, transform=tf)
    print(f"[probe] {len(ds)} labelled images, {len(ds.classes)} classes")

    # Fixed split, so every encoder is scored on exactly the same images.
    rng = np.random.RandomState(0)
    perm = rng.permutation(len(ds))
    n_train = int(len(ds) * args.train_frac)
    idx_train, idx_test = perm[:n_train], perm[n_train:]

    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)

    results: dict[str, dict] = {}
    if out_path.is_file():
        results = json.loads(out_path.read_text())
    embeddings: dict[str, np.ndarray] = {}

    for enc in args.encoders:
        print(f"\n[probe] === {enc} ===")
        model, src = load_encoder(base / "weights", enc, args.arch, args.img_size, args.patch_size)
        model = model.to(device)
        feats, labels = extract_features(model, loader, device, amp)

        knn = knn_accuracy(feats[idx_train], labels[idx_train], feats[idx_test], labels[idx_test], args.k)
        lin = linear_probe(feats[idx_train], labels[idx_train], feats[idx_test], labels[idx_test], device=device)
        results[enc] = {
            "knn_top1": knn,
            "linear_top1": lin,
            "k": args.k,
            "n_train": len(idx_train),
            "n_test": len(idx_test),
            "n_classes": len(ds.classes),
            "weights": str(src) if src else None,
        }
        print(f"[probe] {enc}: k-NN top-1 {knn:.2%}   linear top-1 {lin:.2%}")

        if args.save_embeddings:
            embeddings[enc] = feats

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(results, indent=2) + "\n")
    print(f"\n[probe] wrote {out_path}")

    if args.save_embeddings and embeddings:
        proj: dict[str, np.ndarray] = {"labels": labels}
        for enc, f in embeddings.items():
            try:
                import umap

                xy = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0).fit_transform(f)
            except ImportError:
                from sklearn.decomposition import PCA

                print("[probe] umap-learn not installed; falling back to PCA for the projection")
                xy = PCA(n_components=2, random_state=0).fit_transform(f)
            proj[f"{enc}_xy"] = np.asarray(xy)
        emb_path = out_path.parent / "embeddings.npz"
        np.savez_compressed(emb_path, **proj)
        print(f"[probe] wrote {emb_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/viz/__init__.py
from src.viz.style import COLORS, ENCODER_COLOR, ENCODER_LABEL, save, setup

__all__ = ["COLORS", "ENCODER_COLOR", "ENCODER_LABEL", "save", "setup"]


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/viz/style.py
"""Shared figure style for the thesis.

Every figure is vector PDF (for LaTeX) plus a PNG (to eyeball quickly), and every one
regenerates from JSON/JSONL on disk with no GPU and no checkpoints — so figures can be
iterated on a laptop while Kaggle is busy training.

**Colour policy.** The categorical palette is a validated 6-slot set: it clears the
lightness band, chroma floor, adjacent-pair CVD separation (worst ΔE 9.1, protan) and the
normal-vision floor (worst ΔE 19.6) on a light surface. Three slots sit below 3:1
contrast against white, which triggers the relief rule — so every figure here carries
either direct labels or a legend plus a companion table in `src/eval/tables.py`, and
identity is never carried by colour alone.

The same palette does **not** clear the all-pairs test (needed when marks are scattered
rather than ordered). The one scatter figure therefore direct-labels every point and
varies marker shape, so colour is reinforcement rather than the identity channel.

Colour is assigned to the *encoder*, fixed, and never re-assigned when a figure plots a
subset — a filter that repaints the survivors makes two figures in the same chapter
disagree about what blue means.
"""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Validated categorical palette (light surface). Order is fixed; never cycle it.
COLORS = {
    "blue": "#2a78d6",
    "orange": "#eb6834",
    "aqua": "#1baf7a",
    "yellow": "#eda100",
    "magenta": "#e87ba4",
    "green": "#008300",
    "violet": "#4a3aa7",
    "red": "#e34948",
}
SERIES = [COLORS["blue"], COLORS["orange"], COLORS["aqua"], COLORS["yellow"], COLORS["magenta"], COLORS["green"]]

INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#8a8880"
GRID = "#e3e2dd"
SURFACE = "#ffffff"

# Fixed identity per encoder. Slot order follows the order arms appear in the thesis.
ENCODER_COLOR = {
    "ijepa": COLORS["blue"],
    "mae": COLORS["orange"],
    "simclr": COLORS["aqua"],
    "mocov3": COLORS["yellow"],
    "random": COLORS["magenta"],
    "imagenet": COLORS["green"],
}
ENCODER_LABEL = {
    "ijepa": "I-JEPA (ours)",
    "mae": "MAE",
    "simclr": "SimCLR",
    "mocov3": "MoCo v3",
    "random": "Random init",
    "imagenet": "ImageNet sup.",
}
ENCODER_MARKER = {
    "ijepa": "o",
    "mae": "s",
    "simclr": "^",
    "mocov3": "D",
    "random": "v",
    "imagenet": "P",
}
ENCODER_ORDER = ["ijepa", "mae", "simclr", "mocov3", "random", "imagenet"]


def setup(base_size: int = 9) -> None:
    """Apply the thesis figure style. Recessive grid and axes, thin marks, no chartjunk."""
    plt.rcParams.update(
        {
            "figure.facecolor": SURFACE,
            "axes.facecolor": SURFACE,
            "savefig.facecolor": SURFACE,
            "font.family": "sans-serif",
            "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
            "font.size": base_size,
            "axes.titlesize": base_size + 1,
            "axes.labelsize": base_size,
            "xtick.labelsize": base_size - 1,
            "ytick.labelsize": base_size - 1,
            "legend.fontsize": base_size - 1,
            "axes.edgecolor": INK_SECONDARY,
            "axes.labelcolor": INK,
            "text.color": INK,
            "xtick.color": INK_SECONDARY,
            "ytick.color": INK_SECONDARY,
            "axes.linewidth": 0.8,
            "axes.grid": True,
            "axes.axisbelow": True,
            "grid.color": GRID,
            "grid.linewidth": 0.7,
            "lines.linewidth": 2.0,
            "lines.markersize": 5,
            "legend.frameon": False,
            "figure.constrained_layout.use": True,
            # Keep text as text in the PDF so LaTeX search/copy works.
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def despine(ax, left: bool = False, bottom: bool = False) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if left:
        ax.spines["left"].set_visible(False)
    if bottom:
        ax.spines["bottom"].set_visible(False)


def save(fig, out_dir: Path, name: str, formats: tuple[str, ...] = ("pdf", "png")) -> list[Path]:
    """Write a figure in every requested format and report the paths."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for ext in formats:
        p = out_dir / f"{name}.{ext}"
        fig.savefig(p, dpi=200 if ext == "png" else None, bbox_inches="tight")
        written.append(p)
    plt.close(fig)
    print(f"[viz] {name}: " + ", ".join(str(p) for p in written))
    return written


def encoder_style(encoder: str) -> dict:
    """Colour + marker for an encoder, stable across every figure."""
    return {
        "color": ENCODER_COLOR.get(encoder, INK_SECONDARY),
        "marker": ENCODER_MARKER.get(encoder, "o"),
        "label": ENCODER_LABEL.get(encoder, encoder),
    }


def present_encoders(keys) -> list[str]:
    """Filter to known encoders in the canonical order, so colours never shift."""
    keys = set(keys)
    return [e for e in ENCODER_ORDER if e in keys] + sorted(keys - set(ENCODER_ORDER))


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/viz/figures.py
"""All thesis figures.

Every function takes a data source and an output directory, and each is independently
runnable — `python -m src.viz.make_all --only masking`. Nothing here needs a GPU or a
checkpoint: figures read `metrics.jsonl`, `summary.json`, `test_per_image.json` and
`comparisons.json`, so they can be iterated locally while Kaggle trains.

Chart-form choices, briefly:
  * change-over-time (loss, Dice vs epoch, Dice vs label fraction) -> line
  * identity comparison across a few arms (k-NN accuracy) -> horizontal bar, sorted
  * distribution of a paired measure (per-image Dice) -> violin + box, not a bar of means
  * effect size with uncertainty -> forest plot with CIs, which is the honest form for
    "how much better, and how sure are we" — a bar chart of means hides both
"""
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import numpy as np

from src.viz.style import (
    GRID,
    INK,
    INK_MUTED,
    INK_SECONDARY,
    SERIES,
    despine,
    encoder_style,
    present_encoders,
    save,
)


# ------------------------------------------------------------------ loading


def load_jsonl(path: Path) -> list[dict]:
    if not Path(path).is_file():
        return []
    return [json.loads(ln) for ln in Path(path).read_text().splitlines() if ln.strip()]


def load_seg_summaries(seg_dir: Path) -> list[dict]:
    out = []
    for p in sorted(Path(seg_dir).glob("*/summary.json")):
        try:
            out.append(json.loads(p.read_text()))
        except json.JSONDecodeError:
            pass
    return out


def _no_data(name: str, what: str) -> None:
    print(f"[viz] skipping {name}: no {what} yet")


# ------------------------------------------------- 1. masking (method figure)


def fig_masking(out_dir: Path, image_path: Path | None = None, n_examples: int = 3, seed: int = 0):
    """The I-JEPA mask: one large context block and four disjoint target blocks.

    This is the figure that demonstrates the method was implemented as published — a
    single target block whose complement is the context (the common shortcut) looks
    obviously different here.
    """
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt
    import torch

    from src.masks.multiblock import MaskCollator, masks_to_grid

    torch.manual_seed(seed)
    collator = MaskCollator(input_size=224, patch_size=16, num_enc_masks=1, num_pred_masks=4, min_keep=10)

    if image_path is not None and Path(image_path).is_file():
        from PIL import Image

        img = np.asarray(Image.open(image_path).convert("RGB").resize((224, 224))) / 255.0
        base = [torch.zeros(3, 224, 224) for _ in range(n_examples)]
    else:
        img = None
        base = [torch.zeros(3, 224, 224) for _ in range(n_examples)]

    _, enc_masks, pred_masks = collator(base)
    g = collator.height

    fig, axes = plt.subplots(1, n_examples, figsize=(2.3 * n_examples, 2.7))
    axes = np.atleast_1d(axes)
    ctx_color, tgt_color, unused_color = SERIES[0], SERIES[1], "#e8e7e2"

    def rgb(hex_str: str) -> tuple[float, float, float]:
        h = hex_str.lstrip("#")
        return tuple(int(h[i : i + 2], 16) / 255 for i in (0, 2, 4))

    for i, ax in enumerate(axes):
        ctx = masks_to_grid(enc_masks[0][i], g, g).numpy()
        # Union of the four target blocks, flattened to a single boolean mask. Drawing
        # them as four translucent layers would darken where blocks overlap and read as
        # a third category; they are one category.
        tgt = np.zeros((g, g), dtype=bool)
        for j in range(len(pred_masks)):
            tgt |= masks_to_grid(pred_masks[j][i], g, g).numpy()

        canvas = np.zeros((g, g, 3))
        canvas[:] = rgb(unused_color)
        canvas[ctx] = rgb(ctx_color)
        canvas[tgt] = rgb(tgt_color)

        if img is not None:
            ax.imshow(img, extent=(0, g, g, 0))
            ax.imshow(canvas, extent=(0, g, g, 0), alpha=0.72, interpolation="nearest")
        else:
            ax.imshow(canvas, extent=(0, g, g, 0), interpolation="nearest")

        # Patch grid, so the 14x14 tokenisation is legible.
        for k in range(g + 1):
            ax.axhline(k, color="white", lw=0.4)
            ax.axvline(k, color="white", lw=0.4)

        ax.set_xlim(0, g)
        ax.set_ylim(g, 0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for s in ax.spines.values():
            s.set_color(GRID)
        ax.set_title(
            f"context {int(ctx.sum())}  ·  targets {int(tgt.sum())}", fontsize=8, color=INK_SECONDARY
        )

    handles = [
        mpatches.Patch(facecolor=ctx_color, label="context block (encoder sees)"),
        mpatches.Patch(facecolor=tgt_color, label="4 target blocks (predicted)"),
        mpatches.Patch(facecolor=unused_color, label="unused"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.10))
    fig.suptitle("I-JEPA multi-block masking (14×14 patch grid, 224px input)", fontsize=10)
    fig.text(
        0.5, -0.19,
        "Context is a separately sampled 85–100% block with the target union removed — not the "
        "complement of the targets.\nContext counts are equal across panels because masks are "
        "truncated to the batch-wide minimum so they collate without padding.",
        ha="center", fontsize=6.5, color=INK_MUTED,
    )
    return save(fig, out_dir, "fig01_masking")


# ------------------------------------------------------------ 2. schedules


def fig_schedules(out_dir: Path, epochs: int = 100, iters_per_epoch: int = 194):
    """LR, weight decay and EMA momentum against training step.

    Worth a figure because two of the three are counter-intuitive: weight decay
    *increases* over training (0.04 -> 0.4) and momentum ramps to exactly 1.0, freezing
    the target encoder at the end.
    """
    import matplotlib.pyplot as plt

    from src.utils.schedulers import ScheduleSet, cosine_momentum

    s = ScheduleSet(iters_per_epoch, epochs, 10, 2.0e-4, 2.5e-4, 1.0e-6, 0.04, 0.4, (0.996, 1.0))
    steps = np.arange(0, s.total_steps + 1, max(1, s.total_steps // 500))
    ep = steps / iters_per_epoch

    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.3))
    for ax, (vals, title, ylab, color) in zip(
        axes,
        [
            ([s.lr(int(t)) for t in steps], "Learning rate", "LR", SERIES[0]),
            ([s.wd(int(t)) for t in steps], "Weight decay", "WD", SERIES[1]),
            ([s.momentum(int(t)) for t in steps], "EMA momentum", "m", SERIES[2]),
        ],
    ):
        ax.plot(ep, vals, color=color)
        ax.set_title(title)
        ax.set_xlabel("epoch")
        ax.set_ylabel(ylab)
        despine(ax)

    axes[0].axvline(10, color=INK_MUTED, ls=":", lw=1)
    axes[0].annotate(
        "warmup ends", (10, 2.5e-4), xytext=(6, -10), textcoords="offset points",
        fontsize=7, color=INK_SECONDARY,
    )
    axes[2].plot(
        ep,
        [cosine_momentum(int(t), s.total_steps, (0.99, 1.0)) for t in steps],
        color=SERIES[3],
        ls="--",
    )
    # Offset the labels off their own curves so neither sits on a line.
    axes[2].annotate(
        "I-JEPA (linear)", (epochs * 0.42, s.momentum(int(s.total_steps * 0.42))),
        xytext=(0, 7), textcoords="offset points", fontsize=7, color=SERIES[2],
    )
    axes[2].annotate(
        "MoCo v3 (cosine)",
        (epochs * 0.30, cosine_momentum(int(s.total_steps * 0.30), s.total_steps, (0.99, 1.0))),
        xytext=(0, -13), textcoords="offset points", fontsize=7, color=SERIES[3],
    )
    axes[1].annotate(
        "weight decay increases\nover training (0.04 → 0.4)", (55, 0.16),
        fontsize=6.5, color=INK_MUTED,
    )
    return save(fig, out_dir, "fig02_schedules")


# ------------------------------------------------- 3. pretraining loss curves


def fig_pretrain_curves(out_dir: Path, metrics_path: Path):
    """Per-method pretraining loss against samples seen.

    Separate panels, not shared axes: the four objectives produce losses on completely
    different scales (smooth-L1 on normalised features vs MSE on pixels vs InfoNCE), so
    overlaying them on one axis would invite a comparison that is not meaningful. The
    shared x-axis (samples seen) is the controlled budget and *is* comparable.
    """
    import matplotlib.pyplot as plt

    recs = load_jsonl(metrics_path)
    if not recs:
        return _no_data("fig_pretrain_curves", "pretraining metrics")

    by_method: dict[str, list[dict]] = defaultdict(list)
    for r in recs:
        by_method[r["method"]].append(r)
    methods = present_encoders(by_method)

    fig, axes = plt.subplots(1, len(methods) + 1, figsize=(2.1 * (len(methods) + 1), 2.4))
    axes = np.atleast_1d(axes)

    for ax, method in zip(axes, methods):
        rs = sorted(by_method[method], key=lambda r: r["epoch"])
        st = encoder_style(method)
        x = np.array([r.get("samples_seen", r["epoch"]) for r in rs]) / 1e6
        ax.plot(x, [r["loss"] for r in rs], color=st["color"])
        ax.set_title(st["label"], fontsize=9)
        ax.set_xlabel("samples seen (M)")
        despine(ax)
    axes[0].set_ylabel("training loss")

    # Wall-clock panel: the cost side of the same runs.
    ax = axes[-1]
    for i, method in enumerate(methods):
        rs = sorted(by_method[method], key=lambda r: r["epoch"])
        st = encoder_style(method)
        cum_h = np.cumsum([r.get("epoch_time_s", 0) for r in rs]) / 3600
        ax.plot([r["epoch"] for r in rs], cum_h, color=st["color"], label=st["label"])
    ax.set_title("Cumulative cost", fontsize=9)
    ax.set_xlabel("epoch")
    ax.set_ylabel("GPU-hours")
    ax.legend(fontsize=6.5, loc="upper left")
    despine(ax)
    return save(fig, out_dir, "fig03_pretrain_curves")


# ----------------------------------------------------- 4. segmentation curves


def fig_seg_curves(out_dir: Path, seg_dir: Path):
    """Validation Dice against epoch, mean ± std over seeds, one line per encoder."""
    import matplotlib.pyplot as plt

    per_enc: dict[str, dict[int, list[float]]] = defaultdict(lambda: defaultdict(list))
    per_enc_loss: dict[str, dict[int, list[float]]] = defaultdict(lambda: defaultdict(list))
    for p in sorted(Path(seg_dir).glob("*/metrics.jsonl")):
        for r in load_jsonl(p):
            if r.get("label_fraction", 1.0) != 1.0:
                continue
            per_enc[r["encoder"]][r["epoch"]].append(r["val_dice"])
            per_enc_loss[r["encoder"]][r["epoch"]].append(r["train_loss"])
    if not per_enc:
        return _no_data("fig_seg_curves", "segmentation metrics")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 2.8))
    for enc in present_encoders(per_enc):
        st = encoder_style(enc)
        for ax, src, ylab in ((ax1, per_enc_loss[enc], "train loss"), (ax2, per_enc[enc], "val Dice")):
            eps = sorted(src)
            mu = np.array([np.mean(src[e]) for e in eps])
            sd = np.array([np.std(src[e], ddof=1) if len(src[e]) > 1 else 0.0 for e in eps])
            ax.plot(eps, mu, color=st["color"], label=st["label"])
            if sd.any():
                ax.fill_between(eps, mu - sd, mu + sd, color=st["color"], alpha=0.15, lw=0)
            ax.set_xlabel("epoch")
            ax.set_ylabel(ylab)
            despine(ax)
    ax2.legend(loc="lower right", fontsize=7)
    fig.suptitle("Segmentation fine-tuning on Kvasir-SEG (mean ± s.d. over seeds)", fontsize=10)
    return save(fig, out_dir, "fig04_seg_curves")


# ------------------------------------------------- 5. per-image Dice spread


def fig_dice_distribution(out_dir: Path, seg_dir: Path):
    """Violin + box of per-image test Dice.

    A bar chart of mean Dice would hide the thing that matters clinically: the lower
    tail, where the model misses a polyp entirely. The violin shows it.
    """
    import matplotlib.pyplot as plt

    from src.eval.stats import aggregate_over_seeds, load_per_image_dice

    by_enc = load_per_image_dice(Path(seg_dir))
    if not by_enc:
        return _no_data("fig_dice_distribution", "per-image test results")
    dice = {e: aggregate_over_seeds(v) for e, v in by_enc.items()}
    encs = present_encoders(dice)

    fig, ax = plt.subplots(figsize=(1.15 * len(encs) + 2.0, 3.2))
    data = [list(dice[e].values()) for e in encs]
    parts = ax.violinplot(data, showextrema=False, widths=0.8)
    for body, enc in zip(parts["bodies"], encs):
        body.set_facecolor(encoder_style(enc)["color"])
        body.set_alpha(0.28)
        body.set_edgecolor("none")
    bp = ax.boxplot(data, widths=0.16, patch_artist=True, showfliers=False, medianprops={"color": INK})
    for patch, enc in zip(bp["boxes"], encs):
        patch.set_facecolor("white")
        patch.set_edgecolor(encoder_style(enc)["color"])
        patch.set_linewidth(1.4)

    ax.axhline(0.5, color=INK_MUTED, ls=":", lw=1)
    ax.annotate("Dice < 0.5: polyp effectively missed", (0.55, 0.505), fontsize=7, color=INK_SECONDARY)
    ax.set_xticks(range(1, len(encs) + 1))
    ax.set_xticklabels([encoder_style(e)["label"] for e in encs], rotation=12, ha="right")
    ax.set_ylabel("per-image Dice (test)")
    ax.set_ylim(0, 1.02)
    despine(ax)
    fig.suptitle("Distribution of per-image Dice on the held-out test set", fontsize=10)
    return save(fig, out_dir, "fig05_dice_distribution")


# ---------------------------------------------------------- 6. forest plot


def fig_paired_forest(out_dir: Path, comparisons_path: Path):
    """Median paired Dice difference vs I-JEPA, with 95% BCa CIs.

    The statistical headline. A CI that crosses zero says "no detectable difference"
    far more honestly than a p-value alone, and the effect size stays visible so a
    significant-but-tiny difference cannot masquerade as a result.
    """
    import matplotlib.pyplot as plt

    if not Path(comparisons_path).is_file():
        return _no_data("fig_paired_forest", "comparisons.json (run src.eval.stats)")
    comps = json.loads(Path(comparisons_path).read_text())
    if not comps:
        return _no_data("fig_paired_forest", "comparisons")

    # Largest difference at the top, the usual forest-plot reading order.
    comps = sorted(comps, key=lambda c: c["median_diff"])
    fig, ax = plt.subplots(figsize=(6.2, 0.55 * len(comps) + 1.6))
    ys = np.arange(len(comps))

    for y, c in zip(ys, comps):
        st = encoder_style(c["method_b"])
        ax.plot([c["ci_low"], c["ci_high"]], [y, y], color=st["color"], lw=2, solid_capstyle="round")
        ax.plot(c["median_diff"], y, marker=st["marker"], color=st["color"], ms=7,
                markeredgecolor="white", markeredgewidth=1.2)
        # Annotations live in a fixed right-hand column (axes fraction on x, data on y)
        # rather than trailing each interval — otherwise a negative CI pushes its label
        # across the zero line and the two collide.
        ax.annotate(
            f"{c['median_diff']:+.3f}   p={c['p_adjusted']:.2g}{'*' if c['significant'] else ''}",
            xy=(1.02, y), xycoords=("axes fraction", "data"),
            va="center", ha="left", fontsize=7.5, color=INK_SECONDARY, annotation_clip=False,
        )

    ax.axvline(0, color=INK_MUTED, lw=1)
    ax.set_yticks(ys)
    ax.set_yticklabels([f"vs {encoder_style(c['method_b'])['label']}" for c in comps])
    ax.set_xlabel("median paired Dice difference (I-JEPA − baseline)")
    ax.set_ylim(-0.6, len(comps) - 0.4)
    ax.margins(x=0.10)
    ax.grid(axis="y", visible=False)
    despine(ax, left=True)
    ax.annotate("← baseline better", xy=(0, 1.0), xycoords=("data", "axes fraction"),
                xytext=(-6, 6), textcoords="offset points", ha="right", fontsize=7, color=INK_MUTED)
    ax.annotate("I-JEPA better →", xy=(0, 1.0), xycoords=("data", "axes fraction"),
                xytext=(6, 6), textcoords="offset points", ha="left", fontsize=7, color=INK_MUTED)
    fig.suptitle("Paired per-image comparison with 95% BCa bootstrap CIs", fontsize=10)
    fig.text(0.01, -0.02, "* significant at α=0.05 after Holm–Bonferroni; n = test images",
             fontsize=7, color=INK_MUTED)
    return save(fig, out_dir, "fig06_paired_forest")


# --------------------------------------------------------- 7. low-label regime


def fig_low_label(out_dir: Path, seg_dir: Path):
    """Dice against the fraction of training labels used.

    Usually the strongest result in a study like this: pretraining differences are
    largest exactly where labels are scarce, which is also the regime that matters for
    a clinical dataset.
    """
    import matplotlib.pyplot as plt

    summaries = load_seg_summaries(Path(seg_dir))
    by: dict[str, dict[float, list[float]]] = defaultdict(lambda: defaultdict(list))
    for s in summaries:
        if s.get("decoder", "segformer") != "segformer":
            continue
        by[s["encoder"]][s.get("label_fraction", 1.0)].append(s["dice"])
    fractions = sorted({f for v in by.values() for f in v})
    if len(fractions) < 2:
        return _no_data("fig_low_label", "runs at more than one label fraction")

    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    for enc in present_encoders(by):
        st = encoder_style(enc)
        fr = [f for f in fractions if f in by[enc]]
        mu = np.array([np.mean(by[enc][f]) for f in fr])
        sd = np.array([np.std(by[enc][f], ddof=1) if len(by[enc][f]) > 1 else 0.0 for f in fr])
        ax.plot([f * 100 for f in fr], mu, color=st["color"], marker=st["marker"], label=st["label"])
        if sd.any():
            ax.fill_between([f * 100 for f in fr], mu - sd, mu + sd, color=st["color"], alpha=0.15, lw=0)
    ax.set_xscale("log")
    ax.set_xticks([f * 100 for f in fractions])
    ax.set_xticklabels([f"{f:.0%}" for f in fractions])
    ax.set_xlabel("fraction of training labels")
    ax.set_ylabel("test Dice")
    ax.legend(fontsize=7, loc="lower right")
    despine(ax)
    fig.suptitle("Label efficiency of the pretrained encoders", fontsize=10)
    return save(fig, out_dir, "fig07_low_label")


# ---------------------------------------------------------- 8. efficiency


def fig_efficiency(out_dir: Path, seg_dir: Path):
    """Dice against pretraining cost.

    Every point is direct-labelled and carries its own marker shape, because a scatter
    puts all pairs of colours in play at once and this palette is only validated for
    adjacent pairs. Colour here is reinforcement, not the identity channel.
    """
    import matplotlib.pyplot as plt

    summaries = [
        s for s in load_seg_summaries(Path(seg_dir))
        if s.get("label_fraction", 1.0) == 1.0 and s.get("decoder", "segformer") == "segformer"
    ]
    if not summaries:
        return _no_data("fig_efficiency", "segmentation summaries")

    by_enc: dict[str, list[dict]] = defaultdict(list)
    for s in summaries:
        by_enc[s["encoder"]].append(s)

    hours = {}
    from src.config import default_output_dir

    for r in load_jsonl(default_output_dir() / "ckpt" / "metrics.jsonl"):
        hours[r["method"]] = hours.get(r["method"], 0.0) + r.get("epoch_time_s", 0) / 3600

    fig, ax = plt.subplots(figsize=(4.8, 3.3))
    for enc in present_encoders(by_enc):
        st = encoder_style(enc)
        dice = float(np.mean([s["dice"] for s in by_enc[enc]]))
        cost = hours.get(enc, 0.0)
        ax.scatter([cost], [dice], s=90, color=st["color"], marker=st["marker"],
                   edgecolor="white", linewidth=1.2, zorder=3)
        ax.annotate(st["label"], (cost, dice), xytext=(8, 4), textcoords="offset points",
                    fontsize=8, color=INK)
    ax.set_xlabel("pretraining cost (GPU-hours)")
    ax.set_ylabel("test Dice")
    ax.margins(0.22)
    despine(ax)
    fig.suptitle("Segmentation quality against pretraining cost", fontsize=10)
    return save(fig, out_dir, "fig08_efficiency")


# ---------------------------------------------------------- 9. qualitative


def fig_qualitative(out_dir: Path, seg_dir: Path, kvasir_root: Path | None = None, n_rows: int = 5):
    """Input | ground truth | one column per encoder.

    Rows span the difficulty range by I-JEPA's own Dice quartiles and deliberately
    include the two worst cases. Showing failures is not a weakness in a thesis — a
    qualitative figure with only successes is the one a committee distrusts.
    """
    import matplotlib.pyplot as plt
    from PIL import Image

    from src.config import resolve_dataset_dir
    from src.data.kvasir_seg import find_image_and_mask_dirs

    runs: dict[str, Path] = {}
    for p in sorted(Path(seg_dir).glob("*/summary.json")):
        s = json.loads(p.read_text())
        if s.get("label_fraction", 1.0) != 1.0 or s.get("decoder", "segformer") != "segformer":
            continue
        if s.get("seed", 0) == 0 and (p.parent / "test_predictions.npz").is_file():
            runs[s["encoder"]] = p.parent / "test_predictions.npz"
    if not runs:
        return _no_data("fig_qualitative", "saved test predictions")

    encs = present_encoders(runs)
    preds = {e: np.load(runs[e], allow_pickle=False) for e in encs}
    ref = encs[0]
    stems = [str(s) for s in preds[ref]["stems"]]
    shapes = preds[ref]["shapes"]

    per_image = json.loads((runs[ref].parent / "test_per_image.json").read_text())
    dice_by_stem = {r["stem"]: r["dice"] for r in per_image}
    ranked = sorted(stems, key=lambda s: -dice_by_stem.get(s, 0))
    picks = [ranked[0], ranked[len(ranked) // 4], ranked[len(ranked) // 2]][: max(1, n_rows - 2)]
    picks += ranked[-2:]  # the two worst — failure cases

    root = Path(kvasir_root) if kvasir_root else resolve_dataset_dir("kvasir_seg")
    img_dir, mask_dir = find_image_and_mask_dirs(root)

    ncols = 2 + len(encs)
    fig, axes = plt.subplots(len(picks), ncols, figsize=(1.55 * ncols, 1.55 * len(picks)))
    axes = np.atleast_2d(axes)

    for r, stem in enumerate(picks):
        idx = stems.index(stem)
        h, w = int(shapes[idx][0]), int(shapes[idx][1])
        img_p = next((img_dir / f"{stem}{e}" for e in (".jpg", ".png") if (img_dir / f"{stem}{e}").is_file()), None)
        msk_p = next((mask_dir / f"{stem}{e}" for e in (".jpg", ".png") if (mask_dir / f"{stem}{e}").is_file()), None)
        img = np.asarray(Image.open(img_p).convert("RGB")) if img_p else np.zeros((h, w, 3), np.uint8)
        gt = (np.asarray(Image.open(msk_p).convert("L")) > 127) if msk_p else np.zeros((h, w), bool)

        axes[r, 0].imshow(img)
        axes[r, 1].imshow(gt, cmap="gray")
        for c, enc in enumerate(encs):
            pred = np.unpackbits(preds[enc][stem])[: h * w].reshape(h, w).astype(bool)
            ax = axes[r, 2 + c]
            ax.imshow(img)
            overlay = np.zeros((h, w, 4))
            rgb = tuple(int(encoder_style(enc)["color"].lstrip("#")[i:i + 2], 16) / 255 for i in (0, 2, 4))
            overlay[pred] = [*rgb, 0.55]
            ax.imshow(overlay)
            d = 2 * (pred & gt).sum() / max(1, pred.sum() + gt.sum())
            ax.set_xlabel(f"{d:.3f}", fontsize=7, color=INK_SECONDARY, labelpad=1)

        for c in range(ncols):
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
            axes[r, c].grid(False)

    titles = ["Input", "Ground truth"] + [encoder_style(e)["label"] for e in encs]
    for c, t in enumerate(titles):
        axes[0, c].set_title(t, fontsize=8)
    fig.suptitle("Qualitative results (Dice below each panel; last two rows are failure cases)", fontsize=10)
    return save(fig, out_dir, "fig09_qualitative")


# ------------------------------------------------------- 10. k-NN / probe


def fig_knn_probe(out_dir: Path, probe_path: Path):
    """k-NN and linear-probe accuracy on HyperKvasir-labelled.

    Representation quality with no fine-tuning at all — the cheapest signal available,
    and it can be read before any segmentation run finishes.
    """
    import matplotlib.pyplot as plt

    if not Path(probe_path).is_file():
        return _no_data("fig_knn_probe", "probe results (run src.eval.probe)")
    res = json.loads(Path(probe_path).read_text())
    encs = present_encoders(res)

    fig, ax = plt.subplots(figsize=(5.0, 0.42 * len(encs) + 1.8))
    ys = np.arange(len(encs))
    h = 0.36
    for i, enc in enumerate(encs):
        st = encoder_style(enc)
        knn = res[enc].get("knn_top1", 0) * 100
        lin = res[enc].get("linear_top1", 0) * 100
        ax.barh(ys[i] + h / 2, knn, height=h, color=st["color"], alpha=0.95)
        ax.barh(ys[i] - h / 2, lin, height=h, color=st["color"], alpha=0.45)
        ax.annotate(f"{knn:.1f}", (knn, ys[i] + h / 2), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=7.5, color=INK_SECONDARY)
        ax.annotate(f"{lin:.1f}", (lin, ys[i] - h / 2), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=7.5, color=INK_SECONDARY)
    ax.set_yticks(ys)
    ax.set_yticklabels([encoder_style(e)["label"] for e in encs])
    ax.set_xlabel("top-1 accuracy (%)")
    ax.grid(axis="y", visible=False)
    ax.annotate("solid = k-NN (k=20)   ·   faded = linear probe", (0.0, len(encs) - 0.35),
                fontsize=7.5, color=INK_MUTED)
    despine(ax, left=True)
    fig.suptitle("Frozen-feature evaluation on HyperKvasir-labelled (23 classes)", fontsize=10)
    return save(fig, out_dir, "fig10_knn_probe")


# ---------------------------------------------------- 11. embedding space


def fig_embedding_space(out_dir: Path, embed_path: Path):
    """2-D projection of frozen features, coloured by HyperKvasir class.

    Class count exceeds the categorical palette by a wide margin, so this uses a
    perceptually uniform continuous map for class index — identity is carried by the
    per-panel legend and the structure, not by memorising 23 hues.
    """
    import matplotlib.pyplot as plt

    if not Path(embed_path).is_file():
        return _no_data("fig_embedding_space", "embeddings (run src.eval.probe --save-embeddings)")
    data = np.load(embed_path, allow_pickle=True)
    methods = [k for k in data.files if k.endswith("_xy")]
    if not methods:
        return _no_data("fig_embedding_space", "projected embeddings")

    labels = data["labels"]
    fig, axes = plt.subplots(1, len(methods), figsize=(2.4 * len(methods), 2.6))
    axes = np.atleast_1d(axes)
    for ax, key in zip(axes, methods):
        xy = data[key]
        ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="viridis", s=3, alpha=0.6, linewidths=0)
        ax.set_title(encoder_style(key[:-3])["label"], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    fig.suptitle("UMAP of frozen encoder features, coloured by class", fontsize=10)
    return save(fig, out_dir, "fig11_embedding_space")


FIGURES = {
    "masking": fig_masking,
    "schedules": fig_schedules,
    "pretrain_curves": fig_pretrain_curves,
    "seg_curves": fig_seg_curves,
    "dice_distribution": fig_dice_distribution,
    "paired_forest": fig_paired_forest,
    "low_label": fig_low_label,
    "efficiency": fig_efficiency,
    "qualitative": fig_qualitative,
    "knn_probe": fig_knn_probe,
    "embedding_space": fig_embedding_space,
}


In [ ]:
%%writefile /kaggle/working/jepa-thesis/src/viz/make_all.py
"""Regenerate every thesis figure.

Runs with no GPU and no checkpoints — everything reads from the JSON/JSONL artefacts
written during training, so figures can be iterated on a laptop while Kaggle trains.
Figures whose inputs do not exist yet are skipped with a message rather than crashing,
so this is safe to run at any point in the project.

    python -m src.viz.make_all                 # everything available
    python -m src.viz.make_all --only masking schedules
    python -m src.viz.make_all --list
"""
from __future__ import annotations

import argparse
import traceback
from pathlib import Path

from src.config import REPO_ROOT, default_output_dir
from src.viz.figures import FIGURES
from src.viz.style import setup


def build(names: list[str], out_dir: Path, base: Path, kvasir_root: Path | None = None) -> None:
    setup()
    seg_dir = base / "seg"
    ckpt_metrics = base / "ckpt" / "metrics.jsonl"
    results = base / "results"

    args_for = {
        "masking": {},
        "schedules": {},
        "pretrain_curves": {"metrics_path": ckpt_metrics},
        "seg_curves": {"seg_dir": seg_dir},
        "dice_distribution": {"seg_dir": seg_dir},
        "paired_forest": {"comparisons_path": results / "comparisons.json"},
        "low_label": {"seg_dir": seg_dir},
        "efficiency": {"seg_dir": seg_dir},
        "qualitative": {"seg_dir": seg_dir, "kvasir_root": kvasir_root},
        "knn_probe": {"probe_path": results / "probe.json"},
        "embedding_space": {"embed_path": results / "embeddings.npz"},
    }

    ok, skipped, failed = 0, 0, 0
    for name in names:
        fn = FIGURES[name]
        try:
            result = fn(out_dir, **args_for.get(name, {}))
            if result is None:
                skipped += 1
            else:
                ok += 1
        except FileNotFoundError as e:
            print(f"[viz] skipping {name}: {e}")
            skipped += 1
        except Exception:  # noqa: BLE001 - one broken figure must not block the rest
            print(f"[viz] FAILED {name}:")
            traceback.print_exc(limit=3)
            failed += 1

    print(f"\n[viz] {ok} written, {skipped} skipped (inputs not ready), {failed} failed -> {out_dir}")


def main() -> None:
    ap = argparse.ArgumentParser(description="Generate thesis figures")
    ap.add_argument("--only", nargs="+", choices=sorted(FIGURES), default=None)
    ap.add_argument("--out", default=str(REPO_ROOT / "figures"))
    ap.add_argument("--base", default=None, help="outputs dir holding ckpt/, seg/, results/")
    ap.add_argument("--kvasir-root", default=None)
    ap.add_argument("--list", action="store_true")
    args = ap.parse_args()

    if args.list:
        for k in sorted(FIGURES):
            print(f"  {k}")
        return

    build(
        args.only or list(FIGURES),
        Path(args.out),
        Path(args.base) if args.base else default_output_dir(),
        Path(args.kvasir_root) if args.kvasir_root else None,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/jepa-thesis/configs/pretrain_ijepa.yaml
# I-JEPA pretraining on HyperKvasir unlabeled (~99,417 GI endoscopy images).
# Mask and optimisation values follow reference/ijepa/configs/in1k_vith14_ep300.yaml,
# rescaled to ViT-S/16 at global batch 512.
method: ijepa

model:
  arch: vit_small          # 384-d, 12 blocks, 6 heads
  patch_size: 16
  img_size: 224            # -> 14x14 = 196 tokens
  drop_path_rate: 0.0      # held at 0 across all four methods

optim:
  global_batch: 512        # pinned; per-GPU batch is derived from world size
  accum_steps: 1
  epochs: 100              # 9,941,700 samples seen — the controlled budget
  warmup_epochs: 10
  # i-jepa uses lr 1e-3 at batch 2048; linear scaling to 512 gives 2.5e-4.
  start_lr: 2.0e-4
  ref_lr: 2.5e-4
  final_lr: 1.0e-6
  weight_decay: 0.04       # cosine-INCREASED to final_weight_decay over training
  final_weight_decay: 0.4
  exclude_bias_and_norm_from_wd: true
  ema: [0.996, 1.0]
  ipe_scale: 1.0
  grad_clip: 3.0

mask:
  enc_mask_scale: [0.85, 1.0]   # one large context block ...
  pred_mask_scale: [0.15, 0.2]  # ... minus four target blocks
  aspect_ratio: [0.75, 1.5]
  num_enc_masks: 1
  num_pred_masks: 4
  min_keep: 10
  allow_overlap: false

jepa:
  # 192 = 0.5 * 384, the same predictor/encoder width ratio the paper uses for ViT-B
  # (384/768). Copying the absolute 384 would put a ViT-B-sized head on a ViT-S body.
  pred_emb_dim: 192
  pred_depth: 6

aug:
  # The released i-jepa configs set use_horizontal_flip, use_color_distortion and
  # use_gaussian_blur all to false — RandomResizedCrop really is the only augmentation.
  crop_scale: [0.3, 1.0]
  horizontal_flip: false
  color_distortion: false
  gaussian_blur: false
  two_views: false

runtime:
  seed: 0
  num_workers: 3
  session_guard_hours: 7.5
  ckpt_push_minutes: 45
  ckpt_dataset_slug: ""    # e.g. morsalin101/jepa-thesis-ckpt


In [ ]:
%%writefile /kaggle/working/jepa-thesis/configs/pretrain_mae.yaml
# MAE pretraining. The pixel-space arm of the comparison: everything except the
# objective is identical to configs/pretrain_ijepa.yaml.
#
# Caveat to state in the thesis: MAE is known to keep improving out to 800-1600 epochs
# while contrastive methods saturate earlier, so a fixed 100-epoch budget structurally
# disadvantages it. Disclose, don't hide.
method: mae

model:
  arch: vit_small
  patch_size: 16
  img_size: 224
  drop_path_rate: 0.0

optim:
  global_batch: 512
  accum_steps: 1
  epochs: 100
  warmup_epochs: 10
  # MAE's rule is lr = 1.5e-4 * batch/256 -> 3.0e-4 at batch 512.
  start_lr: 1.0e-5
  ref_lr: 3.0e-4
  final_lr: 1.0e-6
  weight_decay: 0.05       # constant for MAE (final == ref)
  final_weight_decay: 0.05
  exclude_bias_and_norm_from_wd: true
  ema: [0.996, 1.0]        # unused: MAE has no target encoder
  ipe_scale: 1.0
  grad_clip: 3.0

mae:
  mask_ratio: 0.75
  decoder_embed_dim: 256   # 0.67 * 384, preserving MAE's own 512/768 ratio for ViT-B
  decoder_depth: 8
  decoder_num_heads: 8
  norm_pix_loss: true      # per-patch target normalisation; clearly better in the paper

aug:
  crop_scale: [0.2, 1.0]   # MAE's published crop range
  horizontal_flip: true    # MAE does use hflip, unlike i-jepa
  color_distortion: false
  gaussian_blur: false
  two_views: false

runtime:
  seed: 0
  num_workers: 3
  session_guard_hours: 7.5
  ckpt_push_minutes: 45
  ckpt_dataset_slug: ""


In [ ]:
%%writefile /kaggle/working/jepa-thesis/configs/pretrain_simclr.yaml
# SimCLR pretraining. Contrastive arm with no momentum encoder.
#
# Caveat to state in the thesis: SimCLR benefits substantially from batches far larger
# than 512. Pinning global batch across all four methods is the right fairness call
# (equal optimisation budget) but it does disadvantage SimCLR. Disclose it.
#
# Note the augmentation block is NOT the same as I-JEPA's. For SimCLR the augmentations
# *are* the method — equalising them would misrepresent it, not make it fairer.
method: simclr

model:
  arch: vit_small
  patch_size: 16
  img_size: 224
  drop_path_rate: 0.0

optim:
  global_batch: 512
  accum_steps: 1           # must stay 1: InfoNCE is batch-coupled, accumulation
                           # shrinks the negative set instead of emulating a big batch
  epochs: 100
  warmup_epochs: 10
  start_lr: 1.0e-5
  ref_lr: 3.0e-4           # 1.5e-4 * batch/256, the AdamW-for-ViT rule from MoCo v3
  final_lr: 1.0e-6
  weight_decay: 0.1
  final_weight_decay: 0.1
  exclude_bias_and_norm_from_wd: true
  ipe_scale: 1.0
  grad_clip: 3.0

contrastive:
  proj_hidden_dim: 2048
  proj_out_dim: 256
  proj_num_layers: 3
  temperature: 0.1         # SimCLR's tau; MoCo v3 uses 0.2

aug:
  crop_scale: [0.2, 1.0]
  horizontal_flip: true
  color_jitter_strength: 0.5
  color_distortion: true
  gaussian_blur: true
  two_views: true

runtime:
  seed: 0
  num_workers: 3
  session_guard_hours: 7.5
  ckpt_push_minutes: 45
  ckpt_dataset_slug: ""


In [ ]:
%%writefile /kaggle/working/jepa-thesis/configs/pretrain_mocov3.yaml
# MoCo v3 pretraining. The closest "fair" contrastive comparison for I-JEPA, since it
# also carries an EMA target encoder — so a difference between the two is attributable
# to the objective rather than to the presence of a momentum branch.
#
# Most expensive method in the study: 2 full views with gradients through the query
# encoder plus 2 more without through the momentum encoder, and a symmetrised loss.
method: mocov3

model:
  arch: vit_small
  patch_size: 16
  img_size: 224
  drop_path_rate: 0.0

optim:
  global_batch: 512
  accum_steps: 1           # see pretrain_simclr.yaml
  epochs: 100
  warmup_epochs: 10
  start_lr: 1.0e-5
  ref_lr: 3.0e-4
  final_lr: 1.0e-6
  weight_decay: 0.1
  final_weight_decay: 0.1
  exclude_bias_and_norm_from_wd: true
  ipe_scale: 1.0
  grad_clip: 3.0

contrastive:
  proj_hidden_dim: 2048    # 4096 in the paper for ViT-B; 2048 is the ratio-faithful
                           # scaling to a 384-d encoder and saves ~330 MB of checkpoint
  proj_out_dim: 256
  proj_num_layers: 3
  pred_hidden_dim: 2048
  temperature: 0.2
  moco_momentum: [0.99, 1.0]
  freeze_patch_embed: true # the paper's own fix for ViT training collapse
  symmetric_loss: true

aug:
  crop_scale: [0.2, 1.0]
  horizontal_flip: true
  color_jitter_strength: 0.5
  color_distortion: true
  gaussian_blur: true
  solarize: true           # MoCo v3 uses BYOL-style asymmetric solarization
  two_views: true

runtime:
  seed: 0
  num_workers: 3
  session_guard_hours: 7.5
  ckpt_push_minutes: 45
  ckpt_dataset_slug: ""


In [ ]:
%%writefile /kaggle/working/jepa-thesis/configs/segment_segformer.yaml
# Segmentation fine-tuning on Kvasir-SEG.
#
# IMPORTANT: this file is identical for every encoder arm. Hyperparameters were chosen
# once, on the validation split, using the random-init encoder, and then frozen. Never
# re-tune per encoder — that would make the comparison meaningless.
encoder: ijepa            # ijepa | mae | simclr | mocov3 | random  (set per run)
decoder: segformer        # segformer (primary) | unet (ablation)

model:
  arch: vit_small
  patch_size: 16
  img_size: 352           # polyp-literature standard (PraNet, Polyp-PVT);
                          # 352/16 = 22 -> a clean 484-token grid
  drop_path_rate: 0.1

# 0-indexed encoder blocks {3, 6, 9, 12} -> simple feature pyramid at strides {4,8,16,32}
fpn_layers: [2, 5, 8, 11]
decoder_embed_dim: 256

epochs: 100
batch_size: 16
enc_lr: 1.0e-4
dec_lr: 1.0e-3
layer_decay: 0.75         # earlier blocks keep generic structure, later blocks adapt
weight_decay: 0.05
warmup_epochs: 5
grad_clip: 1.0
early_stop_patience: 20

bce_weight: 0.5
dice_weight: 0.5

label_fraction: 1.0       # low-label ablation: 0.1 / 0.25 / 0.5 / 1.0
split: "800_100_100"      # primary; "880_120" for the literature-comparability table
pretrained_ckpt: ""       # blank -> auto-resolve outputs/weights/<encoder>_<arch>_*.pt

runtime:
  seed: 0                 # 5 seeds (0-4) per encoder for the main table
  num_workers: 3


In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')


In [ ]:
sh('python -m src.engine.pretrain --method simclr --ckpt-slug morsalin101/jepa-thesis-ckpt --guard-hours 7.5', check=False)


In [ ]:
# If the cell above printed 'N epochs remaining', the session guard stopped it
# cleanly — just Save & Run All again to continue. If it printed 'run complete',
# the exported encoder is in /kaggle/working/weights/ and the next cell ships it.
!ls -la /kaggle/working/weights/ 2>/dev/null || echo 'not finished yet — re-run'


In [ ]:
# Publish the finished encoder (~88 MB) to the shared weights dataset that the
# segmentation notebook mounts. Safe to re-run; a no-op until the run completes.
import glob, json, pathlib, shutil, subprocess

SLUG = 'morsalin101/jepa-thesis-weights'
found = glob.glob('/kaggle/working/weights/*.pt')
if not found:
    print('nothing to publish yet — pretraining has not finished')
else:
    stage = pathlib.Path('/kaggle/working/weights_upload')
    stage.mkdir(exist_ok=True)
    # Carry over any encoders already in the dataset so a new version never
    # drops the other methods' weights.
    for p in glob.glob('/kaggle/input/jepa-thesis-weights/*.pt'):
        shutil.copy(p, stage)
    for p in found:
        shutil.copy(p, stage)
    (stage / 'dataset-metadata.json').write_text(json.dumps(
        {'title': 'jepa-thesis-weights', 'id': SLUG,
         'licenses': [{'name': 'CC0-1.0'}]}, indent=2))
    exists = subprocess.run(['kaggle','datasets','status',SLUG],
                            capture_output=True).returncode == 0
    cmd = (['kaggle','datasets','version','-p',str(stage),'-m',
            'add simclr','--dir-mode','zip']
           if exists else
           ['kaggle','datasets','create','-p',str(stage),'--dir-mode','zip','--private'])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)
    print('contents:', sorted(p.name for p in stage.glob('*.pt')))
